# Respiratory Disease Cohort (eICU-CRD v2.0): Bias Detection, Method Comparison and Mitigation

**Goal.** Prepare the respiratory-disease data in this folder for machine learning **free of the eight data
biases** in the project's method table. For every bias the notebook:

1. **Detects** whether the bias is present, using a quantitative test with a documented threshold.
2. **Runs every method** that the method table lists for that bias.
3. **Scores every method with every metric** from the table and **selects the best method** by mean rank,
   with a *do-no-harm* check on predictive performance wherever a model is involved.
4. **Applies the selected method**, but only when the bias is present. When it is absent the data is left unchanged.
5. Draws a **before vs after dashboard**: histograms, box plots, Q-Q plot, correlation heatmap and a
   bias-indicator heatmap. When a bias is present and removed, *before* and *after* differ; when it is absent,
   they are identical.

| Bias type | Methods compared (method table) | Evaluation metrics (method table) |
|---|---|---|
| Missing Data Bias | HyperImpute, CSDI, MIP-VAE | Wasserstein Distance, Correlation Preservation, MAE, RMSE, NRMSE, CRPS, NLL |
| Class Imbalance Bias | TabDDPM, CTGAN, Class-Balanced Focal Loss, Deep SMOTE | Imbalance Ratio, Class Entropy, Balanced F1-score, Macro-F1, Balanced Accuracy, PR-AUC, G-Mean, KS Statistic, TVD |
| Outlier Bias | ECOD, Deep SVDD, Explainable Boosting Machines (EBM) | Outlier Score Stability, Neighborhood Consistency, AUROC, AUPRC, Precision@K, F1-score, Brier Score |
| Feature Scale Bias | Self-Normalizing Neural Networks (SNN/SELU), FT-Transformer, PowerTransform (Yeo-Johnson) | Wasserstein Distance, Correlation Preservation, Skewness Reduction, Kurtosis, Activation Variance, Gradient Stability |
| Demographic Bias | Adversarial Debiasing, Counterfactual Algorithmic Recourse (DiCE), Fairness-Aware Contrastive Learning | Statistical Parity Difference, Disparate Impact Difference, Equal Opportunity Difference, Equalized Odds Difference, Counterfactual Validity, Proximity, Sparsity |
| Duplicate Bias | Ditto, Graph Neural Network (GNN) Entity Resolution, Locality-Sensitive Hashing (LSH) | Duplicate Rate, Redundancy Reduction, Precision, Recall, F1-score, Pair Completeness, Reduction Ratio |
| Data Noise Bias | Confident Learning (Cleanlab), Robust Loss Functions, Diffusion Denoising Models | Residual Variance, Spectral Distortion, MAE, RMSE, Signal-to-Noise Ratio (SNR), Label-Error Detection F1-score |
| Encoding / Domain Bias | Invariant Risk Minimization (IRM), Optimal Transport Domain Adaptation (OTDA), Domain Generalization Transformers | MMD, Domain Classifier AUC, Wasserstein Distance, Worst-Group Accuracy, Domain Generalization Gap, Target-Domain AUROC, OOD Performance |

**Data.** eICU Collaborative Research Database v2.0 (PhysioNet, credentialed access): the CSV tables in this folder.
**Cohort:** adult ICU stays admitted for a respiratory disease. **Target:** in-hospital mortality.
**Features:** the first 24 hours of the ICU stay (vital signs, laboratory values, ventilator settings, APACHE
components, comorbidities, treatments).

**Order of processing.** Biases are handled in the order the data needs, because each step depends on the one before:

| Step | Section | Why here |
|---|---|---|
| 1 | Encoding / Domain Bias, part A: encoding harmonisation | Units, codes and labels must agree before anything can be measured |
| 2 | Duplicate Bias | One stay per patient before the patient-disjoint train/test split |
| 3 | Outlier Bias, part A: physiologically impossible values | Gross recording errors are removed before smoothing and imputing |
| 4 | Data Noise Bias | Vital-sign series are denoised before the 24-hour features are computed |
| 5 | Missing Data Bias | Imputation needs clean, de-noised inputs |
| 6 | Outlier Bias, part B: multivariate outliers | Multivariate detectors need complete (imputed) records |
| 7 | Feature Scale Bias | Scaling of the cleaned features |
| 8 | Encoding / Domain Bias, part B: hospital-region shift | Domain alignment works on scaled features |
| 9 | Demographic Bias | Fairness is checked on the model-ready data |
| 10 | Class Imbalance Bias | Rebalancing changes only the training split, so it comes last |

**Selection rule (same for every bias).** Every metric ranks the methods (rank 1 = best in that metric's direction).
The method with the **lowest mean rank** is selected. Where methods train a predictive model, a method whose
ROC-AUC is more than **0.02** below the no-mitigation baseline is not eligible (do-no-harm guard).

**Leakage control.** The train/test split is patient-disjoint. Every transformation is fitted on the training split
only and then applied to the test split. Synthetic rows are added to the training split only.

**Runtime.** A first run takes roughly 2-3 hours on a laptop CPU. Every expensive result is cached in
`preprocessing_output/cache/`, so later runs take minutes. Set `FORCE_RECOMPUTE = True` to recompute everything.

## 0. Setup
### 0.1 Required packages
All packages are on PyPI. The notebook was run with the kernel **Python 3.12 (resp_physionet)**: the `.venv` in
this folder, which adds `interpret-core`, `transformers`, `torch-geometric` and `rtdl_revisiting_models` to the
system Python. Set `AUTO_INSTALL = True` to install anything that is missing into the running kernel.

In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED = {  # import name -> pip package
    "numpy": "numpy", "pandas": "pandas", "scipy": "scipy", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "pyarrow": "pyarrow", "torch": "torch", "pyod": "pyod",
    "cleanlab": "cleanlab", "interpret": "interpret-core", "transformers": "transformers",
    "torch_geometric": "torch-geometric", "rtdl_revisiting_models": "rtdl_revisiting_models", "ot": "POT",
}
AUTO_INSTALL = False
missing_pkgs = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs and AUTO_INSTALL:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_pkgs])
    missing_pkgs = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing_pkgs:
    raise ImportError(f"Install these packages first: pip install {' '.join(missing_pkgs)}")
print(f"Python {sys.version.split()[0]} - all {len(REQUIRED)} required packages are available")

### 0.2 Imports, paths and configuration

In [ ]:
import contextlib
import hashlib
import itertools
import json
import logging
import math
import os

# keep library chatter (model-load reports, progress bars) out of the notebook output
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
import pickle
import random
import textwrap
import time
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.csv as pacsv
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cbook import boxplot_stats
from matplotlib.colors import to_rgba
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator
from scipy import stats
from scipy.ndimage import median_filter
from scipy.signal import welch
from scipy.stats import kurtosis, skew, spearmanr, wasserstein_distance
from IPython.display import Markdown, display

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, brier_score_loss, f1_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold, cross_val_predict, cross_val_score,
                                     train_test_split)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, QuantileTransformer, RobustScaler, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 80)

# ---------------------------------------------------------------- paths
NOTEBOOK_DIR = Path.cwd()
DATA_CANDIDATES = [NOTEBOOK_DIR, NOTEBOOK_DIR / "resp_physionet" / "resp",
                   Path(r"D:\complete\resp_physionet\resp")]
DATA_DIR = next((_d for _d in DATA_CANDIDATES if (_d / "patient.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Open this notebook from D:/complete or its resp_physionet/resp folder")
# Complete gzip sources avoid FAT32's maximum single-file size for expanded CSVs.
GZ_DIR = DATA_DIR / "official_verified"
OUT_DIR = DATA_DIR / "preprocessing_output"
FIG_DIR, TAB_DIR, DATA_OUT, CACHE_DIR = (OUT_DIR / d for d in ("figures", "tables", "data", "cache"))
for _d in (FIG_DIR, TAB_DIR, DATA_OUT, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- reproducibility / compute
SEED = 42
TEST_SIZE = 0.25
FORCE_RECOMPUTE = False          # True = ignore every cached result
N_THREADS = min(12, os.cpu_count() or 4)
torch.set_num_threads(N_THREADS)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()

# ---------------------------------------------------------------- cohort definition
# APACHE IV admission diagnoses of the respiratory system (medical and operative)
RESP_APACHE_DX = {
    "Sepsis, pulmonary": "pneumonia_sepsis", "Pneumonia, bacterial": "pneumonia_sepsis",
    "Pneumonia, aspiration": "pneumonia_sepsis", "Pneumonia, other": "pneumonia_sepsis",
    "Pneumonia, viral": "pneumonia_sepsis", "Pneumonia, fungal": "pneumonia_sepsis",
    "Pneumonia, parasitic (i.e., Pneumocystic pneumonia)": "pneumonia_sepsis",
    "Emphysema/bronchitis": "copd_asthma", "Asthma": "copd_asthma",
    "Arrest, respiratory (without cardiac arrest)": "resp_failure_ards",
    "Respiratory - medical, other": "resp_failure_ards",
    "ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema": "resp_failure_ards",
    "Obstruction-airway (i.e., acute epiglottitis, post-extubation edema, foreign body, etc)": "resp_failure_ards",
    "Weaning from mechanical ventilation (transfer from other unit or hospital only)": "resp_failure_ards",
    "Apnea, sleep": "resp_failure_ards",
    "Embolus, pulmonary": "pulmonary_vascular", "Hypertension-pulmonary, primary/idiopathic": "pulmonary_vascular",
    "Hemorrhage/hemoptysis, pulmonary": "pulmonary_vascular",
    "Effusions, pleural": "pleural", "Pneumothorax": "pleural", "Hemothorax": "pleural",
    "Restrictive lung disease (i.e., Sarcoidosis, pulmonary fibrosis)": "chronic_lung_other",
    "Chest pain, respiratory": "chronic_lung_other",
    "Cancer, lung": "thoracic_surgery_cancer", "Thoracotomy for lung cancer": "thoracic_surgery_cancer",
    "Thoracotomy for pleural disease": "thoracic_surgery_cancer",
    "Thoracotomy for thoracic/respiratory infection": "thoracic_surgery_cancer",
    "Thoracotomy for lung reduction": "thoracic_surgery_cancer",
    "Thoracotomy for bronchopleural fistula": "thoracic_surgery_cancer",
    "Respiratory surgery, other": "thoracic_surgery_cancer", "Biopsy, open lung": "thoracic_surgery_cancer",
    "Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)": "thoracic_surgery_cancer",
    "Lung transplant, bilateral": "thoracic_surgery_cancer", "Lung transplant, single": "thoracic_surgery_cancer",
}
# second entry route: a *primary* diagnosis whose top-level category is pulmonary
PULM_DX_SUBGROUP = {
    "respiratory failure": "resp_failure_ards", "pulmonary infections": "pneumonia_sepsis",
    "disorders of the airways": "copd_asthma", "disorders of vasculature": "pulmonary_vascular",
    "pleural disorders": "pleural", "post thoracic surgery": "thoracic_surgery_cancer",
}
MIN_AGE, MIN_UNIT_LOS_MIN = 18, 240           # adults; >= 4 h of ICU data
VITAL_WINDOW, LAB_WINDOW, RESP_WINDOW, INFUSION_WINDOW = (0, 1440), (-360, 1440), (0, 1440), (0, 1440)

# ---------------------------------------------------------------- raw-source dictionaries
VITAL_PERIODIC_COLS = ["heartrate", "respiration", "sao2", "temperature"]
VITAL_APERIODIC_COLS = ["noninvasivesystolic", "noninvasivediastolic", "noninvasivemean"]
LAB_NAMES = {
    "pH": "ph", "paO2": "pao2", "paCO2": "paco2", "HCO3": "hco3", "Base Excess": "base_excess",
    "FiO2": "lab_fio2", "lactate": "lactate", "sodium": "sodium", "potassium": "potassium",
    "chloride": "chloride", "bicarbonate": "bicarbonate", "BUN": "bun", "creatinine": "creatinine",
    "glucose": "glucose", "bedside glucose": "glucose", "calcium": "calcium", "albumin": "albumin",
    "total bilirubin": "bilirubin", "WBC x 1000": "wbc", "Hgb": "hgb", "platelets x 1000": "platelets",
    "PT - INR": "inr", "BNP": "bnp", "troponin - I": "troponin_i", "anion gap": "anion_gap",
}
# a naive reader only knows the canonical spelling of each ventilator concept ...
RESP_LABEL_CANONICAL = {
    "FiO2": "fio2", "PEEP": "peep", "Tidal Volume (set)": "tidal_volume_set",
    "Peak Insp. Pressure": "peak_insp_pressure", "Plateau Pressure": "plateau_pressure",
    "Mean Airway Pressure": "mean_airway_pressure", "Total RR": "rr_total", "Exhaled MV": "minute_vent",
}
# ... while hospitals chart the same concept under site-specific labels (encoding bias)
RESP_LABEL_SYNONYMS = {
    "FiO2": "fio2", "FIO2 (%)": "fio2", "O2 Percentage": "fio2", "Set Fraction of Inspired Oxygen (FIO2)": "fio2",
    "PEEP": "peep", "PEEP/CPAP": "peep",
    "Tidal Volume (set)": "tidal_volume_set", "Adult Con Setting Set Vt": "tidal_volume_set",
    "Set Vt (Servo,LTV)": "tidal_volume_set", "Set Vt (Drager)": "tidal_volume_set",
    "Peak Insp. Pressure": "peak_insp_pressure", "Plateau Pressure": "plateau_pressure",
    "Mean Airway Pressure": "mean_airway_pressure",
    "Total RR": "rr_total", "f Total": "rr_total", "Resp Rate Total": "rr_total",
    "Exhaled MV": "minute_vent", "MVe": "minute_vent", "Adult Con Pt/Vent MinuteVentil": "minute_vent",
    "Measured Ve": "minute_vent",
}
RESP_LABELS_ALL = sorted(set(RESP_LABEL_SYNONYMS) | set(RESP_LABEL_CANONICAL))
VASOPRESSORS = ["norepinephrine", "vasopressin", "epinephrine", "dopamine", "phenylephrine", "dobutamine"]
SEDATIVES = ["propofol", "midazolam", "fentanyl", "dexmedetomidine", "lorazepam", "ketamine"]
PAST_HISTORY_FLAGS = {
    "hx_copd": "Pulmonary/COPD", "hx_asthma": "Pulmonary/Asthma", "hx_home_o2": "Pulmonary/Home Oxygen",
    "hx_restrictive_lung": "Pulmonary/Restrictive Disease", "hx_resp_failure": "Pulmonary/Respiratory Failure",
    "hx_chf": "Congestive Heart Failure", "hx_hypertension": "Hypertension Requiring Treatment",
    "hx_renal": "Renal  (R)/Renal", "hx_cancer": "Hematology/Oncology (R)",
}

# ---------------------------------------------------------------- physiological plausibility limits
PLAUSIBLE = {
    "heartrate": (20, 300), "respiration": (2, 80), "sao2": (50, 100), "temperature": (25, 45),
    "noninvasivesystolic": (40, 300), "noninvasivediastolic": (10, 200), "noninvasivemean": (20, 250),
    "ph": (6.5, 8.0), "pao2": (15, 700), "paco2": (5, 200), "hco3": (2, 60), "base_excess": (-40, 40),
    "lab_fio2": (21, 100), "lactate": (0.1, 30), "sodium": (100, 200), "potassium": (1, 12),
    "chloride": (50, 160), "bicarbonate": (2, 60), "bun": (1, 300), "creatinine": (0.1, 25),
    "glucose": (10, 2000), "calcium": (2, 20), "albumin": (0.5, 7), "bilirubin": (0.05, 60),
    "wbc": (0.05, 300), "hgb": (2, 25), "platelets": (1, 2000), "inr": (0.5, 20), "bnp": (1, 100000),
    "troponin_i": (0, 1000), "anion_gap": (-10, 60),
    "fio2": (21, 100), "peep": (0, 40), "tidal_volume_set": (100, 1500), "peak_insp_pressure": (1, 100),
    "plateau_pressure": (1, 80), "mean_airway_pressure": (1, 60), "rr_total": (1, 80), "minute_vent": (0.5, 40),
    "admission_height_cm": (120, 230), "admission_weight_kg": (25, 350), "bmi": (10, 100),
    "gcs_eyes": (1, 4), "gcs_motor": (1, 6), "gcs_verbal": (1, 5), "urine_output_24h": (0, 20000),
}

# ---------------------------------------------------------------- bias-presence thresholds (decision rules)
T = {
    "missing_feature_rate": 0.05,       # a feature with >5% missing counts as affected
    "missing_mechanism_auc": 0.60,      # missingness predictable from observed data => not MCAR
    "missing_drop_rate": 0.70,          # features above this are dropped, not imputed
    "imbalance_ratio": 1.5,             # majority/minority
    "outlier_multivariate_rate": 0.01,  # >1% of stays with a robust |z| > 4.5 value
    "scale_std_ratio": 10.0,            # max/min feature SD
    "scale_skew_share": 0.25,           # >25% of features with |skew| > 1
    "di_low": 0.80, "di_high": 1.25,    # four-fifths rule
    "spd_abs": 0.10,
    "min_group_size": 100,
    "duplicate_rate": 0.01,             # >1% redundant records at any level
    "noise_residual_var": 0.10,         # >10% of within-series variance is high-frequency residual
    "noise_artifact_rate": 0.005,       # >0.5% artefact spikes
    "domain_auc": 0.60,                 # domain-classifier AUC
    "mmd_pvalue": 0.01,
}
MAX_AUC_DROP = 0.02                     # do-no-harm guard

ID_COLS = ["patientunitstayid", "patienthealthsystemstayid", "uniquepid", "hospitalid", "region",
           "hospitaldischargeyear", "unitvisitnumber", "split"]
OUTCOME_COLS = ["hospital_mortality", "icu_mortality", "icu_los_days", "hospital_los_days"]
TARGET, DOMAIN_COL = "hospital_mortality", "region"
PROTECTED = ["gender", "ethnicity"]
PRIVILEGED = {"gender": "Male", "ethnicity": "Caucasian"}

print(f"Data folder   : {DATA_DIR}")
print(f"Output folder : {OUT_DIR}")
print(f"Torch threads : {N_THREADS}")

### 0.3 Helper functions: logging, caching, method selection and tables

In [ ]:
RUN_LOG = OUT_DIR / "run_log.txt"


def log(msg):
    line = f"[{datetime.now():%H:%M:%S}] {msg}"
    print(line, flush=True)
    with open(RUN_LOG, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


@contextlib.contextmanager
def timer(label):
    t0 = time.time()
    yield
    log(f"{label} - {time.time() - t0:.1f}s")


def fingerprint(*objs):
    """Short hash of data/arguments, used to key cached results."""
    h = hashlib.sha1()
    for o in objs:
        if isinstance(o, pd.DataFrame) or isinstance(o, pd.Series):
            h.update(pd.util.hash_pandas_object(o, index=True).values.tobytes())
        elif isinstance(o, np.ndarray):
            h.update(np.ascontiguousarray(o).tobytes())
        else:
            h.update(repr(o).encode())
    return h.hexdigest()[:12]


def cached(name, fn, key=""):
    """Run fn() once and keep its result in CACHE_DIR (pickle); later runs load it."""
    path = CACHE_DIR / (f"{name}_{key}.pkl" if key else f"{name}.pkl")
    if path.exists() and not FORCE_RECOMPUTE:
        with open(path, "rb") as f:
            return pickle.load(f)
    out = fn()
    with open(path, "wb") as f:
        pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)
    return out


def rank_select(table, criteria, guard=None, method_col="Method"):
    """Select the best method by mean rank over several metrics.

    criteria: metric -> "min" | "max" | ("target", value)   (closest to value wins)
    guard: optional boolean array; methods with False are not eligible (do-no-harm).
    A metric that is not applicable to a method (NaN) gets the worst rank for that metric.
    """
    t = table.copy().reset_index(drop=True)
    rank_cols = []
    for metric, how in criteria.items():
        col = pd.to_numeric(t[metric], errors="coerce")
        if isinstance(how, tuple):
            score, asc = (col - how[1]).abs(), True
        else:
            score, asc = col, how == "min"
        t[f"rank[{metric}]"] = score.rank(method="min", ascending=asc, na_option="bottom")
        rank_cols.append(f"rank[{metric}]")
    t["mean_rank"] = t[rank_cols].mean(axis=1)
    eligible = np.ones(len(t), bool) if guard is None else np.asarray(guard, bool)
    t["eligible"] = eligible
    pool = t[eligible] if eligible.any() else t
    best = pool.sort_values(["mean_rank"] + rank_cols).iloc[0][method_col]
    t["selected"] = t[method_col] == best
    return t, best


def save_table(df, name):
    df.to_csv(TAB_DIR / f"{name}.csv", index=False)


def show_table(df, title=None, digits=4):
    """Display a table with rounded numbers; the selected method (if any) is marked with a check mark."""
    d = df.copy()
    if "selected" in d:
        d.insert(0, "✔", np.where(d["selected"], "✔ selected", ""))
        d = d.drop(columns=["selected"])
    if "eligible" in d:
        d["eligible"] = np.where(d["eligible"], "yes", "no (AUC guard)")
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].round(digits)
    if title:
        display(Markdown(f"**{title}**"))
    display(d)


def md(text):
    display(Markdown(text))


# ---------------------------------------------------------------- shared evaluation models
def hgb(seed=SEED, **kw):
    params = dict(max_iter=200, learning_rate=0.08, l2_regularization=1.0, random_state=seed)
    params.update(kw)
    return HistGradientBoostingClassifier(**params)


def oof_proba(F_, y, folds=5, seed=SEED):
    """Out-of-fold predicted probabilities of a gradient-boosting model (NaN and categoricals handled natively)."""
    cv = StratifiedKFold(folds, shuffle=True, random_state=seed)
    model = hgb(seed, categorical_features="from_dtype")
    return cross_val_predict(model, F_, y, cv=cv, method="predict_proba")[:, 1]


def _t(x):
    return torch.as_tensor(np.asarray(x), dtype=torch.float32)


def mlp(sizes, act=nn.ReLU, out_act=None, dropout=0.0, bias=True):
    layers = []
    for i in range(len(sizes) - 1):
        layers.append(nn.Linear(sizes[i], sizes[i + 1], bias=bias))
        if i < len(sizes) - 2:
            layers.append(act())
            if dropout:
                layers.append(nn.Dropout(dropout))
    if out_act is not None:
        layers.append(out_act())
    return nn.Sequential(*layers)


class Standardizer:
    """NaN-aware z-scoring fitted on training rows."""

    def fit(self, X):
        self.mu = np.nanmean(X, 0)
        self.sd = np.nanstd(X, 0)
        self.sd[~np.isfinite(self.sd) | (self.sd == 0)] = 1.0
        self.mu[~np.isfinite(self.mu)] = 0.0
        return self

    def tf(self, X):
        return (X - self.mu) / self.sd

    def inv(self, Z):
        return Z * self.sd + self.mu


print("helpers ready")

### 0.4 Plot style and the before-vs-after dashboard
Every bias gets the same dashboard, laid out like the reference design: a navy header, then *Distribution
Comparison* (overlaid histograms), *Box Plot Comparison*, a *Q-Q Plot (Normality Check)*, *Correlation Heatmaps*
and a *bias-indicator heatmap* (sampled ICU stays x variables; a filled square marks a cell affected by the bias).
**Blue = before**, **green = after** (this colour pair passes the colour-blind separation checks). When a variable
has extreme raw values, the axis is limited to the bulk of the data and each panel prints how many values lie
beyond the axis, so extreme values are never hidden.

In [ ]:
C_BEFORE, C_AFTER = "#3f7fd0", "#2e9e5b"
C_SELECTED, C_OTHER, C_BASE = "#2e9e5b", "#93a3b8", "#cdd5df"
NAVY, PAGE_BG, CARD_EDGE, PANEL_BG, PANEL_EDGE = "#0f1f3d", "#eef2f8", "#d9e0ea", "#f8fafc", "#e2e7ef"
TITLE_BLUE, INK, MUTED, GRID = "#1f5fbf", "#1f2937", "#6b7280", "#e7ebf1"
CELL_ON, CELL_EDGE, QQ_LINE = "#3d5a99", "#cfd6e2", "#d62728"
# validated categorical order (identity colours, e.g. hospital regions); markers add a second cue
CAT_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
CAT_MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    "div", ["#2f5fa7", "#9db8e0", "#f1f1f1", "#e9a3a0", "#b8322d"])

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10.5, "axes.edgecolor": "#c4ccd8", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 9.5, "ytick.labelsize": 9.5,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 80, "savefig.dpi": 110,
    "legend.frameon": False, "figure.max_open_warning": 0,
})

LABELS = {
    "age": "Age (years)", "admission_height_cm": "Height (cm)", "admission_weight_kg": "Weight (kg)",
    "bmi": "BMI (kg/m²)", "apache_iva_score": "APACHE IVa score", "acute_physiology_score": "Acute physiology score",
    "gcs_eyes": "GCS eyes", "gcs_motor": "GCS motor", "gcs_verbal": "GCS verbal", "gcs_total": "GCS total",
    "urine_output_24h": "Urine output 24 h (mL)", "aps_temperature_worst": "Worst temperature (°C)",
    "hr_mean": "Heart rate mean (bpm)", "hr_min": "Heart rate min (bpm)", "hr_max": "Heart rate max (bpm)",
    "rr_mean": "Resp. rate mean (/min)", "rr_min": "Resp. rate min (/min)", "rr_max": "Resp. rate max (/min)",
    "spo2_mean": "SpO₂ mean (%)", "spo2_min": "SpO₂ min (%)", "spo2_frac_below_90": "Time with SpO₂ < 90%",
    "temp_mean": "Temperature mean (°C)", "temp_min": "Temperature min (°C)", "temp_max": "Temperature max (°C)",
    "nibp_sys_mean": "NIBP systolic mean (mmHg)", "nibp_sys_min": "NIBP systolic min (mmHg)",
    "nibp_dia_mean": "NIBP diastolic mean (mmHg)", "nibp_map_mean": "NIBP MAP mean (mmHg)",
    "nibp_map_min": "NIBP MAP min (mmHg)", "ph_min": "pH min", "ph_max": "pH max", "pao2_min": "PaO₂ min (mmHg)",
    "paco2_max": "PaCO₂ max (mmHg)", "hco3_min": "HCO₃ min (mmol/L)", "base_excess_min": "Base excess min (mmol/L)",
    "lab_fio2_max": "FiO₂ max, lab (%)", "lactate_max": "Lactate max (mmol/L)", "sodium_min": "Sodium min (mmol/L)",
    "sodium_max": "Sodium max (mmol/L)", "potassium_max": "Potassium max (mmol/L)",
    "chloride_max": "Chloride max (mmol/L)", "bicarbonate_min": "Bicarbonate min (mmol/L)",
    "bun_max": "BUN max (mg/dL)", "creatinine_max": "Creatinine max (mg/dL)", "glucose_max": "Glucose max (mg/dL)",
    "glucose_min": "Glucose min (mg/dL)", "calcium_min": "Calcium min (mg/dL)", "albumin_min": "Albumin min (g/dL)",
    "bilirubin_max": "Bilirubin max (mg/dL)", "wbc_max": "WBC max (×10³/µL)", "wbc_min": "WBC min (×10³/µL)",
    "hgb_min": "Haemoglobin min (g/dL)", "platelets_min": "Platelets min (×10³/µL)", "inr_max": "INR max",
    "bnp_max": "BNP max (pg/mL)", "troponin_i_max": "Troponin I max (ng/mL)", "anion_gap_max": "Anion gap max",
    "pf_ratio_min": "PaO₂/FiO₂ ratio min", "rc_fio2_max": "FiO₂ max, ventilator (%)",
    "rc_fio2_mean": "FiO₂ mean, ventilator (%)", "rc_peep_max": "PEEP max (cmH₂O)",
    "rc_tidal_volume_set_mean": "Tidal volume set (mL)", "rc_peak_insp_pressure_max": "Peak insp. pressure (cmH₂O)",
    "rc_plateau_pressure_max": "Plateau pressure (cmH₂O)", "rc_mean_airway_pressure_mean": "Mean airway pr. (cmH₂O)",
    "rc_rr_total_mean": "Total RR, ventilator (/min)", "rc_minute_vent_mean": "Minute ventilation (L/min)",
    "icu_los_days": "ICU length of stay (days)", "hospital_los_days": "Hospital stay (days)",
    "hospital_mortality": "Hospital mortality (0/1)", "unitvisitnumber": "ICU visit number",
    "stays_per_patient": "ICU stays per patient", "hospitaldischargeyear": "Discharge year",
}
SHORT = {
    "age": "Age", "admission_height_cm": "Height", "admission_weight_kg": "Weight", "bmi": "BMI",
    "apache_iva_score": "APACHE", "acute_physiology_score": "APS", "gcs_total": "GCS", "gcs_motor": "GCS mot",
    "gcs_eyes": "GCS eye", "gcs_verbal": "GCS verb", "urine_output_24h": "Urine 24h",
    "aps_temperature_worst": "Temp worst", "hr_mean": "HR mean", "hr_min": "HR min", "hr_max": "HR max",
    "rr_mean": "RR mean", "rr_min": "RR min", "rr_max": "RR max", "spo2_mean": "SpO₂ mean", "spo2_min": "SpO₂ min",
    "spo2_frac_below_90": "SpO₂<90%", "temp_mean": "Temp mean", "temp_min": "Temp min", "temp_max": "Temp max",
    "nibp_sys_mean": "SBP mean", "nibp_sys_min": "SBP min", "nibp_dia_mean": "DBP mean", "nibp_map_mean": "MAP mean",
    "nibp_map_min": "MAP min", "ph_min": "pH min", "ph_max": "pH max", "pao2_min": "PaO₂ min", "paco2_max": "PaCO₂ max",
    "hco3_min": "HCO₃ min", "base_excess_min": "BE min", "lab_fio2_max": "FiO₂ lab", "lactate_max": "Lactate",
    "sodium_min": "Na min", "sodium_max": "Na max", "potassium_max": "K max", "chloride_max": "Cl max",
    "bicarbonate_min": "Bicarb", "bun_max": "BUN", "creatinine_max": "Creat", "glucose_max": "Gluc max",
    "glucose_min": "Gluc min", "calcium_min": "Ca min", "albumin_min": "Albumin", "bilirubin_max": "Bili",
    "wbc_max": "WBC max", "wbc_min": "WBC min", "hgb_min": "Hgb min", "platelets_min": "Plt min", "inr_max": "INR",
    "anion_gap_max": "Anion gap", "pf_ratio_min": "P/F min", "rc_fio2_max": "FiO₂ max", "rc_fio2_mean": "FiO₂ mean",
    "rc_peep_max": "PEEP", "rc_tidal_volume_set_mean": "Vt set", "rc_minute_vent_mean": "Min vent",
    "rc_rr_total_mean": "RR vent", "rc_peak_insp_pressure_max": "PIP", "rc_plateau_pressure_max": "Pplat",
    "rc_mean_airway_pressure_mean": "MAWP", "icu_los_days": "ICU LOS", "stays_per_patient": "Stays/pt",
    "unitvisitnumber": "Visit no",
}


def label(col):
    if col in LABELS:
        return LABELS[col]
    return col.replace("_", " ").capitalize()


def short_label(col, n=10):
    s = SHORT.get(col, label(col).split(" (")[0])
    return s if len(s) <= n else s[: n - 1] + "…"


def _finite(x):
    a = np.asarray(pd.to_numeric(pd.Series(np.asarray(x).ravel()), errors="coerce"), dtype=float)
    return a[np.isfinite(a)]


def fmt_num(v):
    if v is None or not np.isfinite(v):
        return "–"
    a = abs(v)
    if float(v).is_integer() and a < 1e4:
        return f"{int(v):,}"
    if a >= 1e6:
        return f"{v / 1e6:.1f}M"
    if a >= 1e4:
        return f"{v / 1e3:.0f}k"
    if a >= 100:
        return f"{v:,.0f}"
    if a >= 10:
        return f"{v:.1f}"
    if 0.99 < a < 1:
        return f"{v:.4f}"            # e.g. reduction ratios 0.9964 vs 0.9997
    if a >= 0.01:
        return f"{v:.2f}"
    return f"{v:.1e}"


_COUNT_FMT = FuncFormatter(lambda v, _: f"{v / 1000:.0f}k" if v >= 1000 else f"{v:.0f}")


def show_fig(fig, name):
    """Save the figure as PNG in preprocessing_output/figures and display it."""
    fig.savefig(FIG_DIR / f"{name}.png", dpi=110, facecolor=fig.get_facecolor())
    plt.show()
    plt.close(fig)


class _Canvas:
    """Figure laid out in inches (y measured from the bottom)."""

    def __init__(self, width, height, bg=PAGE_BG):
        self.W, self.H = width, height
        self.fig = plt.figure(figsize=(width, height), facecolor=bg)

    def card(self, x, y, w, h, face="white", edge=CARD_EDGE, r=0.12, lw=1.0):
        p = FancyBboxPatch((x / self.W, y / self.H), w / self.W, h / self.H,
                           boxstyle=f"round,pad=0,rounding_size={r / self.W}", mutation_aspect=self.W / self.H,
                           transform=self.fig.transFigure, facecolor=face, edgecolor=edge, linewidth=lw, zorder=-10)
        self.fig.add_artist(p)

    def text(self, x, y, s, **kw):
        return self.fig.text(x / self.W, y / self.H, s, **kw)

    def axes(self, x, y, w, h):
        return self.fig.add_axes([x / self.W, y / self.H, w / self.W, h / self.H])


def _empty(ax, msg="no data"):
    ax.text(0.5, 0.5, msg, ha="center", va="center", transform=ax.transAxes, color=MUTED, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])


def axis_range(vals):
    """Full range, unless the tails are extreme: then the 0.5-99.5 percentile range widened by 25%, and - when more
    than 0.5% of the values are gross errors - Tukey-style fences at 10 IQR."""
    lo, hi = float(np.min(vals)), float(np.max(vals))
    q1, q3 = np.quantile(vals, [0.25, 0.75])
    q_lo, q_hi = np.quantile(vals, [0.005, 0.995])
    span = q_hi - q_lo
    if span > 0 and (hi - lo) > 3 * span:
        lo, hi = max(lo, q_lo - 0.25 * span), min(hi, q_hi + 0.25 * span)
    iqr = q3 - q1
    if iqr > 0:
        f_lo, f_hi = max(lo, q1 - 10 * iqr), min(hi, q3 + 10 * iqr)
        if f_hi > f_lo and (hi - lo) > 2 * (f_hi - f_lo):
            lo, hi = f_lo, f_hi
    if hi <= lo:
        hi = lo + 1.0
    return lo, hi


def _hist_edges(vals, n=12):
    uniq = np.unique(vals)
    lo, hi = float(uniq[0]), float(uniq[-1])
    if len(uniq) <= 12 and np.allclose(uniq, np.round(uniq)) and hi - lo <= 30:
        return np.arange(lo - 0.5, hi + 1.5, 1.0), True
    lo, hi = axis_range(vals)
    return np.linspace(lo, hi, n + 1), False


def _outside_note(ax, series, lo, hi, names, y=1.01):
    """Count the values outside the axis range; printed just above the plot (inside it when y < 1)."""
    counts = [int(((d < lo) | (d > hi)).sum()) for d in series]
    if sum(counts) == 0:
        return
    ax.text(0.0, y, "beyond axis: " + "  ·  ".join(f"{n} {c:,}" for n, c in zip(names, counts)),
            transform=ax.transAxes, ha="left", va="bottom" if y >= 1 else "top", fontsize=8, color=MUTED,
            clip_on=False)


def _xticks_minmax(ax, edges, discrete):
    if discrete:
        centers = (edges[:-1] + edges[1:]) / 2
        ticks = centers if len(centers) <= 7 else centers[[0, -1]]
        ax.set_xticks(ticks, [fmt_num(t) for t in ticks])
    else:
        ax.set_xticks([edges[0], edges[-1]], [fmt_num(edges[0]), fmt_num(edges[-1])])
        labs = ax.get_xticklabels()
        if len(labs) == 2:
            labs[0].set_horizontalalignment("left")
            labs[1].set_horizontalalignment("right")
    ax.set_xlim(edges[0], edges[-1])


def _hist(ax, b, a):
    b, a = _finite(b), _finite(a)
    if len(b) + len(a) == 0:
        return _empty(ax)
    edges, discrete = _hist_edges(np.concatenate([b, a]))
    for d, c, lab in ((b, C_BEFORE, "Before"), (a, C_AFTER, "After")):
        if len(d):
            ax.hist(d, bins=edges, color=to_rgba(c, 0.33), edgecolor=c, linewidth=1.0, label=lab)
    _xticks_minmax(ax, edges, discrete)
    ax.yaxis.set_major_locator(MaxNLocator(4, integer=True))
    ax.yaxis.set_major_formatter(_COUNT_FMT)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.18)
    if not discrete:
        _outside_note(ax, [b, a], edges[0], edges[-1], ["before", "after"])


def _hist_single(ax, d, color, tag):
    d = _finite(d)
    if len(d) == 0:
        return _empty(ax)
    edges, discrete = _hist_edges(d)
    ax.hist(d, bins=edges, color=to_rgba(color, 0.33), edgecolor=color, linewidth=1.0)
    _xticks_minmax(ax, edges, discrete)
    ax.yaxis.set_major_locator(MaxNLocator(3, integer=True))
    ax.yaxis.set_major_formatter(_COUNT_FMT)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.25)
    ax.text(0.98, 0.95, tag, transform=ax.transAxes, ha="right", va="top", fontsize=8.5, color=color, weight="bold")
    if not discrete:
        _outside_note(ax, [d], edges[0], edges[-1], [tag], y=0.95)


def _box_stats(d, rng):
    s = boxplot_stats(d, whis=1.5)[0]
    if len(s["fliers"]) > 120:
        s["fliers"] = rng.choice(s["fliers"], 120, replace=False)
    return s


def _box(ax, series, colors, names):
    rng = np.random.default_rng(0)
    pos = np.arange(1, len(series) + 1)
    data = [_finite(d) for d in series]
    nonempty = [d for d in data if len(d)]
    if not nonempty:
        return _empty(ax)
    lo, hi = axis_range(np.concatenate(nonempty))
    pad = 0.06 * (hi - lo)
    for p, d, c in zip(pos, data, colors):
        if len(d) == 0:
            continue
        ax.bxp([_box_stats(d, rng)], positions=[p], widths=0.55, patch_artist=True, showfliers=True,
               boxprops=dict(facecolor=to_rgba(c, 0.33), edgecolor=c, linewidth=1.2),
               medianprops=dict(color=c, linewidth=1.8), whiskerprops=dict(color=c, linewidth=1.1),
               capprops=dict(color=c, linewidth=1.1),
               flierprops=dict(marker="o", markersize=3.6, markerfacecolor="none", markeredgecolor=c, alpha=0.75))
    ax.set_xticks(pos, names)
    ax.set_xlim(0.4, len(series) + 0.6)
    ax.set_ylim(lo - pad, hi + 3 * pad)
    ax.yaxis.set_major_locator(MaxNLocator(4))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: fmt_num(v)))
    ax.grid(axis="x", visible=False)
    _outside_note(ax, data, lo - pad, hi + 3 * pad, [n.lower() for n in names])


def _qq(ax, x, color, title):
    x = _finite(x)
    if len(x) < 5:
        return _empty(ax)
    sd = x.std()
    z = (x - x.mean()) / (sd if sd > 0 else 1.0)
    p = (np.arange(1, 81) - 0.5) / 80
    sq, tq = np.quantile(z, p), stats.norm.ppf(p)
    lim = [min(tq.min(), sq.min()) - 0.1, max(tq.max(), sq.max()) + 0.1]
    ax.plot(lim, lim, color=QQ_LINE, linestyle="--", linewidth=1.3, zorder=2)
    ax.scatter(tq, sq, s=18, color=to_rgba(color, 0.5), edgecolor=color, linewidth=0.6, zorder=3)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.text(0.03, 0.97, f"skewness {skew(x):+.2f}\nn = {len(x):,}", transform=ax.transAxes, va="top",
            fontsize=9.5, color=INK)
    ax.set_xlabel(title, fontsize=12, weight="bold", color=INK, labelpad=6)
    ax.tick_params(labelbottom=False, labelleft=False, length=0)
    ax.grid(False)


def _corr(ax, df, cols, title):
    Cm = df[cols].apply(pd.to_numeric, errors="coerce").corr(min_periods=30).to_numpy()
    k = len(cols)
    ax.imshow(np.nan_to_num(Cm), cmap=CMAP_DIV, vmin=-1, vmax=1, aspect="equal")
    for i in range(k):
        for j in range(k):
            v = Cm[i, j]
            ax.text(j, i, "" if not np.isfinite(v) else f"{v:.2f}", ha="center", va="center", fontsize=8.6,
                    color="white" if np.isfinite(v) and abs(v) > 0.55 else INK)
    names = [short_label(c) for c in cols]
    ax.set_xticks(range(k), names, rotation=40, ha="left", fontsize=9, color=INK)
    ax.xaxis.tick_top()
    ax.set_yticks(range(k), names, fontsize=9, color=INK)
    ax.tick_params(length=0)
    ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_xlabel(title, fontsize=12, weight="bold", color=INK, labelpad=6)


def _cells(ax, Mx, row_labels=None):
    nr, nc = Mx.shape
    for i in range(nr):
        for j in range(nc):
            on = bool(Mx[i, j])
            ax.add_patch(FancyBboxPatch((j + 0.12, nr - 1 - i + 0.12), 0.76, 0.76,
                                        boxstyle="round,pad=0,rounding_size=0.14",
                                        facecolor=CELL_ON if on else "white", edgecolor=CELL_ON if on else CELL_EDGE,
                                        linewidth=0.7))
    ax.set_xlim(0, nc)
    ax.set_ylim(0, nr)
    ax.axis("off")
    if row_labels is not None:
        for i, lab in enumerate(row_labels):
            ax.text(-0.4, nr - 1 - i + 0.5, lab, ha="right", va="center", fontsize=10, color=INK, clip_on=False)


def bias_dashboard(bias, method, before, after, variables, *, present=True, indicator=None, qq_var=None,
                   corr_vars=None, same_axis=True, before_name="before mitigation", after_name=None,
                   note=None, fname="dashboard"):
    """Reference-style before/after dashboard for one bias.

    indicator: dict(before=bool[rows x n], after=bool[rows x n], rows=[labels], title=str, name=str, footer=str)
    same_axis=False draws before and after on separate axes (used when the units change, e.g. scaling).
    """
    variables = [v for v in variables if v in before.columns and v in after.columns][:10]
    after_name = after_name or (method if present else "unchanged")
    qq_var = qq_var if qq_var in variables else variables[0]
    corr_vars = [c for c in (corr_vars or variables) if c in variables][:8]
    W, Mg, GAP, NC = 18.0, 0.3, 0.26, 5
    cw = W - 2 * Mg
    nr = max(1, math.ceil(len(variables) / NC))
    PH, PG, TH = 2.45, 0.2, 0.62
    grid_h = TH + nr * PH + (nr - 1) * PG + 0.25
    qq_h = 5.6
    if indicator is not None:
        n_ir, n_ic = np.asarray(indicator["before"]).shape
        label_w = 2.6
        cell = min(0.22, (cw - 0.5 - label_w - 0.6) / (2 * n_ic))
        ind_h = TH + 0.42 + n_ir * cell + 0.42 + (0.32 if indicator.get("footer") else 0) + 0.15
    note_lines = textwrap.wrap(note, 200) if note else []
    note_h = 0.25 * len(note_lines) + (0.15 if note_lines else 0)
    H = Mg + 1.05 + GAP + 2 * (grid_h + GAP) + qq_h + ((GAP + ind_h) if indicator is not None else 0) + note_h + Mg
    cv = _Canvas(W, H)
    top = H - Mg

    # ---- header
    hh = 1.05
    cv.card(Mg, top - hh, cw, hh, face=NAVY, edge=NAVY)
    title = f"EDA Dashboard — {bias}, All Variables (Before vs After {method})"
    cv.text(W / 2, top - 0.38, title, ha="center", va="center", color="white", weight="bold",
            fontsize=min(19.0, (cw - 0.6) * 72 / (0.66 * len(title))))
    status = (f"⚠  Bias detected  →  removed with: {method}" if present
              else "✔  Bias not detected  →  data left unchanged (before = after)")
    cv.text(W / 2, top - 0.8, status, ha="center", va="center", color="#ffd38a" if present else "#a7f3c3",
            fontsize=12.5, weight="bold")
    top -= hh + GAP

    def grid_card(kind):
        nonlocal top
        cv.card(Mg, top - grid_h, cw, grid_h)
        ttl = ("Distribution Comparison" if kind == "hist" else "Box Plot Comparison") + f" (Before vs After {method})"
        cv.text(Mg + 0.25, top - 0.33, ttl, ha="left", va="center", color=TITLE_BLUE, fontsize=14.5, weight="bold")
        if kind == "hist":
            cv.fig.legend(handles=[Patch(facecolor=to_rgba(C_BEFORE, 0.33), edgecolor=C_BEFORE,
                                         label=f"Before ({before_name})"),
                                   Patch(facecolor=to_rgba(C_AFTER, 0.33), edgecolor=C_AFTER,
                                         label=f"After ({after_name})")],
                          loc="center right", bbox_to_anchor=((W - Mg - 0.2) / W, (top - 0.33) / H), ncol=2,
                          fontsize=11, handlelength=1.4, columnspacing=1.6)
        pw = (cw - 0.5 - (NC - 1) * PG) / NC
        for k, v in enumerate(variables):
            r, c = divmod(k, NC)
            px = Mg + 0.25 + c * (pw + PG)
            py = top - TH - (r + 1) * PH - r * PG
            cv.card(px, py, pw, PH, face=PANEL_BG, edge=PANEL_EDGE, r=0.08)
            cv.text(px + pw / 2, py + PH - 0.22, label(v), ha="center", va="center", fontsize=11.5,
                    weight="bold", color=INK)
            l, b_, rgt, t_ = 0.5, 0.33, 0.14, 0.45
            iw, ih = pw - l - rgt, PH - b_ - t_
            if same_axis:
                ax = cv.axes(px + l, py + b_, iw, ih)
                if kind == "hist":
                    _hist(ax, before[v], after[v])
                else:
                    _box(ax, [before[v], after[v]], [C_BEFORE, C_AFTER], ["Before", "After"])
            elif kind == "hist":
                h2 = (ih - 0.3) / 2
                _hist_single(cv.axes(px + l, py + b_ + h2 + 0.3, iw, h2), before[v], C_BEFORE, "before")
                _hist_single(cv.axes(px + l, py + b_, iw, h2), after[v], C_AFTER, "after")
            else:
                w2 = (iw - 0.45) / 2
                _box(cv.axes(px + l, py + b_, w2, ih), [before[v]], [C_BEFORE], ["Before"])
                _box(cv.axes(px + l + w2 + 0.45, py + b_, w2, ih), [after[v]], [C_AFTER], ["After"])
        top -= grid_h + GAP

    grid_card("hist")
    grid_card("box")

    # ---- Q-Q + correlation
    qw = 0.4 * (cw - GAP)
    kw_ = cw - GAP - qw
    cv.card(Mg, top - qq_h, qw, qq_h)
    cv.text(Mg + 0.25, top - 0.33, f"Q-Q Plot (Normality Check) — {label(qq_var)}", ha="left", va="center",
            color=TITLE_BLUE, fontsize=14.5, weight="bold")
    aw = (qw - 0.5 - 0.5) / 2
    ah = min(qq_h - TH - 0.8, aw * 1.25)
    ay = top - TH - 0.1 - ah
    _qq(cv.axes(Mg + 0.25, ay, aw, ah), before[qq_var], C_BEFORE, "Before")
    _qq(cv.axes(Mg + 0.25 + aw + 0.5, ay, aw, ah), after[qq_var], C_AFTER, "After")
    x0 = Mg + qw + GAP
    cv.card(x0, top - qq_h, kw_, qq_h)
    cv.text(x0 + 0.25, top - 0.33, "Correlation Heatmap — core variables", ha="left", va="center",
            color=TITLE_BLUE, fontsize=14.5, weight="bold")
    side = min((kw_ - 0.5 - 2 * 1.0 - 0.35) / 2, qq_h - TH - 1.6)
    for i, (d_, ttl) in enumerate(((before, "Before"), (after, "After"))):
        _corr(cv.axes(x0 + 0.25 + 1.0 + i * (side + 1.0 + 0.35), top - qq_h + 0.7, side, side), d_, corr_vars, ttl)
    top -= qq_h + GAP

    # ---- indicator heatmap
    if indicator is not None:
        cv.card(Mg, top - ind_h, cw, ind_h)
        cv.text(Mg + 0.25, top - 0.33, indicator["title"], ha="left", va="center", color=TITLE_BLUE,
                fontsize=14.5, weight="bold")
        name = indicator.get("name", "affected")
        cv.fig.legend(handles=[Patch(facecolor=CELL_ON, edgecolor=CELL_ON, label=name),
                               Patch(facecolor="white", edgecolor=CELL_EDGE, label=f"not {name}")],
                      loc="center right", bbox_to_anchor=((W - Mg - 0.2) / W, (top - 0.33) / H), ncol=2,
                      fontsize=11, handlelength=1.2)
        gw = n_ic * cell
        gy = top - TH - 0.42 - n_ir * cell
        for key, gx in (("before", Mg + 0.25 + label_w), ("after", Mg + 0.25 + label_w + gw + 0.6)):
            Mx = np.asarray(indicator[key], bool)
            _cells(cv.axes(gx, gy, gw, n_ir * cell), Mx, indicator["rows"] if key == "before" else None)
            cv.text(gx + gw / 2, top - TH - 0.2, key.capitalize(), ha="center", va="center", fontsize=13,
                    weight="bold", color=INK)
            cv.text(gx + gw / 2, gy - 0.22, f"{int(Mx.sum())} of {Mx.size} cells flagged ({Mx.mean():.1%})",
                    ha="center", va="center", fontsize=10.5, color=MUTED)
        if indicator.get("footer"):
            cv.text(Mg + 0.25, gy - 0.55, indicator["footer"], ha="left", va="center", fontsize=10.5, color=INK)
        top -= ind_h
    for i, line in enumerate(note_lines):
        cv.text(Mg, top - 0.2 - 0.25 * i, line, ha="left", va="top", fontsize=10.5, color=MUTED)
    show_fig(cv.fig, fname)


def sample_indicator(mask_before, mask_after, n=40, seed=SEED):
    """mask_*: DataFrames (stays x variables) of booleans; samples the same n stays when possible."""
    rng = np.random.default_rng(seed)
    common = mask_before.index.intersection(mask_after.index)
    pool_b = common if len(common) >= n else mask_before.index
    ib = rng.choice(np.asarray(pool_b), min(n, len(pool_b)), replace=False)
    ia = ib if len(common) >= n else rng.choice(np.asarray(mask_after.index), min(n, len(mask_after)), replace=False)
    return mask_before.loc[ib].to_numpy().T, mask_after.loc[ia].to_numpy().T


def waffle(flags, rows=8, cols=40, seed=SEED):
    """Arrange a random sample of per-record flags as a rows x cols grid."""
    flags = np.asarray(flags, bool)
    rng = np.random.default_rng(seed)
    k = min(rows * cols, len(flags))
    s = rng.choice(len(flags), k, replace=False)
    out = np.zeros(rows * cols, bool)
    out[:k] = flags[s]
    return out.reshape(rows, cols)


def ba_bars(ax, names, before_vals, after_vals, title, value_fmt=fmt_num, xlabel=None):
    """Horizontal grouped bars: blue = before, green = after (one axis, shared scale)."""
    y = np.arange(len(names))[::-1]
    bv, av = np.asarray(before_vals, float), np.asarray(after_vals, float)
    ax.barh(y + 0.19, np.nan_to_num(bv), height=0.36, color=to_rgba(C_BEFORE, 0.75), label="Before")
    ax.barh(y - 0.19, np.nan_to_num(av), height=0.36, color=to_rgba(C_AFTER, 0.85), label="After")
    span = np.nanmax(np.r_[bv, av, 1e-12])
    for yi, v1, v2 in zip(y, bv, av):
        ax.text(np.nan_to_num(v1) + span * 0.01, yi + 0.19, value_fmt(v1), va="center", fontsize=8.5, color=INK)
        ax.text(np.nan_to_num(v2) + span * 0.01, yi - 0.19, value_fmt(v2), va="center", fontsize=8.5, color=INK)
    ax.set_yticks(y, names, fontsize=9.5, color=INK)
    ax.set_xlim(0, span * 1.18)
    ax.set_title(title, loc="left", fontsize=12.5)
    ax.grid(axis="y", visible=False)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.legend(loc="lower right", fontsize=9.5)


def plot_method_comparison(tab, metrics, title, fname, baseline=None, method_col="Method"):
    """One small panel per metric (bars per method); the selected method is green, the others grey.

    metrics: list of (column, direction, label); direction is "max", "min" or ("target", value).
    """
    methods = tab[method_col].tolist()
    sel = tab.loc[tab["selected"].astype(bool), method_col].iloc[0] if "selected" in tab else None
    panels = list(metrics) + [("mean_rank", "min", "Mean rank (selection score)")]
    ncols = min(4, len(panels))
    nrows = math.ceil(len(panels) / ncols)
    ph = 0.95 + 0.42 * len(methods)
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 0.9 + nrows * ph), facecolor="white", squeeze=False)
    y = np.arange(len(methods))[::-1]
    for k, (col, how, lab) in enumerate(panels):
        ax = axes.flat[k]
        vals = pd.to_numeric(tab[col], errors="coerce").to_numpy(float)
        vals = np.where(np.isfinite(vals), vals, np.nan)          # inf (e.g. diverged training) shown as N/A
        colors = [C_SELECTED if m == sel else (C_BASE if m == baseline else C_OTHER) for m in methods]
        ax.barh(y, np.nan_to_num(vals), color=colors, height=0.62)
        lo = min(0.0, np.nanmin(vals)) if np.isfinite(vals).any() else 0.0
        hi = max(0.0, np.nanmax(vals)) if np.isfinite(vals).any() else 1.0
        span = (hi - lo) or 1.0
        for yi, v in zip(y, vals):
            ax.text((v if np.isfinite(v) and v > 0 else 0) + span * 0.015, yi,
                    "N/A" if not np.isfinite(v) else fmt_num(v), va="center", fontsize=9, color=INK)
        ax.set_xlim(lo - (span * 0.05 if lo < 0 else 0), hi + span * 0.32)
        ax.set_yticks(y, methods, fontsize=9.5, color=INK)
        arrow = ("↑ higher is better" if how == "max" else "↓ lower is better" if how == "min"
                 else f"→ closer to {how[1]} is better")
        ax.set_title(f"{lab}\n{arrow}", loc="left", fontsize=11)
        ax.grid(axis="y", visible=False)
        ax.axvline(0, color="#c4ccd8", linewidth=0.8)
    for ax in axes.flat[len(panels):]:
        ax.axis("off")
    handles = [Patch(color=C_SELECTED, label=f"selected: {sel}"), Patch(color=C_OTHER, label="other methods")]
    if baseline is not None:
        handles.append(Patch(color=C_BASE, label=f"reference only: {baseline}"))
    fig.legend(handles=handles, loc="upper right", ncol=3, fontsize=10.5, bbox_to_anchor=(0.99, 1.0))
    fig.suptitle(title, x=0.01, ha="left", fontsize=15, weight="bold", color=TITLE_BLUE)
    fig.tight_layout(rect=(0, 0, 1, 0.96 if nrows > 1 else 0.92))
    show_fig(fig, fname)


def plot_detection_summary(rows, fname="00_bias_detection_summary"):
    """rows: dicts with bias, present, evidence (list of str), rule."""
    W, Mg = 18.0, 0.3
    cw, chh, gap = (W - 2 * Mg - 3 * 0.25) / 4, 3.0, 0.25
    H = Mg + 0.8 + 2 * chh + gap + Mg + 0.3
    cv = _Canvas(W, H)
    cv.card(Mg, H - Mg - 0.75, W - 2 * Mg, 0.75, face=NAVY, edge=NAVY)
    cv.text(W / 2, H - Mg - 0.375, "Bias Detection on the Raw Respiratory Cohort — all eight biases",
            ha="center", va="center", color="white", fontsize=18, weight="bold")
    for k, r in enumerate(rows):
        i, j = divmod(k, 4)
        x = Mg + j * (cw + gap)
        y = H - Mg - 0.75 - 0.3 - (i + 1) * chh - i * gap
        cv.card(x, y, cw, chh)
        cv.text(x + 0.2, y + chh - 0.32, r["bias"], fontsize=13, weight="bold", color=INK, va="center")
        stt = "⚠  DETECTED" if r["present"] else "✔  NOT DETECTED"
        cv.text(x + 0.2, y + chh - 0.7, stt, fontsize=12, weight="bold", va="center",
                color="#b45309" if r["present"] else "#15803d")
        yy = y + chh - 1.05
        for e in r["evidence"][:4]:
            for line in textwrap.wrap(e, 52)[:2]:
                cv.text(x + 0.2, yy, line, fontsize=9.8, color=INK, va="center")
                yy -= 0.24
        rule = textwrap.wrap("Rule: " + r["rule"], 58)[:3]
        for t_, line in enumerate(rule):
            cv.text(x + 0.2, y + 0.62 - t_ * 0.21, line, fontsize=8.6, color=MUTED, va="center")
    show_fig(cv.fig, fname)


print("plotting helpers ready")

## 1. Data: integrity check, respiratory cohort and raw extraction
### 1.1 File integrity
Two of the extracted CSVs in this folder (`vitalPeriodic.csv`, `nurseCharting.csv`) **stop in the middle of a
row**: they were truncated when unzipped. Using them would silently drop most of the vital-sign data. When a
canonical gzip original is available under `official_verified`, the notebook streams it after checking its
actual bytes against the supplied `SHA256SUMS.txt`. A checksum mismatch or a truncated CSV without its verified
original stops execution. `nurseCharting` is not needed: its vital signs are covered by `vitalPeriodic` and APACHE.

In [ ]:
USED_TABLES = ["patient.csv", "hospital.csv", "diagnosis.csv", "vitalPeriodic.csv", "vitalAperiodic.csv", "lab.csv",
               "respiratoryCharting.csv", "respiratoryCare.csv", "infusionDrug.csv", "apacheApsVar.csv",
               "apachePatientResult.csv", "apachePredVar.csv", "pastHistory.csv"]


def csv_is_truncated(path):
    with open(path, "rb") as fh:
        fh.seek(-1, 2)
        return fh.read(1) != b"\n"


def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while True:
            b = fh.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def check_integrity():
    sums = {}
    if (DATA_DIR / "SHA256SUMS.txt").exists():
        for line in (DATA_DIR / "SHA256SUMS.txt").read_text().splitlines():
            if line.strip():
                digest, name = line.split()
                sums[name] = digest
    rows, source = [], {}
    for f in USED_TABLES + ["nurseCharting.csv"]:
        p = DATA_DIR / f
        trunc = csv_is_truncated(p) if p.exists() else True
        gz = GZ_DIR / f"{f}.gz"
        src, verified = str(p), "audited intact raw CSV"
        if gz.exists():
            expected = sums.get(f"{f}.gz")
            if expected is None:
                raise ValueError(f"Missing supplied SHA-256 for {gz.name}")
            # Recheck actual bytes; do not reuse a stale cached checksum.
            if sha256_file(gz) != expected:
                raise ValueError(f"SHA-256 MISMATCH: refusing {gz}")
            src, verified = str(gz), "SHA-256 matches supplied manifest"
        elif trunc:
            raise FileNotFoundError(f"Missing verified original; refusing incomplete CSV: {p}")
        rows.append({"table": f, "size_MB": round((p.stat().st_size if p.exists() else gz.stat().st_size) / 2 ** 20, 1),
                     "ends_mid_row (truncated)": trunc, "used": f in USED_TABLES,
                     "source read by this notebook": Path(src).name if f in USED_TABLES else "not needed",
                     "check": verified})
        source[f] = (Path(src), src.endswith(".gz"))
    return pd.DataFrame(rows), source


INTEGRITY, SOURCE = check_integrity()
show_table(INTEGRITY, "File integrity of the eICU tables in this folder")

### 1.2 Respiratory cohort
An ICU stay enters the cohort if its **APACHE IV admission diagnosis is respiratory** *or* its **primary diagnosis
is pulmonary**. The stay must also be an adult (>= 18 years), have at least 4 h of ICU data and a known
hospital outcome.

In [ ]:
def parse_age(s):
    s = s.astype("string").str.strip()
    out = pd.to_numeric(s, errors="coerce")
    out[s == "> 89"] = 90
    return out


def select_cohort():
    p = pd.read_csv(DATA_DIR / "patient.csv", low_memory=False)
    h = pd.read_csv(DATA_DIR / "hospital.csv")
    d = pd.read_csv(DATA_DIR / "diagnosis.csv",
                    usecols=["patientunitstayid", "diagnosisoffset", "diagnosisstring", "diagnosispriority"])
    prim = d[d.diagnosispriority == "Primary"].copy()
    parts = prim.diagnosisstring.str.split("|")
    prim["top"], prim["sub"] = parts.str[0], parts.str[1]
    prim = prim[prim.top == "pulmonary"].sort_values(["patientunitstayid", "diagnosisoffset"])
    pulm_sub = prim.drop_duplicates("patientunitstayid").set_index("patientunitstayid")["sub"]
    flow = {"all ICU stays in eICU": len(p)}
    by_apache = p.apacheadmissiondx.isin(RESP_APACHE_DX)
    by_dx = p.patientunitstayid.isin(pulm_sub.index)
    sel = p[by_apache | by_dx].copy()
    flow["respiratory APACHE admission diagnosis"] = int(by_apache.sum())
    flow["primary pulmonary diagnosis"] = int(by_dx.sum())
    flow["either (respiratory cohort)"] = len(sel)
    sel = sel[~(parse_age(sel.age) < MIN_AGE)]
    flow["adults (>= 18 y)"] = len(sel)
    sel = sel[sel.unitdischargeoffset >= MIN_UNIT_LOS_MIN]
    flow["ICU stay >= 4 h"] = len(sel)
    sel = sel[sel.hospitaldischargestatus.notna()]
    flow["known hospital outcome"] = len(sel)
    grp = sel.apacheadmissiondx.map(RESP_APACHE_DX)
    grp = grp.fillna(sel.patientunitstayid.map(pulm_sub).map(PULM_DX_SUBGROUP))
    sel["resp_dx_group"] = grp.fillna("other_pulmonary")
    sel = sel.merge(h, on="hospitalid", how="left")
    flow_df = pd.DataFrame({"step": list(flow), "ICU stays": list(flow.values())})
    return sel.reset_index(drop=True), flow_df

### 1.3 Streaming extraction of the raw records (first 24 h)
Nothing is cleaned here. Values, labels, sentinel codes and duplicate rows are kept exactly as recorded, so the
detection step sees the real raw data. Every long-format record gets a `rowhash` (a hash of all its source columns
except the surrogate row id), so exact duplicate records can be counted later.

In [ ]:
def _header(path, gz=False):
    import gzip
    opener = gzip.open if gz else open
    with opener(path, "rt", encoding="utf-8") as fh:
        return fh.readline().strip().split(",")


def stream_extract(path, gz, id_col, row_id, offset_col, window, ids, keep_cols, string_cols=(), row_filter=None,
                   name=""):
    """Stream a (possibly gzip) CSV; keep the cohort's rows inside the time window and add `rowhash`."""
    cols = _header(path, gz)
    types = {c: (pa.string() if c in string_cols else pa.float64()) for c in cols}
    types.update({id_col: pa.int64(), offset_col: pa.int64(), row_id: pa.int64()})
    src = pa.input_stream(str(path), compression="gzip" if gz else None)
    reader = pacsv.open_csv(src, read_options=pacsv.ReadOptions(block_size=1 << 26),
                            parse_options=pacsv.ParseOptions(invalid_row_handler=lambda row: "skip"),
                            convert_options=pacsv.ConvertOptions(column_types=types, strings_can_be_null=True))
    id_arr = pa.array(np.asarray(sorted(ids), dtype=np.int64))
    hash_cols = [c for c in cols if c != row_id]
    chunks, n_read = [], 0
    for batch in reader:
        n_read += batch.num_rows
        tbl = pa.Table.from_batches([batch])
        off = tbl[offset_col]
        mask = pc.and_(pc.is_in(tbl[id_col], value_set=id_arr),
                       pc.and_(pc.greater_equal(off, window[0]), pc.less_equal(off, window[1])))
        tbl = tbl.filter(mask)
        if tbl.num_rows == 0:
            continue
        df = tbl.to_pandas()
        if row_filter is not None:
            df = df[row_filter(df)]
            if df.empty:
                continue
        df["rowhash"] = pd.util.hash_pandas_object(df[hash_cols], index=False).values
        chunks.append(df[keep_cols + ["rowhash"]])
    out = pd.concat(chunks, ignore_index=True)
    log(f"[{name}] scanned {n_read:,} rows -> kept {len(out):,} cohort rows")
    return out, n_read


def _downcast(df):
    for c in df.columns:
        if df[c].dtype == np.float64:
            df[c] = df[c].astype(np.float32)
        elif df[c].dtype == np.int64 and c != "rowhash":
            df[c] = pd.to_numeric(df[c], downcast="integer")
    return df


def run_extraction():
    audit = {}
    cohort, flow = select_cohort()
    ids = set(cohort.patientunitstayid)
    data = {"cohort_patient": cohort}
    vp_path, vp_gz = SOURCE["vitalPeriodic.csv"]
    data["vital_periodic"], audit["vitalPeriodic rows scanned"] = stream_extract(
        vp_path, vp_gz, "patientunitstayid", "vitalperiodicid", "observationoffset", VITAL_WINDOW, ids,
        ["patientunitstayid", "observationoffset"] + VITAL_PERIODIC_COLS, name="vitalPeriodic")
    data["vital_aperiodic"], _ = stream_extract(
        *SOURCE["vitalAperiodic.csv"], "patientunitstayid", "vitalaperiodicid", "observationoffset", VITAL_WINDOW,
        ids, ["patientunitstayid", "observationoffset"] + VITAL_APERIODIC_COLS, name="vitalAperiodic")
    lab_names = set(LAB_NAMES)
    lab, _ = stream_extract(
        *SOURCE["lab.csv"], "patientunitstayid", "labid", "labresultoffset", LAB_WINDOW, ids,
        ["patientunitstayid", "labresultoffset", "labname", "labresult", "labresulttext", "labmeasurenamesystem"],
        string_cols=("labname", "labresulttext", "labmeasurenamesystem", "labmeasurenameinterface"),
        row_filter=lambda df: df.labname.isin(lab_names), name="lab")
    lab["labname"] = lab.labname.astype("category")
    data["lab"] = lab
    labels = set(RESP_LABELS_ALL)
    rc, _ = stream_extract(
        *SOURCE["respiratoryCharting.csv"], "patientunitstayid", "respchartid", "respchartoffset", RESP_WINDOW, ids,
        ["patientunitstayid", "respchartoffset", "respchartvaluelabel", "respchartvalue"],
        string_cols=("respcharttypecat", "respchartvaluelabel", "respchartvalue"),
        row_filter=lambda df: df.respchartvaluelabel.isin(labels), name="respiratoryCharting")
    rc["respchartvaluelabel"] = rc.respchartvaluelabel.astype("category")
    data["resp_charting"] = rc
    pat = "|".join(VASOPRESSORS + SEDATIVES)
    data["infusion"], _ = stream_extract(
        *SOURCE["infusionDrug.csv"], "patientunitstayid", "infusiondrugid", "infusionoffset", INFUSION_WINDOW, ids,
        ["patientunitstayid", "infusionoffset", "drugname"],
        string_cols=("drugname", "drugrate", "infusionrate", "drugamount", "volumeoftestedrug", "patientweight"),
        row_filter=lambda df: df.drugname.str.lower().str.contains(pat, na=False), name="infusionDrug")
    rcare = pd.read_csv(SOURCE["respiratoryCare.csv"][0], dtype=str)
    rcare = rcare[rcare.patientunitstayid.astype(int).isin(ids)]
    audit["respiratoryCare exact duplicates"] = int(rcare.drop(columns="respcareid").duplicated().sum())
    rcare = rcare[["patientunitstayid", "ventstartoffset", "ventendoffset", "respcarestatusoffset"]]
    data["resp_care"] = rcare.apply(pd.to_numeric, errors="coerce")
    aps = pd.read_csv(SOURCE["apacheApsVar.csv"][0])
    data["apache_aps"] = aps[aps.patientunitstayid.isin(ids)].reset_index(drop=True)
    res = pd.read_csv(SOURCE["apachePatientResult.csv"][0], low_memory=False)
    data["apache_result"] = res[res.patientunitstayid.isin(ids)].reset_index(drop=True)
    pv = pd.read_csv(SOURCE["apachePredVar.csv"][0], low_memory=False)
    data["apache_predvar"] = pv[pv.patientunitstayid.isin(ids)].reset_index(drop=True)
    ph = pd.read_csv(SOURCE["pastHistory.csv"][0], dtype=str)
    ph = ph[ph.patientunitstayid.astype(int).isin(ids)]
    audit["pastHistory exact duplicates"] = int(ph.drop(columns="pasthistoryid").duplicated().sum())
    data["past_history"] = ph[["patientunitstayid", "pasthistorypath"]].astype({"patientunitstayid": int})
    dx = pd.read_csv(SOURCE["diagnosis.csv"][0])
    dx = dx[dx.patientunitstayid.isin(ids)]
    audit["diagnosis exact duplicates"] = int(dx.drop(columns="diagnosisid").duplicated().sum())
    audit["diagnosis rows"] = len(dx)
    for k in ("vital_periodic", "vital_aperiodic", "lab", "resp_charting", "infusion"):
        data[k] = _downcast(data[k])
    return data, flow, audit


def load_extraction():
    folder = CACHE_DIR / "extract"
    meta = folder / "meta.pkl"
    if meta.exists() and not FORCE_RECOMPUTE:
        with open(meta, "rb") as f:
            names, flow, audit = pickle.load(f)
        return {n: pd.read_parquet(folder / f"{n}.parquet") for n in names}, flow, audit
    folder.mkdir(exist_ok=True)
    data, flow, audit = run_extraction()
    for n, df in data.items():
        df.to_parquet(folder / f"{n}.parquet", index=False)
    with open(meta, "wb") as f:
        pickle.dump((list(data), flow, audit), f)
    return data, flow, audit


with timer("cohort selection + extraction (cached after the first run)"):
    DATA, COHORT_FLOW, EXTRACT_AUDIT = load_extraction()
show_table(COHORT_FLOW, "Cohort flow")
show_table(pd.DataFrame({"table": list(DATA), "rows": [len(v) for v in DATA.values()],
                         "columns": [v.shape[1] for v in DATA.values()]}), "Extracted raw records (cohort, first 24 h)")

### 1.4 Feature builder
One builder produces both the **raw (naive)** patient-level matrix, with every switch off so labels, units,
sentinel codes and duplicate records stay as recorded, and the **corrected** matrices. Each later section switches
on its own correction, so the effect of each step can be measured on its own:

* `harmonize`: encoding fixes (label synonyms, units, sentinel codes, category codes)
* `dedup_records`: drop exact duplicate source records
* `plausibility`: physiologically impossible values become missing
* `denoise`: a denoiser applied to the 5-minute vital-sign series before the 24-h aggregates are computed

In [ ]:
@dataclass
class BuildOptions:
    harmonize: bool = False
    dedup_records: bool = False
    plausibility: bool = False
    denoise: Optional[Callable] = None


def fahrenheit_to_celsius(t):
    t = t.astype(float).copy()
    f = (t >= 77) & (t <= 113)
    t[f] = (t[f] - 32) * 5.0 / 9.0
    return t


def fix_fio2(v):
    v = v.astype(float).copy()
    frac = (v > 0) & (v <= 1.0)
    v[frac] = v[frac] * 100.0
    return v


def fix_ph(v):
    v = v.astype(float).copy()
    v[(v >= 650) & (v <= 800)] /= 1000.0       # "7410" -> 7.410 (decimal point lost)
    v[(v >= 65) & (v <= 80)] /= 10.0            # "74.1" -> 7.41
    return v


def fix_height(h):
    h = h.astype(float).copy()
    h[(h > 1.2) & (h <= 2.3)] *= 100.0          # metres
    inch = (h >= 48) & (h <= 90)
    h[inch] *= 2.54                             # inches
    return h


def apply_plausibility(df, cols_map):
    counts = {}
    for col, key in cols_map.items():
        lo, hi = PLAUSIBLE[key]
        bad = df[col].notna() & ((df[col] < lo) | (df[col] > hi))
        counts[col] = int(bad.sum())
        df.loc[bad, col] = np.nan
    return df, counts


def subset_data(data, ids):
    ids = set(ids)
    return {k: (v[v.patientunitstayid.isin(ids)].reset_index(drop=True) if "patientunitstayid" in v else v)
            for k, v in data.items()}


def prepare_vital_periodic(vp, opt, audit=None):
    audit = {} if audit is None else audit
    audit.setdefault("duplicates_dropped", {})
    audit.setdefault("implausible_set_nan", {})
    if opt.dedup_records:
        n0 = len(vp)
        vp = vp.drop_duplicates("rowhash")
        audit["duplicates_dropped"]["vitalPeriodic"] = n0 - len(vp)
    vp = vp.sort_values(["patientunitstayid", "observationoffset"]).reset_index(drop=True)
    if opt.harmonize:
        audit["temperature_fahrenheit_rows"] = int(((vp.temperature >= 77) & (vp.temperature <= 113)).sum())
        vp["temperature"] = fahrenheit_to_celsius(vp.temperature).astype(np.float32)
    if opt.plausibility:
        vp, cnt = apply_plausibility(vp, {c: c for c in VITAL_PERIODIC_COLS})
        audit["implausible_set_nan"].update({f"vitalPeriodic.{k}": v for k, v in cnt.items()})
    return vp


def vital_aggregates(vp):
    g = vp.groupby("patientunitstayid")
    agg = pd.DataFrame({
        "hr_mean": g.heartrate.mean(), "hr_min": g.heartrate.min(), "hr_max": g.heartrate.max(),
        "rr_mean": g.respiration.mean(), "rr_min": g.respiration.min(), "rr_max": g.respiration.max(),
        "spo2_mean": g.sao2.mean(), "spo2_min": g.sao2.min(),
        "temp_mean": g.temperature.mean(), "temp_min": g.temperature.min(), "temp_max": g.temperature.max()})
    lt90 = (vp.sao2 < 90).astype(float).where(vp.sao2.notna())
    agg["spo2_frac_below_90"] = lt90.groupby(vp.patientunitstayid).mean()
    return agg


def build_matrix(data, opt, return_audit=False):
    audit = {"implausible_set_nan": {}, "duplicates_dropped": {}}
    p = data["cohort_patient"]
    X = pd.DataFrame({"patientunitstayid": p.patientunitstayid.values})
    for c in ["patienthealthsystemstayid", "uniquepid", "hospitalid", "hospitaldischargeyear", "unitvisitnumber"]:
        X[c] = p[c].values
    if opt.harmonize:
        X["age"] = parse_age(p.age).values
        X["gender"] = p.gender.where(p.gender.isin(["Male", "Female"])).values
        X["ethnicity"] = p.ethnicity.fillna("Other/Unknown").values
        X["region"] = p.region.fillna("Unknown").values
        X["teachingstatus"] = p.teachingstatus.map({"t": 1, "f": 0, True: 1, False: 0}).values
        X["numbedscategory"] = p.numbedscategory.fillna("Unknown").values
        X["admission_height_cm"] = fix_height(p.admissionheight).values
        for col in ["unittype", "unitadmitsource", "hospitaladmitsource"]:
            s = p[col].fillna("Unknown").str.strip()
            share = s.value_counts(normalize=True)
            X[col] = s.where(s.map(share) >= 0.01, "Other").values
    else:
        X["age"] = pd.to_numeric(p.age, errors="coerce").values          # "> 89" silently lost
        X["gender"] = p.gender.values
        X["ethnicity"] = p.ethnicity.values
        X["region"] = p.region.values
        X["teachingstatus"] = p.teachingstatus.values
        X["numbedscategory"] = p.numbedscategory.values
        X["admission_height_cm"] = p.admissionheight.values
        for col in ["unittype", "unitadmitsource", "hospitaladmitsource"]:
            X[col] = p[col].values
    X["admission_weight_kg"] = p.admissionweight.values
    X["resp_dx_group"] = p.resp_dx_group.values
    # outcomes (never used as features)
    X["hospital_mortality"] = (p.hospitaldischargestatus == "Expired").astype(int).values
    X["icu_mortality"] = (p.unitdischargestatus == "Expired").astype(float).where(p.unitdischargestatus.notna()).values
    X["icu_los_days"] = (p.unitdischargeoffset / 1440.0).values
    X["hospital_los_days"] = ((p.hospitaldischargeoffset - p.hospitaladmitoffset) / 1440.0).values

    # ---------------- APACHE
    res = data["apache_result"]
    res = res[res.apacheversion == "IVa"].drop_duplicates("patientunitstayid")
    res = res.set_index("patientunitstayid")[["apachescore", "acutephysiologyscore"]]
    res.columns = ["apache_iva_score", "acute_physiology_score"]
    aps = data["apache_aps"].drop_duplicates("patientunitstayid").set_index("patientunitstayid")
    aps = aps[["eyes", "motor", "verbal", "meds", "urine", "intubated", "vent", "dialysis", "temperature"]]
    aps.columns = ["gcs_eyes", "gcs_motor", "gcs_verbal", "gcs_unable_meds", "urine_output_24h",
                   "aps_intubated", "aps_vent", "aps_dialysis", "aps_temperature_worst"]
    pv = data["apache_predvar"].drop_duplicates("patientunitstayid").set_index("patientunitstayid")
    pv = pv[["diabetes", "immunosuppression", "cirrhosis", "hepaticfailure", "metastaticcancer", "lymphoma",
             "leukemia", "aids", "ventday1"]]
    pv.columns = ["diabetes", "immunosuppression", "cirrhosis", "hepatic_failure", "metastatic_cancer", "lymphoma",
                  "leukemia", "aids", "vent_day1"]
    X = X.join(res, on="patientunitstayid").join(aps, on="patientunitstayid").join(pv, on="patientunitstayid")
    if opt.harmonize:
        sent_cols = ["apache_iva_score", "acute_physiology_score", "gcs_eyes", "gcs_motor", "gcs_verbal",
                     "gcs_unable_meds", "urine_output_24h", "aps_temperature_worst"]
        audit["apache_minus1_sentinels"] = {c: int((X[c] == -1).sum()) for c in sent_cols}
        X[sent_cols] = X[sent_cols].mask(X[sent_cols] == -1)
    X["gcs_total"] = X[["gcs_eyes", "gcs_motor", "gcs_verbal"]].sum(axis=1, min_count=3)

    # ---------------- past history / treatments
    ph = data["past_history"]
    for flag, key in PAST_HISTORY_FLAGS.items():
        ids = set(ph.patientunitstayid[ph.pasthistorypath.str.contains(key, regex=False, na=False)])
        X[flag] = X.patientunitstayid.isin(ids).astype(int)
    inf = data["infusion"]
    dn = inf.drugname.str.lower()
    X["vasopressor_24h"] = X.patientunitstayid.isin(set(inf.patientunitstayid[dn.str.contains("|".join(VASOPRESSORS))])).astype(int)
    X["sedative_infusion_24h"] = X.patientunitstayid.isin(set(inf.patientunitstayid[dn.str.contains("|".join(SEDATIVES))])).astype(int)
    rcare = data["resp_care"]
    vstart = rcare[(rcare.ventstartoffset > 0) & (rcare.ventstartoffset <= 1440)].patientunitstayid
    X["vent_start_charted_24h"] = X.patientunitstayid.isin(set(vstart)).astype(int)

    # ---------------- vital signs (periodic, 5-min)
    vp = prepare_vital_periodic(data["vital_periodic"], opt, audit)
    if opt.denoise is not None:
        vp = opt.denoise(vp)
    X = X.join(vital_aggregates(vp), on="patientunitstayid")

    # ---------------- vital signs (aperiodic, NIBP)
    va = data["vital_aperiodic"]
    if opt.dedup_records:
        n0 = len(va)
        va = va.drop_duplicates("rowhash")
        audit["duplicates_dropped"]["vitalAperiodic"] = n0 - len(va)
    if opt.plausibility:
        va, cnt = apply_plausibility(va.copy(), {c: c for c in VITAL_APERIODIC_COLS})
        audit["implausible_set_nan"].update({f"vitalAperiodic.{k}": v for k, v in cnt.items()})
    g = va.groupby("patientunitstayid")
    X = X.join(pd.DataFrame({
        "nibp_sys_mean": g.noninvasivesystolic.mean(), "nibp_sys_min": g.noninvasivesystolic.min(),
        "nibp_dia_mean": g.noninvasivediastolic.mean(),
        "nibp_map_mean": g.noninvasivemean.mean(), "nibp_map_min": g.noninvasivemean.min()}), on="patientunitstayid")

    # ---------------- laboratory
    lab = data["lab"]
    if opt.dedup_records:
        n0 = len(lab)
        lab = lab.drop_duplicates("rowhash")
        audit["duplicates_dropped"]["lab"] = n0 - len(lab)
    names = dict(LAB_NAMES) if opt.harmonize else {k: v for k, v in LAB_NAMES.items() if k != "bedside glucose"}
    lab = lab[lab.labname.isin(names)].copy()
    lab["var"] = lab.labname.astype(str).map(names)
    lab["val"] = lab.labresult.astype(float)
    if opt.harmonize:
        m = lab["var"] == "lab_fio2"
        audit["lab_fio2_fraction_rows"] = int((m & (lab.val > 0) & (lab.val <= 1)).sum())
        lab.loc[m, "val"] = fix_fio2(lab.loc[m, "val"])
        m = lab["var"] == "ph"
        audit["lab_ph_decimal_rows"] = int((m & (lab.val >= 65) & (lab.val <= 800)).sum())
        lab.loc[m, "val"] = fix_ph(lab.loc[m, "val"])
        audit["bedside_glucose_rows_merged"] = int((lab.labname.astype(str) == "bedside glucose").sum())
    if opt.plausibility:
        bad_total = {}
        for var in lab["var"].unique():
            lo, hi = PLAUSIBLE[var]
            m = (lab["var"] == var) & lab.val.notna() & ((lab.val < lo) | (lab.val > hi))
            bad_total[f"lab.{var}"] = int(m.sum())
            lab.loc[m, "val"] = np.nan
        audit["implausible_set_nan"].update(bad_total)
    piv = lab.pivot_table(index="patientunitstayid", columns="var", values="val", aggfunc=["min", "max"])
    want = {
        "ph_min": ("min", "ph"), "ph_max": ("max", "ph"), "pao2_min": ("min", "pao2"),
        "paco2_max": ("max", "paco2"), "hco3_min": ("min", "hco3"), "base_excess_min": ("min", "base_excess"),
        "lab_fio2_max": ("max", "lab_fio2"), "lactate_max": ("max", "lactate"),
        "sodium_min": ("min", "sodium"), "sodium_max": ("max", "sodium"), "potassium_max": ("max", "potassium"),
        "chloride_max": ("max", "chloride"), "bicarbonate_min": ("min", "bicarbonate"), "bun_max": ("max", "bun"),
        "creatinine_max": ("max", "creatinine"), "glucose_max": ("max", "glucose"), "glucose_min": ("min", "glucose"),
        "calcium_min": ("min", "calcium"), "albumin_min": ("min", "albumin"), "bilirubin_max": ("max", "bilirubin"),
        "wbc_max": ("max", "wbc"), "wbc_min": ("min", "wbc"), "hgb_min": ("min", "hgb"),
        "platelets_min": ("min", "platelets"), "inr_max": ("max", "inr"), "bnp_max": ("max", "bnp"),
        "troponin_i_max": ("max", "troponin_i"), "anion_gap_max": ("max", "anion_gap"),
    }
    labf = pd.DataFrame(index=piv.index)
    for out, key in want.items():
        labf[out] = piv[key] if key in piv.columns else np.nan
    abg = lab[lab["var"].isin(["pao2", "lab_fio2"])].pivot_table(
        index=["patientunitstayid", "labresultoffset"], columns="var", values="val", aggfunc="mean")
    if {"pao2", "lab_fio2"} <= set(abg.columns):
        abg = abg.dropna()
        abg = abg[abg.lab_fio2 > 0]
        labf["pf_ratio_min"] = (abg.pao2 / (abg.lab_fio2 / 100.0)).groupby(level=0).min()
    X = X.join(labf, on="patientunitstayid")

    # ---------------- respiratory charting (ventilator)
    rc = data["resp_charting"]
    if opt.dedup_records:
        n0 = len(rc)
        rc = rc.drop_duplicates("rowhash")
        audit["duplicates_dropped"]["respiratoryCharting"] = n0 - len(rc)
    labels_ = RESP_LABEL_SYNONYMS if opt.harmonize else RESP_LABEL_CANONICAL
    rc = rc[rc.respchartvaluelabel.isin(labels_)].copy()
    rc["var"] = rc.respchartvaluelabel.astype(str).map(labels_)
    if opt.harmonize:
        txt = rc.respchartvalue.astype("string").str.strip().str.rstrip("%")
        rc["val"] = pd.to_numeric(txt, errors="coerce")
        m = rc["var"] == "fio2"
        audit["resp_fio2_fraction_rows"] = int((m & (rc.val > 0) & (rc.val <= 1)).sum())
        audit["resp_percent_string_rows"] = int(rc.respchartvalue.astype("string").str.contains("%", na=False).sum())
        rc.loc[m, "val"] = fix_fio2(rc.loc[m, "val"])
    else:
        rc["val"] = pd.to_numeric(rc.respchartvalue, errors="coerce")
    if opt.plausibility:
        bad_total = {}
        for var in rc["var"].unique():
            lo, hi = PLAUSIBLE[var]
            m = (rc["var"] == var) & rc.val.notna() & ((rc.val < lo) | (rc.val > hi))
            bad_total[f"respiratoryCharting.{var}"] = int(m.sum())
            rc.loc[m, "val"] = np.nan
        audit["implausible_set_nan"].update(bad_total)
    piv = rc.pivot_table(index="patientunitstayid", columns="var", values="val", aggfunc=["mean", "max"])
    want = {
        "rc_fio2_max": ("max", "fio2"), "rc_fio2_mean": ("mean", "fio2"), "rc_peep_max": ("max", "peep"),
        "rc_tidal_volume_set_mean": ("mean", "tidal_volume_set"),
        "rc_peak_insp_pressure_max": ("max", "peak_insp_pressure"), "rc_plateau_pressure_max": ("max", "plateau_pressure"),
        "rc_mean_airway_pressure_mean": ("mean", "mean_airway_pressure"),
        "rc_rr_total_mean": ("mean", "rr_total"), "rc_minute_vent_mean": ("mean", "minute_vent"),
    }
    rcf = pd.DataFrame(index=piv.index)
    for out, key in want.items():
        rcf[out] = piv[key] if key in piv.columns else np.nan
    X = X.join(rcf, on="patientunitstayid")
    vent_settings = set(rc.patientunitstayid[rc["var"].isin(["peep", "tidal_volume_set"]) & rc.val.notna()])
    X["mech_vent_24h"] = (X.patientunitstayid.isin(vent_settings) | (X.aps_vent == 1) | (X.vent_day1 == 1)
                          | (X.vent_start_charted_24h == 1)).astype(int)

    # ---------------- derived + static plausibility
    X["bmi"] = X.admission_weight_kg / (X.admission_height_cm / 100.0) ** 2
    if opt.plausibility:
        static_map = {c: c for c in ["admission_height_cm", "admission_weight_kg", "gcs_eyes", "gcs_motor",
                                     "gcs_verbal", "urine_output_24h"]}
        static_map["aps_temperature_worst"] = "temperature"
        X, cnt = apply_plausibility(X, static_map)
        X["bmi"] = X.admission_weight_kg / (X.admission_height_cm / 100.0) ** 2
        X, cnt2 = apply_plausibility(X, {"bmi": "bmi"})
        audit["implausible_set_nan"].update({f"static.{k}": v for k, v in {**cnt, **cnt2}.items()})
        X["gcs_total"] = X[["gcs_eyes", "gcs_motor", "gcs_verbal"]].sum(axis=1, min_count=3)
        for c in ["rc_fio2_max", "rc_fio2_mean", "lab_fio2_max"]:
            X.loc[X[c] < 21, c] = np.nan
        X.loc[(X.pf_ratio_min < 20) | (X.pf_ratio_min > 700), "pf_ratio_min"] = np.nan
    return (X, audit) if return_audit else X


CATEGORICAL = ["gender", "ethnicity", "unittype", "unitadmitsource", "hospitaladmitsource", "resp_dx_group",
               "numbedscategory"]
BINARY = ["aps_intubated", "aps_vent", "aps_dialysis", "gcs_unable_meds", "diabetes", "immunosuppression",
          "cirrhosis", "hepatic_failure", "metastatic_cancer", "lymphoma", "leukemia", "aids", "vent_day1",
          "vasopressor_24h", "sedative_infusion_24h", "vent_start_charted_24h", "mech_vent_24h",
          "teachingstatus"] + list(PAST_HISTORY_FLAGS)
META = ["patientunitstayid", "patienthealthsystemstayid", "uniquepid", "hospitalid", "region",
        "hospitaldischargeyear", "unitvisitnumber"]


def continuous_columns(X):
    skip = set(CATEGORICAL) | set(BINARY) | set(META) | set(OUTCOME_COLS) | {"split"}
    return [c for c in X.columns if c not in skip and pd.api.types.is_numeric_dtype(X[c])
            and not c.endswith("_was_missing") and not c.startswith("outlier_")
            and c not in ("fairness_weight", "is_synthetic", "stays_per_patient")]


def model_frame(X, drop=()):
    """Feature frame for tree models: numerics + pandas categoricals; NaN kept."""
    skip = set(META) | set(OUTCOME_COLS) | {"split", "stays_per_patient"} | set(drop)
    F_ = X[[c for c in X.columns if c not in skip]].copy()
    for c in F_.columns:
        if not pd.api.types.is_numeric_dtype(F_[c]):
            F_[c] = F_[c].astype("category")
    return F_


def build_cached(name, data, opt, return_audit=False, extra=""):
    key = fingerprint(sorted(data["cohort_patient"].patientunitstayid), opt.harmonize, opt.dedup_records,
                      opt.plausibility, opt.denoise is not None, extra)
    return cached(f"build_{name}", lambda: build_matrix(data, opt, return_audit=return_audit), key)


with timer("raw (naive) feature matrix"):
    RAW, AUDIT_RAW = build_cached("raw", DATA, BuildOptions(), return_audit=True)
print(f"Raw matrix: {RAW.shape[0]:,} ICU stays x {RAW.shape[1]} columns "
      f"({len(continuous_columns(RAW))} continuous features)")
display(RAW.head(3))

## 2. Bias detection on the raw data (all eight biases)
Each bias is decided by a quantitative test on the raw (naive) matrix and raw records. The rules and thresholds
are in the configuration cell (`T`). A bias is **present** when its rule fires. Each later section re-checks its
bias on the data entering that step, because earlier corrections can already remove a bias. Encoding
harmonisation, for example, can remove an apparent demographic bias.

In [ ]:
def detect_missing(X, n_sub=12000):
    feats = list(model_frame(X).columns)
    miss = X[feats].isna().mean().sort_values(ascending=False)
    sentinel_cols = ["apache_iva_score", "acute_physiology_score", "gcs_eyes", "gcs_motor", "gcs_verbal",
                     "urine_output_24h", "gcs_unable_meds"]
    disguised = {c: float((X[c] == -1).mean()) for c in sentinel_cols if c in X}
    affected = miss[miss > T["missing_feature_rate"]]
    sub = X.sample(min(n_sub, len(X)), random_state=SEED)
    F_ = model_frame(sub)
    y = sub[TARGET].values
    mech = []
    for c in affected.index:
        if miss[c] >= 0.98:
            continue
        ind = F_[c].isna().astype(int).values
        # missingness predictable from the observed variables -> MAR/MNAR, not MCAR
        p = cross_val_predict(hgb(max_depth=4, categorical_features="from_dtype"), F_.drop(columns=[c]), ind, cv=3,
                              method="predict_proba")[:, 1]
        tab = pd.crosstab(ind, y)
        chi_p = stats.chi2_contingency(tab)[1] if tab.shape == (2, 2) else np.nan
        mech.append({"feature": c, "missing_rate": miss[c], "missingness_pred_AUC": roc_auc_score(ind, p),
                     "mortality_if_missing": y[ind == 1].mean(), "mortality_if_observed": y[ind == 0].mean(),
                     "outcome_chi2_p": chi_p})
    mech = pd.DataFrame(mech)
    not_mcar = mech[(mech.missingness_pred_AUC > T["missing_mechanism_auc"]) | (mech.outcome_chi2_p < 0.01)]
    return {"present": bool(len(affected) and len(not_mcar)),
            "rule": f">=1 feature with >{T['missing_feature_rate']:.0%} missing AND missingness not MCAR "
                    f"(predictable with AUC>{T['missing_mechanism_auc']} or outcome-associated p<0.01)",
            "missing_cell_rate": float(X[feats].isna().to_numpy().mean()), "n_features": len(feats),
            "n_features_gt_5pct": int((miss > 0.05).sum()), "n_features_gt_70pct": int((miss > 0.7).sum()),
            "disguised_minus1_rate": disguised, "n_not_mcar": int(len(not_mcar)), "per_feature": miss,
            "mechanism_table": mech}


def detect_imbalance(X):
    counts = X[TARGET].value_counts()
    ir = counts.max() / counts.min()
    pk = counts / counts.sum()
    ent = float(-(pk * np.log2(pk)).sum() / np.log2(len(pk)))
    return {"present": bool(ir > T["imbalance_ratio"]), "rule": f"imbalance ratio (majority/minority) > {T['imbalance_ratio']}",
            "class_counts": counts.to_dict(), "minority_share": float(pk.min()), "imbalance_ratio": float(ir),
            "class_entropy": ent}


def implausible_counts(data):
    imp = {}
    vp = data["vital_periodic"]
    for c in VITAL_PERIODIC_COLS:
        lo, hi = PLAUSIBLE[c]
        imp[f"vitalPeriodic.{c}"] = int(((vp[c] < lo) | (vp[c] > hi)).sum())
    va = data["vital_aperiodic"]
    for c in VITAL_APERIODIC_COLS:
        lo, hi = PLAUSIBLE[c]
        imp[f"vitalAperiodic.{c}"] = int(((va[c] < lo) | (va[c] > hi)).sum())
    lab = data["lab"]
    lab_var = lab.labname.astype(str).map(LAB_NAMES)
    for v in lab_var.dropna().unique():
        lo, hi = PLAUSIBLE[v]
        vals = lab.labresult[lab_var == v]
        imp[f"lab.{v}"] = int(((vals < lo) | (vals > hi)).sum())
    n_meas = len(vp) * 4 + len(va) * 3 + len(lab)
    return imp, n_meas


def detect_outliers(X, data):
    imp, n_meas = implausible_counts(data)
    n_bad = sum(imp.values())
    cont = continuous_columns(X)
    Z = X[cont].astype(float)
    med = Z.median()
    mad = ((Z - med).abs().median() * 1.4826).replace(0, np.nan)
    rz = ((Z - med) / mad).abs()
    extreme_rows = float((rz > 4.5).any(axis=1).mean())
    return {"present": bool(n_bad > 0 or extreme_rows > T["outlier_multivariate_rate"]),
            "rule": "any physiologically impossible measurement, or >1% of stays with a robust |z| > 4.5",
            "implausible_measurements": int(n_bad), "measurements_checked": int(n_meas),
            "implausible_by_variable": {k: v for k, v in sorted(imp.items(), key=lambda kv: -kv[1]) if v > 0},
            "stays_with_robust_z_gt_4_5": extreme_rows}


def detect_scale(X):
    Z = X[continuous_columns(X)].astype(float)
    sd = Z.std()
    sd = sd[sd > 0]
    sk = Z.skew()
    ratio = float(sd.max() / sd.min())
    share = float((sk.abs() > 1).mean())
    return {"present": bool(ratio > T["scale_std_ratio"] or share > T["scale_skew_share"]),
            "rule": f"max/min feature SD > {T['scale_std_ratio']:.0f} or >{T['scale_skew_share']:.0%} of features |skew| > 1",
            "std_ratio_max_min": ratio, "largest_sd_feature": sd.idxmax(), "smallest_sd_feature": sd.idxmin(),
            "share_abs_skew_gt_1": share, "mean_abs_skew": float(sk.abs().mean())}


def fairness_metrics(y, score, groups, privileged, threshold=None):
    """Group fairness at a prevalence-matched threshold (same share of positive decisions as positive labels)."""
    y = np.asarray(y)
    score = np.asarray(score)
    groups = np.asarray(groups, dtype=object)
    thr = np.quantile(score, 1 - y.mean()) if threshold is None else threshold
    yhat = (score >= thr).astype(int)
    pm = groups == privileged
    sr_p = yhat[pm].mean()
    tpr_p = yhat[pm & (y == 1)].mean() if (pm & (y == 1)).any() else np.nan
    fpr_p = yhat[pm & (y == 0)].mean() if (pm & (y == 0)).any() else np.nan
    out = []
    for g in pd.unique(groups):
        m = groups == g
        if g == privileged or m.sum() < T["min_group_size"]:
            continue
        sr = yhat[m].mean()
        tpr = yhat[m & (y == 1)].mean() if (m & (y == 1)).any() else np.nan
        fpr = yhat[m & (y == 0)].mean() if (m & (y == 0)).any() else np.nan
        out.append({"group": g, "n": int(m.sum()), "base_rate": float(y[m].mean()), "selection_rate": float(sr),
                    "SPD": float(sr - sr_p), "DI": float(sr / sr_p) if sr_p > 0 else np.nan,
                    "EOD": float(tpr - tpr_p), "FPR_diff": float(fpr - fpr_p),
                    "EqOdds": float(np.nanmax([abs(tpr - tpr_p), abs(fpr - fpr_p)]))})
    return pd.DataFrame(out), float(thr)


def detect_demographic(X, oof):
    y = X[TARGET].values
    res, flagged = {}, []
    for attr in PROTECTED:
        g = X[attr].astype(object).where(X[attr].notna(), "NaN (missing code)").astype(str).values
        tab, _ = fairness_metrics(y, oof, g, PRIVILEGED[attr])
        tab["violates"] = (tab.DI < T["di_low"]) | (tab.DI > T["di_high"]) | (tab.SPD.abs() > T["spd_abs"])
        res[attr] = tab
        flagged += [f"{attr}={r.group} (DI {r.DI:.2f})" for r in tab.itertuples() if r.violates]
    return {"present": bool(flagged),
            "rule": f"a group (n>={T['min_group_size']}) with disparate impact outside [{T['di_low']}, {T['di_high']}] "
                    f"or |SPD|>{T['spd_abs']} (out-of-fold model, prevalence-matched threshold)",
            "violating_groups": flagged, "oof_auc": float(roc_auc_score(y, oof)), "tables": res}


def detect_duplicates(X, data, audit):
    rec = {}
    for name, label_ in [("vital_periodic", "vitalPeriodic"), ("vital_aperiodic", "vitalAperiodic"), ("lab", "lab"),
                         ("resp_charting", "respiratoryCharting"), ("infusion", "infusionDrug")]:
        df = data[name]
        rec[label_] = {"rows": len(df), "exact_duplicates": int(df.rowhash.duplicated().sum()),
                       "rate": float(df.rowhash.duplicated().mean())}
    for k in ["diagnosis", "pastHistory", "respiratoryCare"]:
        if f"{k} exact duplicates" in audit:
            rec[k] = {"exact_duplicates": audit[f"{k} exact duplicates"]}
    n = len(X)
    repeat = 1 - X.uniquepid.nunique() / n
    dup_vec = float(X[list(model_frame(X).columns)].duplicated().mean())
    max_rec = max(v.get("rate", 0) for v in rec.values())
    return {"present": bool(max(max_rec, repeat, dup_vec) > T["duplicate_rate"]),
            "rule": f">{T['duplicate_rate']:.0%} redundant records at record, patient or feature level",
            "record_level": rec, "icu_stays": n, "unique_patients": int(X.uniquepid.nunique()),
            "repeat_stay_rate": float(repeat), "patients_with_multiple_stays": int((X.uniquepid.value_counts() > 1).sum()),
            "identical_feature_vectors_rate": dup_vec}


CLIN_DELTA = {"heartrate": 30, "respiration": 15, "sao2": 10, "temperature": 1.5}


def series_noise_stats(vp, cols=("heartrate", "respiration", "sao2", "temperature"), max_series=6000, seed=SEED):
    """Per-variable residual variance ratio (high-frequency share) and artefact-spike rate of the 5-min series."""
    rng = np.random.default_rng(seed)
    stays = vp.patientunitstayid.unique()
    pick = set(rng.choice(stays, min(max_series, len(stays)), replace=False))
    sub = vp[vp.patientunitstayid.isin(pick)].sort_values(["patientunitstayid", "observationoffset"])
    out = {}
    for c in cols:
        rv, spikes, npts = [], 0, 0
        for _, s in sub.groupby("patientunitstayid")[c]:
            x = s.dropna().to_numpy(dtype=float)
            if len(x) < 24 or np.var(x) == 0:
                continue
            r = x - median_filter(x, size=5, mode="nearest")
            rv.append(np.var(r) / np.var(x))
            spikes += int((np.abs(r) > CLIN_DELTA[c]).sum())
            npts += len(x)
        out[c] = {"series": len(rv), "residual_variance_ratio": float(np.mean(rv)) if rv else np.nan,
                  "artifact_spike_rate": spikes / max(npts, 1)}
    return out


def outcome_sources(data):
    p = data["cohort_patient"].set_index("patientunitstayid")
    res = data["apache_result"]
    res = res[res.apacheversion == "IVa"].drop_duplicates("patientunitstayid").set_index("patientunitstayid")
    pv = data["apache_predvar"].drop_duplicates("patientunitstayid").set_index("patientunitstayid")
    return pd.concat([(p.hospitaldischargestatus == "Expired").astype(float).rename("patient"),
                      res.actualhospitalmortality.map({"EXPIRED": 1.0, "ALIVE": 0.0}).rename("apache_result"),
                      pv.diedinhospital.astype(float).rename("apache_predvar")], axis=1)


def detect_noise(X, data):
    st = series_noise_stats(data["vital_periodic"])
    resid = float(np.nanmean([v["residual_variance_ratio"] for v in st.values()]))
    spike = float(np.nanmean([v["artifact_spike_rate"] for v in st.values()]))
    j = outcome_sources(data).dropna()
    disagree = float(((j.patient != j.apache_result) | (j.patient != j.apache_predvar)).mean())
    p = data["cohort_patient"]
    icu_dead_alive = int(((p.unitdischargestatus == "Expired") & (p.unitdischargelocation == "Death")
                          & (p.hospitaldischargestatus == "Alive")).sum())
    sig = resid > T["noise_residual_var"] or spike > T["noise_artifact_rate"]
    lab = disagree > 0.01 or icu_dead_alive > 0
    return {"present": bool(sig or lab),
            "rule": f"vital-series residual variance ratio > {T['noise_residual_var']} or spike rate > "
                    f"{T['noise_artifact_rate']:.1%}; label noise if outcome sources disagree",
            "signal_noise_present": bool(sig), "label_noise_present": bool(lab), "vital_series": st,
            "mean_residual_variance_ratio": resid, "mean_spike_rate": spike,
            "label_cross_source_disagreement": disagree, "icu_death_but_hospital_alive": icu_dead_alive}


def mmd_rbf(A, B, gamma):
    def k(U, V):
        d = (U ** 2).sum(1)[:, None] + (V ** 2).sum(1)[None, :] - 2 * U @ V.T
        return np.exp(-gamma * np.maximum(d, 0))
    return float(k(A, A).mean() + k(B, B).mean() - 2 * k(A, B).mean())


def mmd_test(A, B, n_perm=200, seed=SEED):
    rng = np.random.default_rng(seed)
    Zs = np.vstack([A, B])
    sq = ((Zs[:300, None, :] - Zs[None, :300, :]) ** 2).sum(-1)
    gamma = 1.0 / np.median(sq[sq > 0])
    obs = mmd_rbf(A, B, gamma)
    n = len(A)
    null = [mmd_rbf(*np.split(Zs[rng.permutation(len(Zs))], [n])[:2], gamma) for _ in range(n_perm)]
    return obs, float((np.sum(np.array(null) >= obs) + 1) / (n_perm + 1))


def domain_auc(R, dom, seed=SEED):
    """Macro one-vs-rest AUC of a classifier that predicts the domain from R (0.5 = no domain signal)."""
    d = pd.Series(dom).astype(str).values
    cv = StratifiedKFold(3, shuffle=True, random_state=seed)
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, random_state=seed), R, d,
                          cv=cv, method="predict_proba")
    return float(roc_auc_score(d, p, multi_class="ovr", average="macro", labels=np.unique(d))), p


def domain_shift_metrics(Xnum, domains, n_per=700, seed=SEED, n_perm=200):
    rng = np.random.default_rng(seed)
    d = pd.Series(domains).astype(str).values
    auc, _ = domain_auc(Xnum, d, seed)
    Xf = np.nan_to_num((Xnum - np.nanmean(Xnum, 0)) / (np.nanstd(Xnum, 0) + 1e-9))
    mmd = {}
    for g in np.unique(d):
        a, b = np.where(d == g)[0], np.where(d != g)[0]
        v, pv = mmd_test(Xf[rng.choice(a, min(n_per, len(a)), replace=False)],
                         Xf[rng.choice(b, min(n_per, len(b)), replace=False)], n_perm=n_perm, seed=seed)
        mmd[g] = {"mmd": v, "p_value": pv, "n": int(len(a))}
    return auc, mmd


def detect_encoding_domain(X, data, harm_audit):
    rc = data["resp_charting"]
    lab_counts = rc.respchartvaluelabel.astype(str).value_counts()
    syn = {k: int(lab_counts.get(k, 0)) for k in RESP_LABEL_SYNONYMS if k not in RESP_LABEL_CANONICAL}
    p = data["cohort_patient"]
    sent = harm_audit.get("apache_minus1_sentinels", {})
    issues = {
        "ventilator values under hospital-specific label synonyms (rows)": sum(syn.values()),
        "glucose results under the separate name 'bedside glucose' (rows)":
            int((data["lab"].labname.astype(str) == "bedside glucose").sum()),
        "temperatures recorded in °F instead of °C (rows)": harm_audit.get("temperature_fahrenheit_rows", 0),
        "FiO2 stored as a fraction instead of % (lab rows)": harm_audit.get("lab_fio2_fraction_rows", 0),
        "FiO2 stored as a fraction instead of % (ventilator rows)": harm_audit.get("resp_fio2_fraction_rows", 0),
        "FiO2 stored as text with a '%' sign (ventilator rows)": harm_audit.get("resp_percent_string_rows", 0),
        "pH with a lost decimal point (lab rows)": harm_audit.get("lab_ph_decimal_rows", 0),
        "APACHE 'not available' coded as -1 (values)": int(sum(sent.values())),
        "age coded as text '> 89' (stays)": int((p.age.astype(str).str.strip() == "> 89").sum()),
        "ethnicity missing instead of 'Other/Unknown' (stays)": int(p.ethnicity.isna().sum()),
        "gender outside Male/Female, e.g. 'Unknown' (stays)": int((~p.gender.isin(["Male", "Female"])).sum()),
    }
    cont = continuous_columns(X)
    auc, mmd = domain_shift_metrics(X[cont].to_numpy(float), X[DOMAIN_COL].astype(object).fillna("Unknown").values)
    n_enc = int(sum(issues.values()))
    min_p = min(v["p_value"] for v in mmd.values())
    return {"present": bool(n_enc > 0 or auc > T["domain_auc"] or min_p < T["mmd_pvalue"]),
            "rule": f"any inconsistent encoding, or region-classifier AUC > {T['domain_auc']}, or MMD p < {T['mmd_pvalue']}",
            "encoding_issue_records": n_enc, "issues": issues, "domain_classifier_auc": auc, "mmd_by_region": mmd,
            "hospitals": int(X.hospitalid.nunique())}


def run_detection():
    _, harm_audit = build_matrix(DATA, BuildOptions(harmonize=True), return_audit=True)
    oof = oof_proba(model_frame(RAW), RAW[TARGET].values)
    det = {"Missing Data Bias": detect_missing(RAW), "Class Imbalance Bias": detect_imbalance(RAW),
           "Outlier Bias": detect_outliers(RAW, DATA), "Feature Scale Bias": detect_scale(RAW),
           "Demographic Bias": detect_demographic(RAW, oof),
           "Duplicate Bias": detect_duplicates(RAW, DATA, EXTRACT_AUDIT),
           "Data Noise Bias": detect_noise(RAW, DATA),
           "Encoding / Domain Bias": detect_encoding_domain(RAW, DATA, harm_audit)}
    return det, harm_audit


with timer("bias detection on the raw data"):
    DET, HARM_AUDIT = cached("detection_raw", run_detection, fingerprint(RAW))

BIASES = ["Missing Data Bias", "Class Imbalance Bias", "Outlier Bias", "Feature Scale Bias", "Demographic Bias",
          "Duplicate Bias", "Data Noise Bias", "Encoding / Domain Bias"]
d = DET  # short alias for the evidence strings below
evidence = {
    "Missing Data Bias": [f"{d['Missing Data Bias']['missing_cell_rate']:.1%} of feature cells missing",
                          f"{d['Missing Data Bias']['n_features_gt_5pct']} of {d['Missing Data Bias']['n_features']} features >5% missing",
                          f"{d['Missing Data Bias']['n_not_mcar']} features with non-random (MAR/MNAR) missingness"],
    "Class Imbalance Bias": [f"mortality {d['Class Imbalance Bias']['minority_share']:.1%}",
                             f"imbalance ratio {d['Class Imbalance Bias']['imbalance_ratio']:.2f} : 1",
                             f"normalised class entropy {d['Class Imbalance Bias']['class_entropy']:.3f}"],
    "Outlier Bias": [f"{d['Outlier Bias']['implausible_measurements']:,} physiologically impossible measurements",
                     f"{d['Outlier Bias']['stays_with_robust_z_gt_4_5']:.1%} of stays with a robust |z| > 4.5 value"],
    "Feature Scale Bias": [f"max/min feature SD ratio {d['Feature Scale Bias']['std_ratio_max_min']:,.0f}",
                           f"({d['Feature Scale Bias']['largest_sd_feature']} vs {d['Feature Scale Bias']['smallest_sd_feature']})",
                           f"{d['Feature Scale Bias']['share_abs_skew_gt_1']:.0%} of features with |skew| > 1"],
    "Demographic Bias": (d["Demographic Bias"]["violating_groups"][:3] or ["no group breaks the four-fifths rule"])
                        + [f"out-of-fold AUC {d['Demographic Bias']['oof_auc']:.3f}"],
    "Duplicate Bias": [f"{d['Duplicate Bias']['repeat_stay_rate']:.1%} of ICU stays are repeat stays",
                       f"of {d['Duplicate Bias']['patients_with_multiple_stays']:,} patients",
                       f"{sum(v['exact_duplicates'] for v in d['Duplicate Bias']['record_level'].values()):,} exact duplicate source records"],
    "Data Noise Bias": [f"vital-series residual variance ratio {d['Data Noise Bias']['mean_residual_variance_ratio']:.2f}",
                        f"artefact-spike rate {d['Data Noise Bias']['mean_spike_rate']:.2%}",
                        f"{d['Data Noise Bias']['icu_death_but_hospital_alive']} labels contradict the ICU death record"],
    "Encoding / Domain Bias": [f"{d['Encoding / Domain Bias']['encoding_issue_records']:,} inconsistently encoded records",
                               f"region classifier AUC {d['Encoding / Domain Bias']['domain_classifier_auc']:.3f}",
                               f"MMD p < 0.01 in {sum(v['p_value'] < 0.01 for v in d['Encoding / Domain Bias']['mmd_by_region'].values())} "
                               f"of {len(d['Encoding / Domain Bias']['mmd_by_region'])} regions"],
}
plot_detection_summary([{"bias": b, "present": DET[b]["present"], "evidence": evidence[b], "rule": DET[b]["rule"]}
                        for b in BIASES])
PRESENCE = pd.DataFrame({"Bias": BIASES, "Detected in raw data": ["present" if DET[b]["present"] else "absent" for b in BIASES],
                         "Evidence": ["; ".join(evidence[b]) for b in BIASES], "Decision rule": [DET[b]["rule"] for b in BIASES]})
save_table(PRESENCE, "00_bias_presence_raw")
show_table(PRESENCE, "Bias presence in the raw data")

## 3. Encoding / Domain Bias, part A: encoding harmonisation
**What the bias is.** The same clinical fact is stored in different ways by different hospitals and EHR systems:
ventilator settings under hospital-specific labels, temperatures in °F, FiO₂ as a fraction instead of a percentage,
pH with a lost decimal point, APACHE's `-1` meaning "not available", ages above 89 stored as the text `"> 89"`,
and missing ethnicity coded differently from `Other/Unknown`. A model then learns the *hospital's coding habits*
instead of the patient's physiology.

**How it is removed.** Each inconsistency is fixed with a deterministic, documented rule: map the label synonym,
convert the unit, turn the sentinel into a real missing value, decode the text. A rule is either right or wrong, so
there is no method to compare for this part. The **domain** part of this bias (systematic differences between
hospital regions) is handled with the table's methods (IRM, OTDA, Domain Generalization Transformer) in
**section 10**, once the features are clean and scaled.

In [ ]:
def encoding_rules(X):
    """Cell-level rules that flag inconsistently encoded values in a patient-level matrix."""
    m = pd.DataFrame(False, index=X.index, columns=X.columns)
    temp_cols = [c for c in ["temp_mean", "temp_min", "temp_max", "aps_temperature_worst"] if c in X]
    m[temp_cols] = (X[temp_cols] >= 77) & (X[temp_cols] <= 113)                   # °F
    fio2_cols = [c for c in ["lab_fio2_max", "rc_fio2_max", "rc_fio2_mean"] if c in X]
    m[fio2_cols] = (X[fio2_cols] > 0) & (X[fio2_cols] <= 1)                       # fraction instead of %
    ph_cols = [c for c in ["ph_min", "ph_max"] if c in X]
    m[ph_cols] = X[ph_cols] >= 65                                                 # decimal point lost
    sent_cols = [c for c in ["apache_iva_score", "acute_physiology_score", "gcs_eyes", "gcs_motor", "gcs_verbal",
                             "gcs_unable_meds", "urine_output_24h", "aps_temperature_worst"] if c in X]
    m[sent_cols] |= X[sent_cols] == -1                                            # APACHE "not available"
    m["age"] = X["age"].isna()                                                    # "> 89" lost by a numeric reader
    h = X["admission_height_cm"]
    m["admission_height_cm"] = ((h > 1.2) & (h <= 2.3)) | ((h >= 48) & (h <= 90))   # metres / inches
    m["ethnicity"] = X["ethnicity"].isna()
    m["gender"] = X["gender"].notna() & ~X["gender"].isin(["Male", "Female"])
    return m.fillna(False).astype(bool)


with timer("harmonised feature matrix"):
    X_ENC, AUDIT_ENC = build_cached("harmonized", DATA, BuildOptions(harmonize=True), return_audit=True)

num_cols = [c for c in X_ENC.columns if c not in set(META) | set(OUTCOME_COLS) and pd.api.types.is_numeric_dtype(X_ENC[c])
            and c in RAW.columns and pd.api.types.is_numeric_dtype(RAW[c])]
a_, b_ = RAW[num_cols].astype(float), X_ENC[num_cols].astype(float)
changed = ~(np.isclose(a_, b_, rtol=1e-6, atol=1e-6) | (a_.isna() & b_.isna()))
recovered = a_.isna() & b_.notna()          # values a naive reader misses (label synonyms, bedside glucose, "> 89")
ENC_MASK_BEFORE = encoding_rules(RAW)
ENC_MASK_BEFORE[num_cols] |= recovered
ENC_MASK_AFTER = encoding_rules(X_ENC)

issues = DET["Encoding / Domain Bias"]["issues"]
after_counts = {
    "temperatures recorded in °F instead of °C (rows)": int(ENC_MASK_AFTER[["temp_mean", "temp_min", "temp_max"]].to_numpy().sum()),
    "FiO2 stored as a fraction instead of % (lab rows)": int(ENC_MASK_AFTER["lab_fio2_max"].sum()),
    "FiO2 stored as a fraction instead of % (ventilator rows)": int(ENC_MASK_AFTER[["rc_fio2_max", "rc_fio2_mean"]].to_numpy().sum()),
    "pH with a lost decimal point (lab rows)": int(ENC_MASK_AFTER[["ph_min", "ph_max"]].to_numpy().sum()),
    "APACHE 'not available' coded as -1 (values)": int(sum((X_ENC[c] == -1).sum() for c in ["apache_iva_score", "acute_physiology_score", "gcs_eyes", "gcs_motor", "gcs_verbal", "gcs_unable_meds", "urine_output_24h", "aps_temperature_worst"])),
    "age coded as text '> 89' (stays)": int((X_ENC.age.isna() & (DATA["cohort_patient"].age.astype(str).str.strip() == "> 89").values).sum()),
    "ethnicity missing instead of 'Other/Unknown' (stays)": int(X_ENC.ethnicity.isna().sum()),
    "gender outside Male/Female, e.g. 'Unknown' (stays)": int((X_ENC.gender.notna() & ~X_ENC.gender.isin(["Male", "Female"])).sum()),
}
ENC_TABLE = pd.DataFrame({"Encoding inconsistency": list(issues), "Records before": list(issues.values()),
                          "Records after harmonisation": [after_counts.get(k, 0) for k in issues],
                          "Fix (rule)": ["map to the canonical concept", "merge into 'glucose'", "(°F - 32) x 5/9",
                                         "x 100", "x 100", "strip '%' and parse", "/10 or /1000", "set to missing",
                                         "decode as 90", "merge into 'Other/Unknown'", "set to missing"]})
save_table(ENC_TABLE, "03_encoding_issues")
show_table(ENC_TABLE, "Encoding inconsistencies found in the raw records and what the harmonisation did")

fig, ax = plt.subplots(figsize=(14, 5.2), facecolor="white")
order = np.argsort(ENC_TABLE["Records before"].to_numpy())
y = np.arange(len(order))
vals = ENC_TABLE["Records before"].to_numpy()[order]
ax.barh(y, np.maximum(vals, 0.8), color=to_rgba(C_BEFORE, 0.75), height=0.55, label="Before (as recorded)")
for yi, v, v2 in zip(y, vals, ENC_TABLE["Records after harmonisation"].to_numpy()[order]):
    ax.text(max(v, 1) * 1.15, yi, f"{v:,}  →  after: {v2:,}", va="center", fontsize=9.5, color=INK)
ax.set_xscale("log")
ax.set_yticks(y, ENC_TABLE["Encoding inconsistency"].to_numpy()[order], fontsize=9.5, color=INK)
ax.set_xlim(0.8, vals.max() * 40)
ax.set_xlabel("records (log scale)")
ax.set_title("Encoding inconsistencies by type: before harmonisation (bars) and after (all 0)", loc="left", fontsize=13)
ax.grid(axis="y", visible=False)
fig.tight_layout()
show_fig(fig, "03a_encoding_issue_counts")

**Before vs after dashboard.** *Before* is the matrix a naive reader produces; *after* is the harmonised matrix. The
ten variables shown are the ones whose values the harmonisation changed most. In the indicator heatmap, a filled
square is a cell whose raw value was inconsistently encoded or lost.

In [ ]:
change_rate = changed.mean().sort_values(ascending=False)
ENC_VARS = [c for c in change_rate.index if c in continuous_columns(X_ENC) or c == "age"][:10]
qq_enc = max(ENC_VARS[:6], key=lambda c: abs(RAW[c].skew() - X_ENC[c].skew()) if RAW[c].notna().sum() > 30 else 0)
ib, ia = sample_indicator(ENC_MASK_BEFORE[ENC_VARS], ENC_MASK_AFTER[ENC_VARS])
bias_dashboard(
    "Encoding / Domain Bias (part A: encoding)", "Rule-based harmonisation", RAW, X_ENC, ENC_VARS,
    present=DET["Encoding / Domain Bias"]["encoding_issue_records"] > 0, qq_var=qq_enc,
    before_name="raw encoded values", after_name="harmonised values",
    indicator=dict(before=ib, after=ia, rows=[label(v) for v in ENC_VARS], name="inconsistently encoded",
                   title="Encoding-Issue Heatmap (40 randomly sampled ICU stays)",
                   footer=f"Whole cohort: {ENC_MASK_BEFORE[ENC_VARS].to_numpy().mean():.1%} of these cells were inconsistently encoded "
                          f"before, {ENC_MASK_AFTER[ENC_VARS].to_numpy().mean():.1%} after."),
    note="Share of stays whose value changed: " + ", ".join(f"{label(c)} {change_rate[c]:.0%}" for c in ENC_VARS[:6]) + ".",
    fname="03b_dashboard_encoding")

## 4. Duplicate Bias
**What the bias is.** The bias works at two levels:
* **Record level:** the same measurement is stored twice (identical rows), which inflates counts and shifts means.
  Exact duplicates are found by their row hash and dropped. This is unambiguous, so no method comparison is needed.
* **Patient level:** about one ICU stay in five is a *repeat stay* of a patient who is already in the data. These
  patients are over-represented, and the same person can land in both the training and the test data. Finding the
  stays that belong to one person is an **entity-resolution** problem, so the three methods from the table are
  compared here.

**Evaluation protocol.** The eICU patient key `uniquepid` is used **only as ground truth for scoring**; the methods
never see it. Hospitals are split in half. The supervised matchers (Ditto, GNN) and the LSH similarity threshold are
trained or tuned on record pairs from one half, and all three methods are scored on the other half.

| Metric | Meaning | Better |
|---|---|---|
| Duplicate Rate | share of the remaining stays that are still repeat stays (after the method's de-duplication) | lower |
| Redundancy Reduction | share of the truly redundant stays that the method removed correctly | higher |
| Precision / Recall / F1-score | pairwise: predicted same-patient pairs vs true pairs | higher |
| Pair Completeness | share of the true pairs inside the method's candidate set (blocking / hashing) | higher |
| Reduction Ratio | 1 - candidate pairs / all possible pairs (how much comparison work is saved) | higher |

In [ ]:
with timer("record-level de-duplication"):
    X_REC, AUDIT_REC = build_cached("harm_dedup", DATA, BuildOptions(harmonize=True, dedup_records=True),
                                    return_audit=True)
rec = DET["Duplicate Bias"]["record_level"]
used_in_features = {"vitalPeriodic": "24-h vital-sign aggregates", "vitalAperiodic": "24-h blood-pressure aggregates",
                    "lab": "24-h laboratory aggregates", "respiratoryCharting": "ventilator settings"}
REC_TABLE = pd.DataFrame([{"source table": k, "exact duplicate rows (before)": v["exact_duplicates"],
                           "dropped by row hash": AUDIT_REC["duplicates_dropped"].get(k, 0),
                           "after": v["exact_duplicates"] - AUDIT_REC["duplicates_dropped"].get(k, 0) if k in used_in_features else "-",
                           "feeds": used_in_features.get(k, "binary flags only (a duplicate cannot change a yes/no flag)")}
                          for k, v in rec.items()])
show_table(REC_TABLE, "Record-level exact duplicates")

### 4.1 The three entity-resolution methods
* **LSH (locality-sensitive hashing).** Random-hyperplane SimHash over each stay's static descriptors (age, height,
  weight, sex, ethnicity, hospital, comorbidities). Stays that collide in a hash band become candidates. A candidate
  pair is a match when its cosine similarity passes a threshold tuned on the training hospitals.
* **GNN entity resolution.** A GraphSAGE network (PyTorch Geometric) on the blocking graph. Nodes are stays and edges
  are candidate pairs. A link-prediction head scores each edge from the node embeddings plus pair features.
* **Ditto** (Li et al., VLDB 2021). Each stay is serialised as text (`COL age VAL 67 COL sex VAL male ...`), and a
  pre-trained DistilBERT is fine-tuned to classify record pairs, with Ditto's MixDA augmentation (span deletion).
  CPU budget: 1,200 training pairs, 2 epochs, max length 128. Scoring uses int8 dynamic quantisation (about 2x
  faster), and the decision threshold is tuned on validation pairs with that same int8 scorer. The fine-tuned
  weights are saved, so later runs do not retrain.

Candidate pairs for GNN and Ditto come from the same **blocking**: same hospital, sex and ethnicity, ages within 1
year, heights within 10 cm and weights within 40%. On the whole cohort this keeps 80.5% of the true same-patient
pairs with 99k candidates (allowing 2 years keeps 82.0% but needs 157k candidates).

In [ ]:
HX_NAMES = {"hx_copd": "copd", "hx_asthma": "asthma", "hx_home_o2": "home oxygen",
            "hx_restrictive_lung": "restrictive lung disease", "hx_resp_failure": "respiratory failure",
            "hx_chf": "chf", "hx_hypertension": "hypertension", "hx_renal": "renal disease", "hx_cancer": "cancer",
            "diabetes": "diabetes", "cirrhosis": "cirrhosis", "immunosuppression": "immunosuppression",
            "metastatic_cancer": "metastatic cancer", "lymphoma": "lymphoma", "leukemia": "leukemia", "aids": "aids"}


def pairs_from_groups(labels_):
    s = pd.Series(np.arange(len(labels_)), index=pd.Index(labels_))
    s = s[s.index.duplicated(keep=False)]
    out = []
    for _, idx in s.groupby(level=0):
        v = np.sort(idx.values)
        i, j = np.triu_indices(len(v), 1)
        out.append(np.column_stack([v[i], v[j]]))
    return np.vstack(out) if out else np.empty((0, 2), int)


def pairset(P):
    return set(map(tuple, np.sort(np.asarray(P), axis=1)))


def candidate_pairs(X, max_age_gap=1, max_height_gap=10.0, max_weight_ratio=0.4):
    """Blocking: same hospital, sex and ethnicity; compatible age, height and weight."""
    keys = X.groupby(["hospitalid", "gender", "ethnicity"], dropna=False).ngroup().to_numpy()
    age = X.age.to_numpy(float)
    h = X.admission_height_cm.to_numpy(float)
    w = X.admission_weight_kg.to_numpy(float)
    out = []
    for _, idx in pd.Series(np.arange(len(X))).groupby(keys):
        v = idx.to_numpy()
        if len(v) < 2:
            continue
        i, j = np.triu_indices(len(v), 1)
        a, b = v[i], v[j]
        ok = np.abs(age[a] - age[b]) <= max_age_gap
        dh = np.abs(h[a] - h[b])
        ok &= np.isnan(dh) | (dh <= max_height_gap)
        rw = np.abs(w[a] - w[b]) / np.fmax(w[a], w[b])
        ok &= np.isnan(rw) | (rw <= max_weight_ratio)
        out.append(np.column_stack([a[ok], b[ok]]))
    return np.vstack(out) if out else np.empty((0, 2), int)


def pair_labels(X, P):
    pid = X.uniquepid.to_numpy()
    return (pid[P[:, 0]] == pid[P[:, 1]]).astype(int)


def best_threshold(y, p, grid=np.arange(0.05, 1.0, 0.05)):
    best, th = -1.0, 0.5
    for t in grid:
        f = f1_score(y, (p > t).astype(int), zero_division=0)
        if f > best:
            best, th = f, float(t)
    return best, th


# ---------------------------------------------------------------- LSH
def lsh_vectors(X):
    num = X[["age", "admission_height_cm", "admission_weight_kg"]].astype(float)
    num = ((num - num.mean()) / num.std()).fillna(0).to_numpy()
    cat = pd.get_dummies(X[["gender", "ethnicity"]].astype(object).fillna("NA")).to_numpy(float) * 2
    hosp = pd.get_dummies(X.hospitalid.astype(str)).to_numpy(float) * 3
    V = np.hstack([num, cat, hosp, X[list(HX_NAMES)].fillna(0).to_numpy(float)])
    return V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)


def lsh_candidates(V, bits=18, bands=8, seed=SEED):
    """SimHash banding: pairs that share a bucket in at least one band."""
    rng = np.random.default_rng(seed)
    n = len(V)
    codes = []
    for _ in range(bands):
        sig = (V @ rng.standard_normal((V.shape[1], bits)) > 0).astype(np.int64) @ (1 << np.arange(bits))
        P = pairs_from_groups(sig)
        if len(P):
            codes.append(P[:, 0].astype(np.int64) * n + P[:, 1])
    u = np.unique(np.concatenate(codes)) if codes else np.empty(0, np.int64)
    return np.column_stack([u // n, u % n])


def lsh_cosine(V, P):
    return (V[P[:, 0]] * V[P[:, 1]]).sum(1)


# ---------------------------------------------------------------- GNN entity resolution
NODE_NUM = ["age", "admission_height_cm", "admission_weight_kg", "hospitaldischargeyear"]
NODE_CAT = ["gender", "ethnicity", "unittype", "resp_dx_group"]


def node_features(X, ref=None):
    num = X[NODE_NUM].astype(float)
    cat = X[NODE_CAT].astype(object).fillna("NA").astype(str)
    if ref is None:
        ref = {"mean": num.mean(), "std": num.std(), "levels": {c: sorted(cat[c].unique()) for c in NODE_CAT}}
    Z = ((num - ref["mean"]) / ref["std"]).fillna(0).to_numpy()
    onehot = np.hstack([(cat[c].to_numpy()[:, None] == np.array(ref["levels"][c])[None, :]).astype(float)
                        for c in NODE_CAT])
    hx = X[list(HX_NAMES)].fillna(0).to_numpy(float)
    return np.hstack([Z, num.isna().to_numpy(float), onehot, hx]).astype(np.float32), ref


def edge_features(X, P):
    a, b = P[:, 0], P[:, 1]
    age, h, w = (X[c].to_numpy(float) for c in ("age", "admission_height_cm", "admission_weight_kg"))
    yr = X.hospitaldischargeyear.to_numpy(float)
    hx = X[list(HX_NAMES)].fillna(0).to_numpy(bool)
    dh, rw = np.abs(h[a] - h[b]), np.abs(w[a] - w[b]) / np.fmax(w[a], w[b])
    inter, union = (hx[a] & hx[b]).sum(1), (hx[a] | hx[b]).sum(1)
    same = lambda col: (X[col].astype(str).to_numpy()[a] == X[col].astype(str).to_numpy()[b]).astype(float)
    return np.column_stack([np.abs(age[a] - age[b]) / 2.0, np.nan_to_num(dh / 5.0), np.isnan(dh).astype(float),
                            np.nan_to_num(rw / 0.25), np.isnan(rw).astype(float),
                            np.where(union == 0, 1.0, inter / np.maximum(union, 1)), (yr[a] == yr[b]).astype(float),
                            same("unittype"), same("resp_dx_group")]).astype(np.float32)


class GraphSAGEMatcher(nn.Module):
    def __init__(self, d_node, d_edge, hidden=128, dropout=0.2):
        super().__init__()
        from torch_geometric.nn import SAGEConv
        self.inp = nn.Linear(d_node, hidden)
        self.c1, self.c2 = SAGEConv(hidden, hidden), SAGEConv(hidden, hidden)
        self.drop = dropout
        self.edge = nn.Sequential(nn.Linear(2 * hidden + d_edge, hidden), nn.ReLU(), nn.Dropout(dropout),
                                  nn.Linear(hidden, 1))

    def forward(self, x, ei, pairs, ef):
        h = F.relu(self.inp(x))
        h = F.dropout(F.relu(self.c1(h, ei)), self.drop, self.training)
        h = self.c2(h, ei)
        hu, hv = h[pairs[:, 0]], h[pairs[:, 1]]
        return self.edge(torch.cat([hu * hv, (hu - hv).abs(), ef], 1)).squeeze(1)


def _graph(X, P, ref=None):
    x, ref = node_features(X, ref)
    ei = torch.as_tensor(np.vstack([np.r_[P[:, 0], P[:, 1]], np.r_[P[:, 1], P[:, 0]]]), dtype=torch.long)
    return torch.as_tensor(x), ei, torch.as_tensor(P, dtype=torch.long), torch.as_tensor(edge_features(X, P)), ref


def train_gnn(X_tr, P_tr, y_tr, epochs=300, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    x, ei, pairs, ef, ref = _graph(X_tr, P_tr)
    y = torch.as_tensor(y_tr, dtype=torch.float32)
    val = rng.random(len(P_tr)) < 0.2
    model = GraphSAGEMatcher(x.shape[1], ef.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    pos_w = torch.tensor(float((y_tr[~val] == 0).sum() / max((y_tr[~val] == 1).sum(), 1)))
    tr_idx = torch.as_tensor(np.flatnonzero(~val))
    for ep in range(epochs):
        model.train()
        loss = F.binary_cross_entropy_with_logits(model(x, ei, pairs[tr_idx], ef[tr_idx]), y[tr_idx], pos_weight=pos_w)
        opt.zero_grad()
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        pv = torch.sigmoid(model(x, ei, pairs[np.flatnonzero(val)], ef[np.flatnonzero(val)])).numpy()
    f1v, th = best_threshold(y_tr[val], pv)
    return model, th, ref, f1v


def predict_gnn(model, X, P, ref):
    x, ei, pairs, ef, _ = _graph(X, P, ref)
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(x, ei, pairs, ef)).numpy()


# ---------------------------------------------------------------- Ditto
def _fmt(v, digits=0):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if isinstance(v, (float, np.floating)):
        return f"{v:.{digits}f}"
    return str(v).lower()


def serialize(X):
    """Ditto serialisation: 'COL <attribute> VAL <value>' for every attribute."""
    hx = list(HX_NAMES)
    H = X[hx].fillna(0).to_numpy(bool)
    out = []
    for k, r in enumerate(X.itertuples(index=False)):
        hist = ", ".join(HX_NAMES[c] for c, f in zip(hx, H[k]) if f)
        out.append(" ".join(["COL age VAL", _fmt(r.age), "COL sex VAL", _fmt(r.gender), "COL ethnicity VAL",
                             _fmt(r.ethnicity), "COL height VAL", _fmt(r.admission_height_cm), "COL weight VAL",
                             _fmt(r.admission_weight_kg, 1), "COL discharge year VAL", _fmt(r.hospitaldischargeyear),
                             "COL unit VAL", _fmt(r.unittype), "COL diagnosis VAL",
                             _fmt(r.resp_dx_group).replace("_", " "), "COL history VAL", hist]))
    return out


def ditto_augment(text, rng):
    """Ditto's `del` operator: swap the two records with p = 0.5, then delete a 1-2 token span (never COL/VAL)."""
    if " [SEP] " in text and rng.random() < 0.5:
        left, right = text.split(" [SEP] ")
        text = right + " [SEP] " + left
    tokens = text.split(" ")
    kinds = ["HD" if t in ("COL", "VAL") else "SEP" if t == "[SEP]" else "O" for t in tokens]
    span = int(rng.integers(1, 3))
    cand = [i for i in range(len(tokens) - span + 1) if all(k == "O" for k in kinds[i:i + span])]
    if cand:
        p = cand[int(rng.integers(len(cand)))]
        tokens = tokens[:p] + tokens[p + span:]
    return " ".join(tokens)


class DittoModel(nn.Module):
    """Pre-trained LM + linear layer on [CLS]; MixDA interpolates the [CLS] encodings of the original and the
    augmented pair with lambda ~ Beta(0.8, 0.8)."""

    def __init__(self, lm="distilbert-base-uncased", alpha_aug=0.8):
        super().__init__()
        from transformers import AutoModel
        self.bert = AutoModel.from_pretrained(lm)
        self.alpha_aug = alpha_aug
        self.fc = nn.Linear(self.bert.config.hidden_size, 2)

    def _cls(self, ids, mask):
        return self.bert(input_ids=ids, attention_mask=mask)[0][:, 0, :]

    def forward(self, ids, mask, ids_aug=None, mask_aug=None):
        if ids_aug is None:
            return self.fc(self._cls(ids, mask))
        n, L = len(ids), max(ids.shape[1], ids_aug.shape[1])
        pad = lambda t: F.pad(t, (0, L - t.shape[1]))
        enc = self._cls(torch.cat([pad(ids), pad(ids_aug)]), torch.cat([pad(mask), pad(mask_aug)]))
        lam = float(np.random.beta(self.alpha_aug, self.alpha_aug))
        return self.fc(enc[:n] * lam + enc[n:] * (1.0 - lam))


def _encode(tok, pairs, max_len=128):
    e = tok([p[0] for p in pairs], [p[1] for p in pairs], max_length=max_len, truncation=True, padding=True,
            return_tensors="pt")
    return e["input_ids"], e["attention_mask"]


def ditto_predict(model, tok, pairs, batch=128):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(pairs), batch):
            ids, mask = _encode(tok, pairs[i:i + batch])
            out.append(model(ids, mask).softmax(1)[:, 1].numpy())
    return np.concatenate(out) if out else np.zeros(0)


def load_tokenizer_and_model():
    from transformers import AutoTokenizer
    try:
        os.environ["HF_HUB_OFFLINE"] = "1"          # use the local HuggingFace cache when it has the model
        return AutoTokenizer.from_pretrained("distilbert-base-uncased"), DittoModel()
    except Exception:
        os.environ["HF_HUB_OFFLINE"] = "0"          # otherwise download it once
        return AutoTokenizer.from_pretrained("distilbert-base-uncased"), DittoModel()


def train_ditto(train_pairs, train_y, valid_pairs, valid_y, epochs=2, batch=32, lr=3e-5, seed=SEED):
    from transformers import get_linear_schedule_with_warmup
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tok, model = load_tokenizer_and_model()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    sched = get_linear_schedule_with_warmup(opt, 0, (len(train_pairs) // batch) * epochs)
    y_t = torch.as_tensor(np.asarray(train_y), dtype=torch.long)
    best_f1, best_th, best_state, hist = -1.0, 0.5, None, []
    for ep in range(1, epochs + 1):
        model.train()
        order = rng.permutation(len(train_pairs))
        for i in range(0, len(order), batch):
            b = order[i:i + batch]
            pairs = [train_pairs[j] for j in b]
            ids, mask = _encode(tok, pairs)
            aug = [ditto_augment(l + " [SEP] " + r, rng).split(" [SEP] ") for l, r in pairs]
            ids_a, mask_a = _encode(tok, [(a[0], a[1] if len(a) > 1 else "") for a in aug])
            loss = F.cross_entropy(model(ids, mask, ids_a, mask_a), y_t[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
        f1v, th = best_threshold(np.asarray(valid_y), ditto_predict(model, tok, valid_pairs))
        hist.append({"epoch": ep, "valid_F1": f1v, "threshold": th})
        log(f"[Ditto] epoch {ep}: validation F1 {f1v:.3f} at threshold {th:.2f}")
        if f1v > best_f1:
            best_f1, best_th = f1v, th
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    return model, tok, best_th, best_f1, hist


def quantize(model):
    """int8 dynamic quantisation of the Linear layers: about 2x faster scoring on CPU."""
    return torch.ao.quantization.quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)


# ---------------------------------------------------------------- evaluation
def keep_order(X):
    """First stay per patient: earliest discharge year, hospitalisation and unit visit."""
    return np.lexsort((X.patientunitstayid.values, X.unitvisitnumber.values, X.patienthealthsystemstayid.values,
                       X.hospitaldischargeyear.values))


def dedup_by_matches(X, P):
    """Transitive closure of the predicted matches; keep the first stay of every resolved entity."""
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components
    n = len(X)
    if len(P):
        _, lab_ = connected_components(coo_matrix((np.ones(len(P)), (P[:, 0], P[:, 1])), shape=(n, n)), directed=False)
    else:
        lab_ = np.arange(n)
    rank = np.empty(n, int)
    rank[keep_order(X)] = np.arange(n)
    keep = pd.DataFrame({"lab": lab_, "rank": rank}).sort_values("rank").drop_duplicates("lab").index.values
    return np.sort(keep)


def er_metrics(name, P_pred, P_cand, truth, X):
    pred = pairset(P_pred) if len(P_pred) else set()
    cand = pairset(P_cand) if len(P_cand) else set()
    tp = len(pred & truth)
    prec = tp / len(pred) if pred else 0.0
    rec_ = tp / len(truth) if truth else 1.0
    n = len(X)
    keep = dedup_by_matches(X, np.asarray(P_pred).reshape(-1, 2))
    kept = X.iloc[keep]
    pid = X.uniquepid.values
    true_redundant = n - len(np.unique(pid))
    kept_p = set(kept.uniquepid)
    removed = np.setdiff1d(np.arange(n), keep)
    correct = sum(1 for i in removed if pid[i] in kept_p)
    return {"Method": name, "Duplicate Rate": float(kept.uniquepid.duplicated().mean()),
            "Redundancy Reduction": correct / true_redundant if true_redundant else 1.0,
            "Precision": prec, "Recall": rec_, "F1-score": 2 * prec * rec_ / (prec + rec_) if prec + rec_ else 0.0,
            "Pair Completeness": len(cand & truth) / len(truth) if truth else 1.0,
            "Reduction Ratio": 1 - len(cand) / (n * (n - 1) / 2),
            "predicted pairs": len(pred), "candidate pairs": len(cand), "stays removed": int(len(removed)),
            "unique patients lost": int(len(set(pid) - kept_p))}


DITTO_N_TRAIN, DITTO_N_VALID, DITTO_EPOCHS = 1200, 600, 2
DITTO_STATE, DITTO_META = CACHE_DIR / "ditto_finetuned_state.pt", CACHE_DIR / "ditto_finetuned_meta.json"


def ditto_finetuned(train_pairs, train_y, valid_pairs, valid_y, config):
    """Fine-tune Ditto once (about 16 min on a laptop CPU); later runs reload the saved weights."""
    if DITTO_STATE.exists() and DITTO_META.exists() and not FORCE_RECOMPUTE:
        meta = json.loads(DITTO_META.read_text())
        if meta["config"] == config:
            tok, model = load_tokenizer_and_model()
            model.load_state_dict(torch.load(DITTO_STATE, weights_only=False))
            log("[Ditto] fine-tuned weights loaded from the cache")
            return tok, model.eval(), meta["history"]
    model, tok, _, _, hist = train_ditto(train_pairs, train_y, valid_pairs, valid_y, epochs=config["epochs"])
    torch.save(model.state_dict(), DITTO_STATE)
    DITTO_META.write_text(json.dumps({"config": config, "history": hist}))
    return tok, model, hist


def run_duplicate_benchmark(X, key):
    X = X.reset_index(drop=True)
    rng = np.random.default_rng(SEED)
    hosp = np.array(sorted(X.hospitalid.unique()))
    rng.shuffle(hosp)
    h_train = set(hosp[: len(hosp) // 2])
    tr = X[X.hospitalid.isin(h_train)].reset_index(drop=True)
    ev = X[~X.hospitalid.isin(h_train)].reset_index(drop=True)
    truth = pairset(pairs_from_groups(ev.uniquepid.values))
    info = {"training hospitals": len(h_train), "evaluation hospitals": int(ev.hospitalid.nunique()),
            "evaluation stays": len(ev), "evaluation true same-patient pairs": len(truth)}
    rows = []
    # ---- LSH (threshold tuned on the training hospitals)
    with timer("LSH"):
        V_tr, V_ev = lsh_vectors(tr), lsh_vectors(ev)
        C_tr, C_ev = lsh_candidates(V_tr), lsh_candidates(V_ev)
        f1_l, th_l = best_threshold(pair_labels(tr, C_tr), lsh_cosine(V_tr, C_tr), grid=np.arange(0.90, 0.999, 0.005))
        s_ev = lsh_cosine(V_ev, C_ev)
        rows.append(er_metrics("LSH", C_ev[s_ev > th_l], C_ev, truth, ev))
        info.update({"LSH cosine threshold (tuned)": th_l, "LSH training F1": f1_l})
    # ---- blocking for the supervised matchers
    P_tr, P_ev = candidate_pairs(tr), candidate_pairs(ev)
    y_tr = pair_labels(tr, P_tr)
    info.update({"blocking candidate pairs (train / eval)": f"{len(P_tr):,} / {len(P_ev):,}",
                 "training candidate matches": int(y_tr.sum())})
    with timer("GNN entity resolution (GraphSAGE)"):
        gnn, th_g, ref_g, f1_g = cached("er_gnn", lambda: train_gnn(tr, P_tr, y_tr), key)
        p_g = predict_gnn(gnn, ev, P_ev, ref_g)
        rows.append(er_metrics("GNN entity resolution", P_ev[p_g > th_g], P_ev, truth, ev))
        info.update({"GNN threshold": th_g, "GNN validation F1": f1_g})
    with timer("Ditto (fine-tune DistilBERT, score evaluation pairs)"):
        pos, neg = np.flatnonzero(y_tr == 1), np.flatnonzero(y_tr == 0)
        n_pos = min(len(pos), DITTO_N_TRAIN // 3)
        tr_idx = np.r_[rng.choice(pos, n_pos, replace=False), rng.choice(neg, DITTO_N_TRAIN - n_pos, replace=False)]
        va_idx = rng.choice(np.setdiff1d(np.arange(len(P_tr)), tr_idx), DITTO_N_VALID, replace=False)
        s_tr, s_ev = serialize(tr), serialize(ev)
        mk = lambda S, P, idx: [(S[a], S[b]) for a, b in P[idx]]
        config = {"n_train": DITTO_N_TRAIN, "n_valid": DITTO_N_VALID, "epochs": DITTO_EPOCHS, "batch": 32, "lr": 3e-5,
                  "max_len": 128, "seed": SEED, "data": key}
        tok, dmodel, hist = ditto_finetuned(mk(s_tr, P_tr, tr_idx), y_tr[tr_idx], mk(s_tr, P_tr, va_idx),
                                            y_tr[va_idx], config)
        scorer = quantize(dmodel)
        # the decision threshold is tuned with the same (int8) scorer that scores the evaluation pairs
        f1_d, th_d = best_threshold(y_tr[va_idx], ditto_predict(scorer, tok, mk(s_tr, P_tr, va_idx)))
        ev_ids = ev.patientunitstayid.values[P_ev]
        ev_file = CACHE_DIR / f"ditto_eval_scores_{fingerprint(ev_ids, key)}.npy"
        if ev_file.exists() and not FORCE_RECOMPUTE:
            p_d = np.load(ev_file)
        else:
            p_d = ditto_predict(scorer, tok, mk(s_ev, P_ev, np.arange(len(P_ev))))
            np.save(ev_file, p_d)
        rows.append(er_metrics("Ditto", P_ev[p_d > th_d], P_ev, truth, ev))
        info.update({"Ditto threshold (int8 scorer)": th_d, "Ditto validation F1": f1_d,
                     "Ditto fine-tuning (validation F1 per epoch)": "; ".join(f"epoch {h['epoch']}: {h['valid_F1']:.3f}"
                                                                         for h in hist)})
    tab = pd.DataFrame(rows)
    return tab, info, {"th_l": th_l, "gnn": gnn, "th_g": th_g, "ref_g": ref_g, "th_d": th_d, "h_train": h_train,
                       "P_ev_scores": {"ditto": (ev_ids, p_d)}}


ER_KEY = fingerprint(X_REC.patientunitstayid, DITTO_N_TRAIN, DITTO_EPOCHS, "v1")
DUP_TAB, DUP_INFO, DUP_MODELS = cached("duplicates_benchmark_v2", lambda: run_duplicate_benchmark(X_REC, ER_KEY), ER_KEY)
DUP_CRIT = {"Duplicate Rate": "min", "Redundancy Reduction": "max", "Precision": "max", "Recall": "max",
            "F1-score": "max", "Pair Completeness": "max", "Reduction Ratio": "max"}
DUP_TAB, DUP_BEST = rank_select(DUP_TAB, DUP_CRIT)
save_table(DUP_TAB, "04_duplicates_method_comparison")
show_table(pd.DataFrame([DUP_INFO]).T.rename(columns={0: "value"}).reset_index(names="protocol"), "Evaluation protocol")
show_table(DUP_TAB, "Duplicate Bias: method comparison on the held-out hospitals")
plot_method_comparison(DUP_TAB, [(k, v, k) for k, v in DUP_CRIT.items()],
                       "Duplicate Bias — Ditto vs GNN entity resolution vs LSH", "04a_duplicates_method_comparison")
md(f"**Selected method: {DUP_BEST}** (lowest mean rank over the seven metrics).")

### 4.2 Apply the selected method to the whole cohort
The selected matcher scores every candidate pair of the full cohort. The predicted matches are closed transitively
(connected components), and the first ICU stay of every resolved patient is kept. The train/test split that follows
is grouped by the true patient key, so no patient can be in both training and test data, even if the matcher missed
a repeat stay.

In [ ]:
def apply_duplicates(X, method, models):
    X = X.reset_index(drop=True)
    if method == "LSH":
        V = lsh_vectors(X)
        C = lsh_candidates(V)
        P = C[lsh_cosine(V, C) > models["th_l"]]
    else:
        P_all = candidate_pairs(X)
        if method == "GNN entity resolution":
            P = P_all[predict_gnn(models["gnn"], X, P_all, models["ref_g"]) > models["th_g"]]
        else:  # Ditto: reuse the evaluation-pair scores, score the remaining pairs
            tok, dmodel = load_tokenizer_and_model()
            dmodel.load_state_dict(torch.load(DITTO_STATE, weights_only=False))
            dmodel = quantize(dmodel.eval())
            ids = X.patientunitstayid.values
            known_pairs, known_p = models["P_ev_scores"]["ditto"]
            known = {tuple(sorted(p)): s for p, s in zip(map(tuple, known_pairs), known_p)}
            key = [tuple(sorted((ids[a], ids[b]))) for a, b in P_all]
            todo = np.array([k not in known for k in key])
            S = serialize(X)
            p = np.array([known.get(k, np.nan) for k in key])
            log(f"[Ditto] scoring {todo.sum():,} new candidate pairs ({(~todo).sum():,} reused)")
            p[todo] = ditto_predict(dmodel, tok, [(S[a], S[b]) for a, b in P_all[todo]])
            P = P_all[p > models["th_d"]]
    keep = dedup_by_matches(X, P)
    return X.patientunitstayid.values[keep], len(P)


KEEP_IDS, N_MATCHES = cached("duplicates_apply", lambda: apply_duplicates(X_REC, DUP_BEST, DUP_MODELS),
                             fingerprint(ER_KEY, DUP_BEST, DUP_MODELS["th_d"], DUP_MODELS["th_g"], DUP_MODELS["th_l"]))
X_DUP = X_REC[X_REC.patientunitstayid.isin(set(KEEP_IDS))].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
_, te_idx = next(sgkf.split(X_DUP, X_DUP[TARGET], groups=X_DUP.uniquepid))
SPLIT = pd.Series("train", index=X_DUP.patientunitstayid.values)
SPLIT.iloc[te_idx] = "test"
TRAIN_IDS = set(SPLIT.index[SPLIT == "train"])
dup_before = X_REC.uniquepid.duplicated().mean()
dup_after = X_DUP.uniquepid.duplicated().mean()
md(f"**{DUP_BEST}** found {N_MATCHES:,} same-patient pairs. Stays: {len(X_REC):,} → {len(X_DUP):,}. "
   f"Repeat-stay rate (true key): **{dup_before:.1%} → {dup_after:.1%}**. Train / test split (patient-disjoint): "
   f"{(SPLIT == 'train').sum():,} / {(SPLIT == 'test').sum():,} stays; mortality {X_DUP[TARGET][SPLIT.values == 'train'].mean():.1%} / "
   f"{X_DUP[TARGET][SPLIT.values == 'test'].mean():.1%}.")

In [ ]:
def with_stay_counts(X):
    X = X.copy()
    X["stays_per_patient"] = X.uniquepid.map(X.uniquepid.value_counts()).astype(float)
    return X


DB, DA = with_stay_counts(X_REC), with_stay_counts(X_DUP)
DUP_VARS = ["stays_per_patient", "unitvisitnumber", "age", "apache_iva_score", "hr_mean", "rr_mean", "spo2_min",
            "creatinine_max", "admission_weight_kg", "icu_los_days"]
rep_b, rep_a = DB.uniquepid.duplicated(keep=False).values, DA.uniquepid.duplicated(keep=False).values
bias_dashboard("Duplicate Bias", DUP_BEST, DB, DA, DUP_VARS, present=DET["Duplicate Bias"]["present"],
               qq_var="age", before_name="all ICU stays", after_name=f"de-duplicated with {DUP_BEST}",
               indicator=dict(before=waffle(rep_b), after=waffle(rep_a), rows=[f"stays {40 * i + 1}-{40 * i + 40}" for i in range(8)],
                              name="stay of a patient with >1 stay", title="Repeat-Stay Map (320 randomly sampled ICU stays)",
                              footer=f"Whole cohort: {rep_b.mean():.1%} of stays belong to a patient with several stays before, {rep_a.mean():.1%} after."),
               fname="04b_dashboard_duplicates")

fig, ax = plt.subplots(1, 2, figsize=(16, 4.2), facecolor="white")
for a_, d_, ttl, c in ((ax[0], DB, "Before", C_BEFORE), (ax[1], DA, "After", C_AFTER)):
    cnt = d_.groupby("uniquepid").size().clip(upper=5).value_counts().sort_index()
    a_.bar(cnt.index.astype(int), cnt.values, color=to_rgba(c, 0.75), width=0.65)
    for xi, v in zip(cnt.index, cnt.values):
        a_.text(xi, v, f"{v:,}", ha="center", va="bottom", fontsize=9.5, color=INK)
    a_.set_xticks(range(1, 6), ["1", "2", "3", "4", "5+"])
    a_.set_xlabel("ICU stays per patient")
    a_.set_ylabel("patients")
    a_.set_title(f"{ttl}: {len(d_):,} stays of {d_.uniquepid.nunique():,} patients", loc="left")
    a_.grid(axis="x", visible=False)
fig.tight_layout()
show_fig(fig, "04c_stays_per_patient")

## 5. Outlier Bias, part A: physiologically impossible values
**What the bias is.** Outlier bias comes in two forms. (a) **Impossible values** cannot be real: a respiratory rate
of 0 or 250/min, an SpO₂ of 12%, a temperature of 0 °C, a height of 4 cm. They are device or charting errors and
pull means, minima and maxima away from the truth. (b) **Real but extreme** values belong to very sick patients.
They must be kept, but their influence on models has to be controlled.

**Part A (here)** removes (a) with physiological limits, at the level of single measurements and *before* any
smoothing or imputation, so the errors cannot leak into the 24-hour features. The removed values become missing and
are imputed in section 7. **Part B (section 8)** compares the table's multivariate detectors (ECOD, Deep SVDD, EBM)
for (b), once every record is complete.

In [ ]:
def plaus_key(col):
    pref = {"hr_": "heartrate", "rr_": "respiration", "spo2_m": "sao2", "temp_": "temperature",
            "nibp_sys": "noninvasivesystolic", "nibp_dia": "noninvasivediastolic", "nibp_map": "noninvasivemean",
            "rc_fio2": "fio2", "rc_peep": "peep", "rc_tidal_volume_set": "tidal_volume_set",
            "rc_peak_insp_pressure": "peak_insp_pressure", "rc_plateau_pressure": "plateau_pressure",
            "rc_mean_airway_pressure": "mean_airway_pressure", "rc_rr_total": "rr_total",
            "rc_minute_vent": "minute_vent", "aps_temperature": "temperature", "lab_fio2": "lab_fio2"}
    for pre, key in pref.items():
        if col.startswith(pre):
            return key
    base = col.rsplit("_", 1)[0]
    return base if base in PLAUSIBLE else (col if col in PLAUSIBLE else None)


def impossible_mask(X):
    m = pd.DataFrame(False, index=X.index, columns=continuous_columns(X))
    for c in m.columns:
        k = plaus_key(c)
        if k:
            lo, hi = PLAUSIBLE[k]
            m[c] = (X[c] < lo - 1e-6) | (X[c] > hi + 1e-6)
    if "pf_ratio_min" in m:
        m["pf_ratio_min"] = (X.pf_ratio_min < 20) | (X.pf_ratio_min > 700)
    return m.fillna(False)


DATA_K = subset_data(DATA, KEEP_IDS)               # records of the de-duplicated cohort
with timer("matrix with impossible values removed"):
    X_PLAUS, AUDIT_PLAUS = build_cached("plaus", DATA_K, BuildOptions(True, True, True), return_audit=True)
X_OA = X_DUP.set_index("patientunitstayid").loc[X_PLAUS.patientunitstayid].reset_index()
IMP_B, IMP_A = impossible_mask(X_OA), impossible_mask(X_PLAUS)
meas = pd.Series(AUDIT_PLAUS["implausible_set_nan"]).sort_values(ascending=False)
meas = meas[meas > 0]
IMP_TABLE = pd.DataFrame({"measurement": meas.index, "impossible values removed": meas.values,
                          "plausible range": [str(PLAUSIBLE.get(plaus_key(k.split(".")[-1]) or k.split(".")[-1], "")) for k in meas.index]})
save_table(IMP_TABLE, "05_impossible_values")
show_table(IMP_TABLE.head(20), f"Physiologically impossible measurements set to missing (total {meas.sum():,})")

In [ ]:
imp_rate = IMP_B.sum().sort_values(ascending=False)
OA_VARS = list(imp_rate.index[:10])
md(f"Patient-level features with impossible values before: **{int(IMP_B.to_numpy().sum()):,} cells** in "
   f"{int(IMP_B.any(axis=1).sum()):,} stays; after: **{int(IMP_A.to_numpy().sum()):,}**.")
# show stays that carry at least one impossible value next to random ones, so the 40 columns are informative
rng_ = np.random.default_rng(SEED)
hit = X_OA.patientunitstayid[IMP_B[OA_VARS].any(axis=1)].to_numpy()
pick = np.r_[rng_.choice(hit, min(20, len(hit)), replace=False),
             rng_.choice(X_OA.patientunitstayid.to_numpy(), 40 - min(20, len(hit)), replace=False)]
ib = IMP_B[OA_VARS].set_axis(X_OA.patientunitstayid).loc[pick].to_numpy().T
ia = IMP_A[OA_VARS].set_axis(X_PLAUS.patientunitstayid).loc[pick].to_numpy().T
bias_dashboard("Outlier Bias (part A: impossible values)", "Physiological plausibility limits", X_OA, X_PLAUS, OA_VARS,
               present=DET["Outlier Bias"]["present"], before_name="as recorded", after_name="impossible values removed",
               indicator=dict(before=ib, after=ia, rows=[label(v) for v in OA_VARS], name="physiologically impossible",
                              title="Impossible-Value Heatmap (20 stays with an impossible value + 20 random stays)",
                              footer=f"Whole cohort: {int(IMP_B[OA_VARS].to_numpy().sum()):,} impossible cells in these variables before, "
                                     f"{int(IMP_A[OA_VARS].to_numpy().sum()):,} after."),
               fname="05a_dashboard_outliers_impossible")

## 6. Data Noise Bias
**What the bias is.** The bias takes two forms.
* **Measurement noise** in the 5-minute vital-sign series (heart rate, respiratory rate, SpO₂): sensor jitter and
  artefact spikes (a probe slipping off, a cough, motion). The 24-hour minimum and maximum features end up describing
  artefacts instead of the patient.
* **Label noise:** outcome labels that contradict other records of the same stay.

**The three methods from the table.** Each handles both kinds of noise where it can:

| Method | Measurement noise (vital-sign series) | Label noise (hospital outcome) |
|---|---|---|
| **Confident Learning (Cleanlab)** | Each value is treated as a noisy *label* of its temporal context. Values are binned into 20 ordinal classes; a context model (±3 neighbours, trained like the networks on corrupted inputs) gives each value's class probabilities; `cleanlab.filter.find_label_issues` flags values whose recorded class contradicts their context, and these are replaced by the context prediction | `cleanlab.filter.find_label_issues` on out-of-fold probabilities of a gradient-boosting model |
| **Robust Loss Functions** | 1-D convolutional denoising network trained with the **Huber** loss, so real artefacts in its training targets do not teach it to reproduce spikes | MLP trained with **Symmetric Cross-Entropy** (robust to noisy labels); labels it confidently contradicts are flagged |
| **Diffusion Denoising Model** | DDPM prior learned on vital-sign windows; reverse diffusion started at each series' estimated noise level (SDEdit-style) | not applicable: it does not model labels, so it flags nothing (F1 = 0) |

**Evaluation (semi-synthetic, so the truth is known).** On real series from training patients, Gaussian noise (20% of
the series SD) and 2% artefact spikes (±3 SD) are injected. On the real training labels, 5% are flipped at random.

| Metric | Meaning | Better |
|---|---|---|
| Residual Variance | share of the injected noise still present after denoising | lower |
| Spectral Distortion (dB) | how much the method changes the *real* signal in the physiological band (trends >= 30 min) | lower |
| MAE, RMSE | error of the denoised series against the real series (in units of the variable's SD) | lower |
| SNR (dB) | signal-to-noise ratio of the denoised series | higher |
| Label-Error Detection F1 | flagged labels vs truly flipped labels | higher |

In [ ]:
NOISE_VARS = ["heartrate", "respiration", "sao2"]          # temperature is >90% empty in vitalPeriodic
NOISE_LABEL = {"heartrate": "Heart rate (bpm)", "respiration": "Respiratory rate (/min)", "sao2": "SpO₂ (%)"}
MIN_LEN, WIN = 48, 48
STD_FLOOR = {"heartrate": 2.0, "respiration": 1.0, "sao2": 0.5}
LF_MAX = 1 / 6.0                                           # cycles/sample: periods >= 30 min at 5-min sampling

VP_CLEAN = prepare_vital_periodic(DATA_K["vital_periodic"], BuildOptions(True, True, True))
NOISE_STATS = cached("noise_stats_before", lambda: series_noise_stats(VP_CLEAN, cols=NOISE_VARS),
                     fingerprint(len(VP_CLEAN), "v1"))
SRC = outcome_sources(DATA_K)
P_ = DATA_K["cohort_patient"].set_index("patientunitstayid")
icu_death = (P_.unitdischargestatus == "Expired") & (P_.unitdischargelocation == "Death")
n_contra = int((icu_death & (P_.hospitaldischargestatus == "Alive")).sum())
NOISE_PRESENT = (np.nanmean([v["residual_variance_ratio"] for v in NOISE_STATS.values()]) > T["noise_residual_var"]
                 or np.nanmean([v["artifact_spike_rate"] for v in NOISE_STATS.values()]) > T["noise_artifact_rate"]
                 or n_contra > 0)
show_table(pd.DataFrame([{"variable": NOISE_LABEL[k], **v} for k, v in NOISE_STATS.items()]),
           "Noise in the vital-sign series entering this step (after impossible values were removed)")
md(f"Labels contradicting the ICU death record: **{n_contra}**. Data Noise Bias present at this step: **{NOISE_PRESENT}**.")

### 6.1 The three denoisers

In [ ]:
def series_of(vp, stays, var, max_n, rng):
    sub = vp[vp.patientunitstayid.isin(stays)][["patientunitstayid", var]].dropna()
    out = [g.to_numpy(float) for _, g in sub.groupby("patientunitstayid")[var] if len(g) >= MIN_LEN]
    rng.shuffle(out)
    return out[:max_n]


def corrupt(x, rng):
    """Additive Gaussian noise (20% of the series SD) + 2% impulsive artefacts (±3 SD)."""
    x = np.asarray(x, float)
    s = np.std(x, axis=-1, keepdims=True) + 1e-6
    y = x + rng.normal(0, 0.2, x.shape) * s
    return y + (rng.random(x.shape) < 0.02) * rng.choice([-3.0, 3.0], x.shape) * s


def _length_batches(series, max_batch=512):
    order = np.argsort([len(s) for s in series], kind="stable")
    for i in range(0, len(order), max_batch):
        idx = order[i:i + max_batch]
        L = max(len(series[j]) for j in idx)
        yield idx, np.stack([np.pad(series[j], (0, L - len(series[j])), mode="edge") for j in idx]).astype(np.float32)


class _TimeEmb(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        f = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32) / half)
        a = t.float().unsqueeze(1) * f.unsqueeze(0)
        return torch.cat([torch.sin(a), torch.cos(a)], 1)


class RobustLossDenoiser:
    """Fully convolutional 1-D denoiser (residual learning) trained with the Huber loss."""

    def __init__(self, epochs=6, batch=256, delta=0.5, seed=SEED):
        torch.manual_seed(seed)
        self.epochs, self.batch, self.delta, self.seed = epochs, batch, delta, seed
        self.net = nn.Sequential(nn.Conv1d(1, 32, 7, padding=3), nn.ReLU(), nn.Conv1d(32, 32, 7, padding=3), nn.ReLU(),
                                 nn.Conv1d(32, 16, 5, padding=2), nn.ReLU(), nn.Conv1d(16, 1, 5, padding=2))

    def fit(self, W):
        rng = np.random.default_rng(self.seed)
        opt = torch.optim.Adam(self.net.parameters(), lr=1e-3)
        for _ in range(self.epochs):
            perm = rng.permutation(len(W))
            for i in range(0, len(W), self.batch):
                x = W[perm[i:i + self.batch]]
                xn = _t(corrupt(x, rng)).unsqueeze(1)
                loss = F.huber_loss((xn - self.net(xn)).squeeze(1), _t(x), delta=self.delta)
                opt.zero_grad()
                loss.backward()
                opt.step()
        return self

    def denoise(self, series):
        out = [None] * len(series)
        with torch.no_grad():
            for idx, A in _length_batches(series):
                z = torch.as_tensor(A).unsqueeze(1)
                r = (z - self.net(z)).squeeze(1).numpy()
                for k, j in enumerate(idx):
                    out[j] = r[k, :len(series[j])]
        return out


class DiffusionDenoiser:
    """DDPM prior on vital-sign windows (Ho et al., 2020) used as a denoiser: the observed series is treated as a
    diffusion state at the step whose noise level matches the series' estimated noise, and deterministic DDIM reverse
    steps return it to the data manifold (SDEdit-style)."""

    def __init__(self, T_=100, epochs=6, batch=256, seed=SEED):
        torch.manual_seed(seed)
        self.T, self.epochs, self.batch, self.seed = T_, epochs, batch, seed
        self.beta = torch.linspace(1e-4, 0.05, T_)
        self.abar = torch.cumprod(1 - self.beta, 0)
        self.temb = _TimeEmb(16)
        self.c1, self.tproj = nn.Conv1d(1, 32, 7, padding=3), nn.Linear(16, 32)
        self.body = nn.Sequential(nn.SiLU(), nn.Conv1d(32, 32, 7, padding=3), nn.SiLU(),
                                  nn.Conv1d(32, 32, 5, padding=2), nn.SiLU(), nn.Conv1d(32, 1, 5, padding=2))
        self.params = list(self.c1.parameters()) + list(self.tproj.parameters()) + list(self.body.parameters())

    def _eps(self, x, t):
        return self.body(self.c1(x) + self.tproj(self.temb(t)).unsqueeze(-1))

    def fit(self, W):
        rng = np.random.default_rng(self.seed)
        opt = torch.optim.Adam(self.params, lr=1e-3)
        for _ in range(self.epochs):
            perm = rng.permutation(len(W))
            for i in range(0, len(W), self.batch):
                x0 = _t(W[perm[i:i + self.batch]]).unsqueeze(1)
                t = torch.randint(0, self.T, (len(x0),))
                ab = self.abar[t].view(-1, 1, 1)
                e = torch.randn_like(x0)
                loss = F.mse_loss(self._eps(ab.sqrt() * x0 + (1 - ab).sqrt() * e, t), e)
                opt.zero_grad()
                loss.backward()
                opt.step()
        return self

    def start_step(self, sigma):
        ratio = ((1 - self.abar) / self.abar).sqrt()
        return int(torch.argmin((ratio - float(sigma)).abs()))

    def denoise(self, series):
        sig = [max(1.4826 * np.median(np.abs(np.diff(s))) / np.sqrt(2), 1e-3) for s in series]
        steps = np.array([self.start_step(s) for s in sig])
        out = [None] * len(series)
        with torch.no_grad():
            for ts in np.unique(steps):
                members = np.flatnonzero(steps == ts)
                for idx, A in _length_batches([series[j] for j in members]):
                    x = self.abar[ts].sqrt() * torch.as_tensor(A).unsqueeze(1)
                    for t in range(int(ts), -1, -1):
                        e = self._eps(x, torch.full((len(x),), t, dtype=torch.long))
                        ab = self.abar[t]
                        x0 = (x - (1 - ab).sqrt() * e) / ab.sqrt()
                        ab_prev = self.abar[t - 1] if t > 0 else torch.tensor(1.0)
                        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * e
                    r = x.squeeze(1).numpy()
                    for k, j in enumerate(idx):
                        out[members[j]] = r[k, :len(series[members[j]])]
        return out


class CleanlabDenoiser:
    """Confident learning on measurements: 20 ordinal value classes, Gaussian class probabilities from a context
    model (ridge regression on the ±3 neighbours, their median and mean), cleanlab.filter.find_label_issues
    (prune by noise rate) to flag values that contradict their context; flagged values get the context prediction."""

    def __init__(self, n_bins=20):
        self.n_bins = n_bins

    @staticmethod
    def context(x, k=3):
        n = len(x)
        xp = np.pad(x, k, mode="reflect") if n > k else np.pad(x, k, mode="edge")
        C = np.column_stack([xp[k + d: k + d + n] for d in range(-k, k + 1) if d != 0])
        return np.column_stack([C, np.median(C, 1), C.mean(1)])

    def fit(self, series, rng):
        # like the denoising networks, the context model learns from corrupted inputs: noisy neighbours in,
        # recorded value out; sigma describes how far an observed (noisy) value strays from that prediction
        noisy = [corrupt(s, rng) for s in series]
        X = np.vstack([self.context(s) for s in noisy])
        y = np.concatenate(series)
        self.model = Ridge(alpha=1.0).fit(X, y)
        res = np.concatenate(noisy) - self.model.predict(X)
        self.sigma = max(1.4826 * np.median(np.abs(res - np.median(res))), 1e-3)
        self.edges = np.quantile(y, np.linspace(0, 1, self.n_bins + 1))
        self.edges[0], self.edges[-1] = -np.inf, np.inf
        return self

    def denoise(self, series, chunk=400_000):
        from cleanlab.filter import find_label_issues
        lens = np.array([len(s) for s in series])
        y = np.concatenate(series)
        mu = self.model.predict(np.vstack([self.context(s) for s in series]))
        out = y.copy()
        for a in range(0, len(y), chunk):
            sl = slice(a, a + chunk)
            cdf = stats.norm.cdf((self.edges[None, :] - mu[sl, None]) / self.sigma)
            P = np.clip(np.diff(cdf, axis=1), 1e-6, None)
            P /= P.sum(1, keepdims=True)
            lab_ = np.clip(np.searchsorted(self.edges, y[sl], side="right") - 1, 0, self.n_bins - 1)
            present = np.unique(lab_)
            flag = np.zeros(len(lab_), bool)
            if len(present) > 1:
                # cleanlab needs every class to occur; work on the classes present in this chunk
                remap = {c: i for i, c in enumerate(present)}
                Pk = P[:, present] / P[:, present].sum(1, keepdims=True)
                flag = find_label_issues(np.vectorize(remap.get)(lab_), Pk, filter_by="prune_by_noise_rate", n_jobs=1)
            out[sl] = np.where(flag, mu[sl], y[sl])
        return np.split(out, np.cumsum(lens)[:-1])


class Denoisers:
    """Fits the three denoisers per vital sign on training stays; works in normalised units for the networks."""

    METHODS = ["Confident Learning (Cleanlab)", "Robust Loss Functions", "Diffusion Denoising Model"]

    def fit(self, vp, stays, rng, n_series=3000, n_windows=30000):
        self.norm, self.models = {}, {m: {} for m in self.METHODS}
        for v in NOISE_VARS:
            ser = series_of(vp, stays, v, n_series, rng)
            allv = np.concatenate(ser)
            mu, sd = float(allv.mean()), float(allv.std())
            self.norm[v] = (mu, sd)
            W = []
            for s in ser:
                z = (s - mu) / sd
                for st in rng.integers(0, len(z) - WIN + 1, size=max(1, n_windows // len(ser))):
                    W.append(z[st:st + WIN])
            W = np.array(W, dtype=np.float32)
            self.models["Robust Loss Functions"][v] = RobustLossDenoiser().fit(W)
            self.models["Diffusion Denoising Model"][v] = DiffusionDenoiser().fit(W)
            self.models["Confident Learning (Cleanlab)"][v] = CleanlabDenoiser().fit(ser, rng)
        return self

    def apply(self, method, series, var):
        if method == "No denoising (reference)":
            return [np.asarray(s, float) for s in series]
        if method == "Confident Learning (Cleanlab)":
            return self.models[method][var].denoise([np.asarray(s, float) for s in series])
        mu, sd = self.norm[var]
        z = [((np.asarray(s, float) - mu) / sd).astype(np.float32) for s in series]
        return [r.astype(float) * sd + mu for r in self.models[method][var].denoise(z)]


def spectral_distortion(den, ref):
    n = min(64, len(ref))
    f, pr = welch(ref, nperseg=n)
    _, pdn = welch(den, nperseg=n)
    band = f <= LF_MAX
    return float(np.mean(np.abs(10 * np.log10(pdn[band] + 1e-10) - 10 * np.log10(pr[band] + 1e-10))))

### 6.2 Benchmark A: measurement noise (semi-synthetic, real training series)

In [ ]:
def run_signal_benchmark(vp, train_stays, n_eval=300):
    rng = np.random.default_rng(SEED)
    stays = np.array(sorted(train_stays))
    rng.shuffle(stays)
    fit_stays, eval_stays = set(stays[: len(stays) // 2]), set(stays[len(stays) // 2:])
    with timer("fit the three denoisers (HR, RR, SpO2)"):
        den = Denoisers().fit(vp, fit_stays, rng)
    rows = []
    for v in NOISE_VARS:
        ser = [s for s in series_of(vp, eval_stays, v, 3 * n_eval, rng) if np.std(s) >= STD_FLOOR[v]][:n_eval]
        noisy = [corrupt(s, rng) for s in ser]
        sd = den.norm[v][1]
        for m in ["No denoising (reference)"] + Denoisers.METHODS:
            d_ref, d_noisy = den.apply(m, ser, v), den.apply(m, noisy, v)
            rv, sdist, mae, rmse, snr = [], [], [], [], []
            for ref, nz, dr, dn in zip(ser, noisy, d_ref, d_noisy):
                noise = nz - ref
                err = dn - ref
                rv.append(np.var(dn - dr) / np.var(noise))
                sdist.append(spectral_distortion(dr, ref))
                mae.append(np.mean(np.abs(err)) / sd)
                rmse.append(np.sqrt(np.mean(err ** 2)) / sd)
                snr.append(10 * np.log10(np.var(ref) / max(np.mean(err ** 2), 1e-12)))
            rows.append({"Method": m, "variable": v, "series": len(ser), "Residual Variance": np.mean(rv),
                         "Spectral Distortion (dB)": np.mean(sdist), "MAE": np.mean(mae), "RMSE": np.mean(rmse),
                         "SNR (dB)": np.mean(snr)})
    return pd.DataFrame(rows), den


with timer("measurement-noise benchmark"):
    NOISE_SIG_PER_VAR, DENOISERS = cached("noise_signal_benchmark",
                                          lambda: run_signal_benchmark(VP_CLEAN, TRAIN_IDS),
                                          fingerprint(len(VP_CLEAN), len(TRAIN_IDS), "v2"))
show_table(NOISE_SIG_PER_VAR, "Measurement-noise benchmark per vital sign")

### 6.3 Benchmark B: label noise (5% of the real training labels flipped at random)

In [ ]:
def resolve_labels(data, X):
    """Majority vote of the three outcome sources, then a logical rule: a stay discharged from the ICU as
    'Expired' to 'Death' died in hospital."""
    votes = outcome_sources(data)
    maj, n_src = votes.mean(axis=1, skipna=True), votes.notna().sum(axis=1)
    resolved = pd.Series(np.where((n_src >= 2) & (maj != 0.5), (maj > 0.5).astype(int), votes.patient.astype(int)),
                         index=votes.index)
    p = data["cohort_patient"].set_index("patientunitstayid")
    icu_d = (p.unitdischargestatus == "Expired") & (p.unitdischargelocation == "Death")
    resolved[icu_d.reindex(resolved.index, fill_value=False)] = 1
    return X.patientunitstayid.map(resolved).astype(int).to_numpy()


def numeric_matrix(X, fit_mask):
    """Quick numeric design matrix for neural models: median-imputed, z-scored continuous features,
    binary flags and one-hot categoricals (all statistics from the rows in fit_mask)."""
    cont = continuous_columns(X)
    A = X[cont].astype(float)
    med = A[fit_mask].median()
    A = A.fillna(med)
    A = (A - A[fit_mask].mean()) / A[fit_mask].std().replace(0, 1)
    B = X[[c for c in BINARY if c in X]].astype(float).fillna(0)
    cats = [c for c in CATEGORICAL if c in X]
    enc = OneHotEncoder(handle_unknown="ignore", min_frequency=0.005, sparse_output=False)
    enc.fit(X.loc[fit_mask, cats].astype(str))
    Cc = enc.transform(X[cats].astype(str))
    return np.hstack([A.to_numpy(), B.to_numpy(), Cc]).astype(np.float32)


def symmetric_cross_entropy(logits, y, alpha=0.1, beta=1.0, A=-4.0):
    """Symmetric Cross Entropy (Wang et al., ICCV 2019): CE + reverse CE, robust to noisy labels."""
    p = torch.sigmoid(logits).clamp(1e-7, 1 - 1e-7)
    ce = F.binary_cross_entropy(p, y)
    rce = -(p * torch.where(y == 1, 0.0, A) + (1 - p) * torch.where(y == 0, 0.0, A)).mean()
    return alpha * ce + beta * rce


CB_BETA = 0.9999   # effective-number hyper-parameter; 0.999 saturates at 1,000 samples, too few for this cohort


def class_balanced_focal_loss(logits, y, n_per_class, beta=CB_BETA, gamma=2.0):
    """Class-Balanced Focal Loss (Cui et al., CVPR 2019; Lin et al., ICCV 2017)."""
    eff = 1.0 - torch.pow(beta, _t(n_per_class))
    w = (1.0 - beta) / eff
    w = w / w.sum() * 2
    p = torch.sigmoid(logits)
    pt = torch.where(y == 1, p, 1 - p)
    wt = torch.where(y == 1, w[1], w[0])
    return (wt * (1 - pt) ** gamma * F.binary_cross_entropy_with_logits(logits, y, reduction="none")).mean()


class MLPClassifier:
    """Tabular MLP. activation='selu' gives a Self-Normalising Network (LeCun-normal init + AlphaDropout)."""

    def __init__(self, hidden=(128, 64), activation="relu", loss="bce", epochs=20, batch=256, lr=1e-3, dropout=0.1,
                 seed=SEED, track=False):
        self.hidden, self.activation, self.loss, self.epochs = hidden, activation, loss, epochs
        self.batch, self.lr, self.dropout, self.seed, self.track = batch, lr, dropout, seed, track

    def _build(self, d):
        layers, prev = [], d
        for h in self.hidden:
            lin = nn.Linear(prev, h)
            if self.activation == "selu":
                nn.init.normal_(lin.weight, 0, 1 / math.sqrt(prev))
                nn.init.zeros_(lin.bias)
                layers += [lin, nn.SELU(), nn.AlphaDropout(self.dropout)]
            else:
                layers += [lin, nn.ReLU(), nn.Dropout(self.dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        return nn.Sequential(*layers)

    def fit(self, X, y, sample_weight=None):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        X, y = np.asarray(X, np.float32), np.asarray(y, np.float32)
        sw = np.ones(len(y), np.float32) if sample_weight is None else np.asarray(sample_weight, np.float32)
        self.net = self._build(X.shape[1])
        opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=1e-4)
        npc = [float((y == 0).sum()), float((y == 1).sum())]
        self.grad_norms = []
        for _ in range(self.epochs):
            self.net.train()
            perm = rng.permutation(len(X))
            for i in range(0, len(X), self.batch):
                b = perm[i:i + self.batch]
                lg, yy = self.net(_t(X[b])).squeeze(1), _t(y[b])
                if self.loss == "focal_cb":
                    loss = class_balanced_focal_loss(lg, yy, npc)
                elif self.loss == "sce":
                    loss = symmetric_cross_entropy(lg, yy)
                else:
                    loss = (F.binary_cross_entropy_with_logits(lg, yy, reduction="none") * _t(sw[b])).mean()
                opt.zero_grad()
                loss.backward()
                if self.track:
                    self.grad_norms.append(float(torch.sqrt(sum((p.grad ** 2).sum() for p in self.net.parameters()))))
                opt.step()
        return self

    def logits(self, X):
        self.net.eval()
        return self.net(_t(np.asarray(X, np.float32))).squeeze(1)

    def predict_proba(self, X):
        with torch.no_grad():
            p = torch.sigmoid(self.logits(X)).numpy()
        return np.column_stack([1 - p, p])


def flag_by_class_thresholds(y, p):
    """Confident-learning style rule on a model's probabilities: a label is flagged when the model's probability of
    the given class is below that class's average self-confidence and the other class reaches its own average."""
    P = np.column_stack([1 - p, p])
    thr = np.array([P[y == k, k].mean() for k in (0, 1)])
    other = 1 - y
    return (P[np.arange(len(y)), y] < thr[y]) & (P[np.arange(len(y)), other] >= thr[other])


def run_label_benchmark(X, y_clean, flip_rate=0.05):
    from cleanlab.filter import find_label_issues
    rng = np.random.default_rng(SEED)
    flip = rng.random(len(y_clean)) < flip_rate
    y_noisy = np.where(flip, 1 - y_clean, y_clean)
    rows = []
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    p = cross_val_predict(hgb(categorical_features="from_dtype"), model_frame(X), y_noisy, cv=cv,
                          method="predict_proba")[:, 1]
    cl = find_label_issues(y_noisy, np.column_stack([1 - p, p]), filter_by="prune_by_noise_rate", n_jobs=1)
    A = numeric_matrix(X, np.ones(len(X), bool))
    p_rl = np.zeros(len(X))
    for tr, te in StratifiedKFold(3, shuffle=True, random_state=SEED).split(A, y_noisy):
        p_rl[te] = MLPClassifier(loss="sce", epochs=20).fit(A[tr], y_noisy[tr]).predict_proba(A[te])[:, 1]
    rl = flag_by_class_thresholds(y_noisy, p_rl)
    for m, fl in (("Confident Learning (Cleanlab)", cl), ("Robust Loss Functions", rl),
                  ("Diffusion Denoising Model", np.zeros(len(X), bool))):
        tp = int((fl & flip).sum())
        prec = tp / fl.sum() if fl.sum() else 0.0
        rec_ = tp / flip.sum()
        rows.append({"Method": m, "labels flagged": int(fl.sum()), "Label precision": prec, "Label recall": rec_,
                     "Label-Error Detection F1": 2 * prec * rec_ / (prec + rec_) if prec + rec_ else 0.0})
    return pd.DataFrame(rows), int(flip.sum())


tr_mask_p = X_PLAUS.patientunitstayid.isin(TRAIN_IDS).to_numpy()
Y_RESOLVED = resolve_labels(DATA_K, X_PLAUS)
with timer("label-noise benchmark"):
    NOISE_LAB, N_FLIPPED = cached("noise_label_benchmark",
                                  lambda: run_label_benchmark(X_PLAUS[tr_mask_p].reset_index(drop=True),
                                                              Y_RESOLVED[tr_mask_p]),
                                  fingerprint(X_PLAUS.patientunitstayid, "v1"))
show_table(NOISE_LAB, f"Label-noise benchmark ({N_FLIPPED:,} of {tr_mask_p.sum():,} training labels flipped)")

In [ ]:
sig = NOISE_SIG_PER_VAR.groupby("Method", sort=False)[["Residual Variance", "Spectral Distortion (dB)", "MAE", "RMSE",
                                                       "SNR (dB)"]].mean().reset_index()
NOISE_TAB = sig.merge(NOISE_LAB[["Method", "Label-Error Detection F1"]], on="Method", how="left")
NOISE_CRIT = {"Residual Variance": "min", "Spectral Distortion (dB)": "min", "MAE": "min", "RMSE": "min",
              "SNR (dB)": "max", "Label-Error Detection F1": "max"}
ref_row = NOISE_TAB[NOISE_TAB.Method == "No denoising (reference)"]
NOISE_TAB, NOISE_BEST = rank_select(NOISE_TAB[NOISE_TAB.Method != "No denoising (reference)"], NOISE_CRIT)
NOISE_TAB = pd.concat([NOISE_TAB, ref_row], ignore_index=True)
NOISE_TAB["selected"] = NOISE_TAB["selected"].fillna(False).astype(bool)
save_table(NOISE_TAB, "06_noise_method_comparison")
show_table(NOISE_TAB, "Data Noise Bias: method comparison (signal metrics averaged over HR, RR and SpO₂)")
plot_method_comparison(NOISE_TAB, [(k, v, k) for k, v in NOISE_CRIT.items()],
                       "Data Noise Bias — Cleanlab vs Robust Loss vs Diffusion Denoising", "06a_noise_method_comparison",
                       baseline="No denoising (reference)")
md(f"**Selected method: {NOISE_BEST}**")

### 6.4 Apply the selected denoiser to every vital-sign series, then correct the verified label errors
Labels are changed only where the error is **verified** by the independent outcome records (majority vote of
the three outcome sources, plus "discharged from the ICU to death ⇒ died in hospital"). Labels that a model merely
finds surprising are reported but kept: they usually belong to hard-to-predict patients, not to recording errors.

In [ ]:
def collect_segments(vp, var):
    stay = vp.patientunitstayid.to_numpy()
    col = vp[var].to_numpy(float)
    bounds = np.flatnonzero(np.r_[True, stay[1:] != stay[:-1], True])
    segs, locs = [], []
    for a, b in zip(bounds[:-1], bounds[1:]):
        idx = np.arange(a, b)
        ok = idx[~np.isnan(col[a:b])]
        if len(ok) >= 7:
            segs.append(col[ok])
            locs.append(ok)
    return segs, locs


def denoise_all(vp, method):
    vp = vp.sort_values(["patientunitstayid", "observationoffset"]).reset_index(drop=True)
    for v in NOISE_VARS:
        segs, locs = collect_segments(vp, v)
        col = vp[v].to_numpy(float).copy()
        for i in range(0, len(segs), 4000):
            for s, loc in zip(DENOISERS.apply(method, segs[i:i + 4000], v), locs[i:i + 4000]):
                col[loc] = s
        lo, hi = PLAUSIBLE[v]
        vp[v] = np.clip(col, lo, hi).astype(np.float32)
    return vp


with timer(f"denoise every series with {NOISE_BEST}"):
    VP_DEN = cached("vp_denoised", lambda: denoise_all(VP_CLEAN, NOISE_BEST) if NOISE_PRESENT else VP_CLEAN,
                    fingerprint(len(VP_CLEAN), NOISE_BEST, NOISE_PRESENT, "v1"))
with timer("feature matrix from the denoised series"):
    X_NOISE = build_cached("denoised", DATA_K, BuildOptions(True, True, True, denoise=lambda vp: VP_DEN),
                           extra=fingerprint(NOISE_BEST, NOISE_PRESENT, len(VP_DEN)))
X_NOISE = X_NOISE.set_index("patientunitstayid").loc[X_PLAUS.patientunitstayid].reset_index()
y_old = X_NOISE[TARGET].to_numpy()
X_NOISE[TARGET] = Y_RESOLVED
X_NOISE["split"] = X_NOISE.patientunitstayid.map(SPLIT).values
n_label_fixed = int((y_old != Y_RESOLVED).sum())
md(f"Verified label corrections: **{n_label_fixed}** outcome labels changed (cross-source majority vote and "
   f"ICU-death rule). Benchmark B shows what the label methods would flag; on the real labels, the three outcome "
   f"sources agree for every other stay, so no other label is changed.")

In [ ]:
def spike_matrix(vp, vars_=NOISE_VARS):
    """Per stay and vital sign: does the series contain an artefact spike (|x - 5-point median| > clinical delta)?"""
    out = {}
    for v in vars_:
        s = vp[["patientunitstayid", "observationoffset", v]].dropna().sort_values(["patientunitstayid", "observationoffset"])
        x, stay = s[v].to_numpy(float), s.patientunitstayid.to_numpy()
        r = np.abs(x - median_filter(x, size=5, mode="nearest"))
        b = np.flatnonzero(np.r_[True, stay[1:] != stay[:-1]])
        edge = np.zeros(len(x), bool)
        for k in (0, 1):
            edge[np.clip(b + k, 0, len(x) - 1)] = True
            edge[np.clip(b - 1 - k, 0, len(x) - 1)] = True
        spike = (r > CLIN_DELTA[v]) & ~edge
        out[v] = pd.Series(spike).groupby(stay).any()
    return pd.DataFrame(out).fillna(False)


SPK_B, SPK_A = spike_matrix(VP_CLEAN), spike_matrix(VP_DEN)
NOISE_VARS_X = ["hr_mean", "hr_min", "hr_max", "rr_mean", "rr_min", "rr_max", "spo2_mean", "spo2_min", "spo2_frac_below_90"]
src_of = lambda c: "heartrate" if c.startswith("hr") else "respiration" if c.startswith("rr") else "sao2"
ids_all = X_NOISE.patientunitstayid.to_numpy()
mB = pd.DataFrame({c: SPK_B[src_of(c)].reindex(ids_all, fill_value=False).to_numpy() for c in NOISE_VARS_X}, index=ids_all)
mA = pd.DataFrame({c: SPK_A[src_of(c)].reindex(ids_all, fill_value=False).to_numpy() for c in NOISE_VARS_X}, index=ids_all)
rng_ = np.random.default_rng(SEED)
hit = ids_all[mB.any(axis=1).to_numpy()]
pick = np.r_[rng_.choice(hit, min(20, len(hit)), replace=False), rng_.choice(ids_all, 40 - min(20, len(hit)), replace=False)]
bias_dashboard("Data Noise Bias", NOISE_BEST, X_PLAUS, X_NOISE, NOISE_VARS_X, present=NOISE_PRESENT, qq_var="hr_max",
               before_name="raw 5-min series", after_name=f"denoised with {NOISE_BEST}",
               indicator=dict(before=mB.loc[pick].to_numpy().T, after=mA.loc[pick].to_numpy().T,
                              rows=[label(c) for c in NOISE_VARS_X], name="series with an artefact spike",
                              title="Artefact-Spike Heatmap (20 stays with a spike + 20 random stays)",
                              footer=f"Whole cohort - stays with >=1 artefact spike: HR {SPK_B.heartrate.mean():.1%} → {SPK_A.heartrate.mean():.1%}, "
                                     f"RR {SPK_B.respiration.mean():.1%} → {SPK_A.respiration.mean():.1%}, "
                                     f"SpO₂ {SPK_B.sao2.mean():.1%} → {SPK_A.sao2.mean():.1%}."),
               fname="06b_dashboard_noise")

fig, axes = plt.subplots(3, 1, figsize=(16, 9.5), facecolor="white", sharex=True)
tr_ids = np.array(sorted(TRAIN_IDS))
for ax, v in zip(axes, NOISE_VARS):
    cand_ = [s for s in rng_.choice(tr_ids, 400, replace=False) if SPK_B[v].get(s, False)]
    sid = cand_[0] if cand_ else tr_ids[0]
    a_ = VP_CLEAN[VP_CLEAN.patientunitstayid == sid].sort_values("observationoffset")
    b_ = VP_DEN[VP_DEN.patientunitstayid == sid].sort_values("observationoffset")
    ax.plot(a_.observationoffset / 60, a_[v], color=C_BEFORE, linewidth=1.3, label="Before (raw)")
    ax.plot(b_.observationoffset / 60, b_[v], color=C_AFTER, linewidth=2.0, label=f"After ({NOISE_BEST})")
    ax.set_ylabel(NOISE_LABEL[v])
    ax.set_title(f"{NOISE_LABEL[v]} — ICU stay {sid}", loc="left", fontsize=12)
    ax.legend(loc="upper right", fontsize=9.5, ncol=2)
axes[-1].set_xlabel("hours since ICU admission")
fig.suptitle("Example vital-sign series before and after denoising", x=0.01, ha="left", fontsize=15,
             weight="bold", color=TITLE_BLUE)
fig.tight_layout()
show_fig(fig, "06c_noise_example_series")

## 7. Missing Data Bias
**What the bias is.** About a fifth of the feature values are missing, and *not at random*: sicker patients get
more blood gases and lactate measurements, ventilated patients have ventilator settings, and so on. Dropping
incomplete stays or filling in a constant would distort the distributions and the relationships between variables.

**Preparation.** Features missing in more than 70% of training stays are dropped. They are mostly ventilator
settings of non-ventilated patients, where "missing" means "not ventilated", not an unknown value. A missingness
indicator (`*_was_missing`) is added wherever missingness itself is associated with mortality (informative
missingness).

**The three methods from the table.**
* **HyperImpute** (Jarrett et al., ICML 2022): iterative column-wise imputation. For every column, the best learner
  is chosen automatically from a pool (Bayesian ridge, extra trees, gradient boosting) by cross-validation. It is
  implemented here from the paper's algorithm, because the `hyperimpute` package pins old numpy/pandas versions.
* **CSDI** (Tashiro et al., NeurIPS 2021): conditional score-based diffusion imputation. This tabular version uses
  an MLP denoiser instead of the 2-D transformer. It returns several imputations (samples) per missing value.
* **MIP-VAE**: a multiple-imputation *partial* VAE. *No published method with exactly this name was found.* It is
  built from its two published ingredients: the permutation-invariant partial-VAE encoder that reads only the
  observed features (P-VAE, Ma et al., ICML 2019), trained with the importance-weighted bound for missing data
  (MIWAE, Mattei & Frellsen, ICML 2019). Imputations are drawn from the posterior.

**Evaluation (the truth is known).** In 4,000 training stays, 20% of the *observed* values are hidden, in two
scenarios: MCAR (completely at random) and MAR (the hiding probability depends on age and APACHE score). Each
method imputes, and the imputations are compared with the hidden truth. Values are min-max scaled (0-1) per
feature, so the features are comparable.

| Metric | Meaning | Better |
|---|---|---|
| Wasserstein Distance | distance between the distributions of imputed and true values | lower |
| Correlation Preservation | 1 - relative change of the correlation matrix | higher |
| MAE, RMSE | error of the imputed values | lower |
| NRMSE | RMSE divided by the SD of the true values (per feature) | lower |
| CRPS | accuracy of the whole predictive distribution (equals MAE for a single imputation) | lower |
| NLL | Gaussian negative log-likelihood of the true value under the imputation's distribution | lower |

Do-no-harm guard: the 3-fold logistic-regression AUC on the imputed data may not be more than 0.02 below the best
method's.

In [ ]:
X7 = X_NOISE.copy()
trm7 = (X7.split == "train").to_numpy()
CONT7 = continuous_columns(X7)
miss7 = X7.loc[trm7, CONT7].isna().mean().sort_values(ascending=False)
MISS_CELLS_ENTRY = float(X7[CONT7].isna().to_numpy().mean())
MISSING_PRESENT = bool((miss7 > T["missing_feature_rate"]).any())
DROP7 = sorted(miss7[miss7 > T["missing_drop_rate"]].index)
KEEP7 = [c for c in CONT7 if c not in DROP7]
ytr7 = X7.loc[trm7, TARGET].to_numpy()
IND7 = [c for c in KEEP7 if (m_ := X7.loc[trm7, c].isna().to_numpy()).mean() > T["missing_feature_rate"]
        and abs(ytr7[m_].mean() - ytr7[~m_].mean()) > 0.02]
X7 = X7.drop(columns=DROP7)
for c in IND7:
    X7[f"{c}_was_missing"] = X7[c].isna().astype(int)
BIN_NAN7 = [c for c in BINARY if c in X7 and X7[c].isna().any()]
IMP_COLS = KEEP7 + BIN_NAN7
md(f"Missing cells in the continuous features entering this step: **{MISS_CELLS_ENTRY:.1%}**; "
   f"{int((miss7 > 0.05).sum())} features >5% missing. Dropped (>70% missing): {', '.join(DROP7) or 'none'}. "
   f"Missingness indicators added for {len(IND7)} features. Missing Data Bias present at this step: **{MISSING_PRESENT}**.")

### 7.1 The three imputers

In [ ]:
class HyperImputeImputer:
    """Iterative column-wise imputation with automatic per-column model selection (HyperImpute's core algorithm)."""

    def __init__(self, n_iter=2, cv_rows=2000, seed=SEED):
        self.n_iter, self.cv_rows, self.seed = n_iter, cv_rows, seed

    def _pool(self):
        from sklearn.ensemble import ExtraTreesRegressor
        from sklearn.linear_model import BayesianRidge
        return {"bayesian_ridge": BayesianRidge(),
                "extra_trees": ExtraTreesRegressor(n_estimators=40, min_samples_leaf=3, max_features=0.4, n_jobs=-1,
                                                   random_state=self.seed),
                "hist_gbm": HistGradientBoostingRegressor(max_iter=100, random_state=self.seed)}

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X)
        M = ~np.isnan(Z)
        cur = np.where(M, Z, 0.0)
        self.cols = [j for j in np.argsort(M.mean(0))[::-1] if (~M[:, j]).any()]
        self.models, self.choice, self.sigma_z = {}, {}, np.full(X.shape[1], 1.0)
        for it in range(self.n_iter):
            for j in self.cols:
                obs = M[:, j]
                Xo = np.delete(cur, j, axis=1)
                if it == 0:
                    idx = rng.choice(np.flatnonzero(obs), min(self.cv_rows, obs.sum()), replace=False)
                    sc_ = {k: cross_val_score(m, Xo[idx], Z[idx, j], cv=3, scoring="r2").mean()
                           for k, m in self._pool().items()}
                    self.choice[j] = max(sc_, key=sc_.get)
                    self.sigma_z[j] = math.sqrt(max(1 - sc_[self.choice[j]], 0.02))
                m = self._pool()[self.choice[j]].fit(Xo[obs], Z[obs, j])
                cur[~obs, j] = m.predict(Xo[~obs])
                self.models[j] = m
        return self

    def transform(self, X):
        Z = self.sc.tf(X)
        M = ~np.isnan(Z)
        cur = np.where(M, Z, 0.0)
        for _ in range(self.n_iter):
            for j in self.cols:
                if (~M[:, j]).any():
                    cur[~M[:, j], j] = self.models[j].predict(np.delete(cur, j, axis=1)[~M[:, j]])
        return self.sc.inv(cur)

    def predictive_sd(self):
        return self.sigma_z * self.sc.sd


class CSDIImputer:
    """CSDI-style conditional score-based diffusion imputer for tables (MLP noise predictor, 'quad' schedule,
    random-target training strategy); several reverse-diffusion samples per row."""

    def __init__(self, T_=50, hidden=256, steps=2500, batch=256, seed=SEED):
        self.T, self.hidden, self.steps, self.batch, self.seed = T_, hidden, steps, batch, seed
        b = torch.linspace(1e-4 ** 0.5, 0.5 ** 0.5, T_) ** 2
        self.beta, self.alpha = b, 1 - b
        self.abar = torch.cumprod(self.alpha, 0)

    def _eps(self, xt, xc, cm, tm, t):
        return self.net(torch.cat([xt * tm, xc * cm, cm, self.temb(t)], 1))

    def fit(self, X):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X)
        M = (~np.isnan(Z)).astype(np.float32)
        Z = np.nan_to_num(Z)
        n, d = Z.shape
        self.temb = _TimeEmb(32)
        self.net = mlp([3 * d + 32, self.hidden, self.hidden, self.hidden, d], act=nn.SiLU)
        opt = torch.optim.Adam(self.net.parameters(), lr=1e-3)
        for _ in range(self.steps):
            b = rng.integers(0, n, self.batch)
            x0, m = _t(Z[b]), _t(M[b])
            tm = m * (torch.rand_like(m) < torch.rand(len(b), 1)).float()     # random target strategy
            cm = m - tm
            t = torch.randint(0, self.T, (len(b),))
            ab = self.abar[t].unsqueeze(1)
            e = torch.randn_like(x0)
            loss = (((e - self._eps(ab.sqrt() * x0 + (1 - ab).sqrt() * e, x0, cm, tm, t)) * tm) ** 2).sum() / tm.sum().clamp(min=1)
            opt.zero_grad()
            loss.backward()
            opt.step()
        return self

    def sample(self, X, S=10):
        Z = self.sc.tf(X)
        cmask = (~np.isnan(Z)).astype(np.float32)
        Z = np.nan_to_num(Z)
        out = np.repeat(Z[None], S, axis=0)
        with torch.no_grad():
            for i in range(0, len(Z), 2048):
                x0, cm = _t(Z[i:i + 2048]), _t(cmask[i:i + 2048])
                tm = 1 - cm
                for s in range(S):
                    x = torch.randn_like(x0)
                    for t in reversed(range(self.T)):
                        e = self._eps(x, x0, cm, tm, torch.full((len(x0),), t, dtype=torch.long))
                        x = (x - self.beta[t] / (1 - self.abar[t]).sqrt() * e) / self.alpha[t].sqrt()
                        if t > 0:
                            x = x + ((1 - self.abar[t - 1]) / (1 - self.abar[t]) * self.beta[t]).sqrt() * torch.randn_like(x)
                    out[s, i:i + 2048] = (cm * x0 + tm * x).numpy()
        return self.sc.inv(out)

    def transform(self, X, S=10):
        return np.median(self.sample(X, S), axis=0)


class MIPVAEImputer(nn.Module):
    """Multiple-imputation partial VAE: P-VAE set encoder over the observed features, Gaussian decoder, trained with
    the importance-weighted bound on observed entries plus random extra masking; imputations by sampling-
    importance-resampling from the posterior."""

    def __init__(self, latent=16, emb=16, hidden=128, K=20, steps=2500, batch=256, extra_mask=0.2, seed=SEED):
        super().__init__()
        self.L, self.emb, self.hidden, self.K = latent, emb, hidden, K
        self.steps, self.batch, self.extra_mask, self.seed = steps, batch, extra_mask, seed

    def _build(self, d):
        torch.manual_seed(self.seed)
        self.d = d
        self.E = nn.Parameter(torch.randn(d, self.emb) * 0.1)          # feature-identity embeddings
        self.h = mlp([1 + self.emb, 64, 64])                          # per-feature encoder h(x_j, e_j)
        self.g = mlp([64, self.hidden, 2 * self.L])                   # aggregate -> q(z | x_obs)
        self.dec = mlp([self.L, self.hidden, self.hidden, 2 * d], act=nn.Tanh)

    def _q(self, x, m):
        s = torch.cat([x.unsqueeze(-1), self.E.unsqueeze(0).expand(len(x), -1, -1)], -1)
        c = (self.h(s) * m.unsqueeze(-1)).sum(1)                     # sum over observed features only
        o = self.g(c)
        return o[:, :self.L], o[:, self.L:].clamp(-8, 8)

    def _bound(self, x, m_in, m_obs, K):
        mu, logv = self._q(x * m_in, m_in)
        q = torch.distributions.Normal(mu, torch.exp(0.5 * logv))
        z = q.rsample((K,))
        out = self.dec(z)
        xm, xs = out[..., :self.d], F.softplus(out[..., self.d:]) + 1e-3
        lpx = (torch.distributions.Normal(xm, xs).log_prob(x.unsqueeze(0)) * m_obs.unsqueeze(0)).sum(-1)
        lw = lpx + torch.distributions.Normal(0, 1).log_prob(z).sum(-1) - q.log_prob(z).sum(-1)
        return lw, xm, xs

    def fit(self, X):
        rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X)
        M = (~np.isnan(Z)).astype(np.float32)
        Z = np.nan_to_num(Z).astype(np.float32)
        self._build(Z.shape[1])
        opt = torch.optim.Adam(self.parameters(), lr=1e-3)
        for _ in range(self.steps):
            b = rng.integers(0, len(Z), self.batch)
            x, m = _t(Z[b]), _t(M[b])
            m_in = m * (torch.rand_like(m) > self.extra_mask).float()     # learn to predict dropped observed values
            lw, _, _ = self._bound(x, m_in, m, self.K)
            loss = -(torch.logsumexp(lw, 0) - math.log(self.K)).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
        return self

    def sample(self, X, S=10, K=50):
        Z = self.sc.tf(X)
        M = (~np.isnan(Z)).astype(np.float32)
        Z = np.nan_to_num(Z).astype(np.float32)
        out = np.repeat(Z[None], S, axis=0)
        point = Z.copy()
        g = torch.Generator().manual_seed(self.seed)
        with torch.no_grad():
            for i in range(0, len(Z), 1024):
                x, m = _t(Z[i:i + 1024]), _t(M[i:i + 1024])
                lw, xm, xs = self._bound(x, m, m, K)
                w = torch.softmax(lw, 0)                                     # K x B importance weights
                point[i:i + 1024] = (m * x + (1 - m) * (w.unsqueeze(-1) * xm).sum(0)).numpy()
                for s in range(S):
                    k = torch.multinomial(w.T, 1, generator=g).squeeze(1)    # sampling-importance-resampling
                    ar = torch.arange(len(x))
                    draw = xm[k, ar] + xs[k, ar] * torch.randn(xm[k, ar].shape, generator=g)
                    out[s, i:i + 1024] = (m * x + (1 - m) * draw).numpy()
        self.point_ = self.sc.inv(point)
        return self.sc.inv(out)

    def transform(self, X):
        self.sample(X, S=1)
        return self.point_

### 7.2 Benchmark on hidden-but-known values

In [ ]:
def _crps_samples(S_, y):
    """Energy-form CRPS of sample sets S_ (S x n) for observations y (n)."""
    t1 = np.abs(S_ - y[None]).mean(0)
    t2 = np.abs(S_[:, None, :] - S_[None, :, :]).mean((0, 1))
    return t1 - 0.5 * t2


def run_missing_benchmark(A_df, cols, eval_cols, y, n=4000, rate=0.2):
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(A_df), min(n, len(A_df)), replace=False)
    A = A_df[cols].to_numpy(float)[idx]
    yy = y[idx]
    ev = np.array([c in eval_cols for c in cols])
    lo, hi = np.nanmin(A, 0), np.nanmax(A, 0)
    span = np.where(hi - lo > 0, hi - lo, 1.0)
    ref_corr = pd.DataFrame(A[:, ev]).corr(min_periods=50).to_numpy()
    zs = Standardizer().fit(A)
    drv = np.column_stack([np.nan_to_num(zs.tf(A)[:, cols.index(c)]) for c in ("age", "apache_iva_score") if c in cols])
    rows = []
    for scen in ("MCAR", "MAR"):
        Mobs = ~np.isnan(A)
        if scen == "MCAR":
            hide = (rng.random(A.shape) < rate) & Mobs
        else:
            s = drv.sum(1)
            p = 1 / (1 + np.exp(-(s - np.median(s))))
            hide = (rng.random(A.shape) < (p / p.mean() * rate)[:, None]) & Mobs
        hide[:, ~ev] = False
        Am = np.where(hide, np.nan, A)
        for name in ["HyperImpute", "CSDI", "MIP-VAE"]:
            t0 = time.time()
            if name == "HyperImpute":
                m = HyperImputeImputer().fit(Am)
                point, samples, sd = m.transform(Am), None, m.predictive_sd()
            elif name == "CSDI":
                samples = CSDIImputer().fit(Am).sample(Am, S=10)
                point, sd = np.median(samples, 0), None
            else:
                m = MIPVAEImputer().fit(Am)
                samples = m.sample(Am, S=10)
                point, sd = m.point_, None
            sec = time.time() - t0
            jj = np.flatnonzero(ev & hide.any(0))
            err = ((point - A) / span)[hide]
            nrmse = np.mean([np.sqrt(np.mean((point[hide[:, j], j] - A[hide[:, j], j]) ** 2)) / (np.std(A[hide[:, j], j]) + 1e-9)
                             for j in jj])
            w1 = np.mean([wasserstein_distance(point[hide[:, j], j] / span[j], A[hide[:, j], j] / span[j]) for j in jj])
            cimp = np.corrcoef(point[:, ev], rowvar=False)
            ok = np.isfinite(ref_corr) & np.isfinite(cimp)
            corr_pres = 1 - np.linalg.norm((cimp - ref_corr)[ok]) / np.linalg.norm(ref_corr[ok])
            ytrue = (A / span)[hide]
            if samples is not None:
                Ss = (samples / span)[:, hide]
                crps = float(np.mean(_crps_samples(Ss, ytrue)))
                mu_, sd_ = Ss.mean(0), np.maximum(Ss.std(0), 0.01)
            else:
                crps = float(np.mean(np.abs(err)))
                mu_ = (point / span)[hide]
                sd_ = np.maximum(np.broadcast_to(sd / span, A.shape)[hide], 0.01)
            nll = float(np.mean(0.5 * np.log(2 * np.pi * sd_ ** 2) + 0.5 * ((ytrue - mu_) / sd_) ** 2))
            Zi = Standardizer().fit(point).tf(point)
            auc = roc_auc_score(yy, cross_val_predict(LogisticRegression(max_iter=2000, C=0.5), Zi, yy, cv=3,
                                                      method="predict_proba")[:, 1])
            rows.append({"Method": name, "scenario": scen, "Wasserstein Distance": w1, "Correlation Preservation": corr_pres,
                         "MAE": float(np.mean(np.abs(err))), "RMSE": float(np.sqrt(np.mean(err ** 2))), "NRMSE": nrmse,
                         "CRPS": crps, "NLL": nll, "downstream LR AUC": auc, "seconds": sec})
            log(f"[missing] {scen} {name}: MAE={rows[-1]['MAE']:.4f} W1={w1:.4f} CRPS={crps:.4f} ({sec:.0f}s)")
    return pd.DataFrame(rows)


with timer("missing-data benchmark (3 imputers x 2 scenarios)"):
    MISS_PER = cached("missing_benchmark",
                      lambda: run_missing_benchmark(X7[trm7].reset_index(drop=True), IMP_COLS, KEEP7, ytr7),
                      fingerprint(X7.loc[trm7, IMP_COLS], "v1"))
show_table(MISS_PER, "Missing-data benchmark per masking scenario")
MISS_CRIT = {"Wasserstein Distance": "min", "Correlation Preservation": "max", "MAE": "min", "RMSE": "min",
             "NRMSE": "min", "CRPS": "min", "NLL": "min"}
MISS_TAB = MISS_PER.groupby("Method", sort=False)[list(MISS_CRIT) + ["downstream LR AUC", "seconds"]].mean().reset_index()
MISS_TAB, MISS_BEST = rank_select(MISS_TAB, MISS_CRIT,
                                  guard=MISS_TAB["downstream LR AUC"] >= MISS_TAB["downstream LR AUC"].max() - MAX_AUC_DROP)
save_table(MISS_TAB, "07_missing_method_comparison")
show_table(MISS_TAB, "Missing Data Bias: method comparison (mean of the MCAR and MAR scenarios)")
plot_method_comparison(MISS_TAB, [(k, v, k) for k, v in MISS_CRIT.items()],
                       "Missing Data Bias — HyperImpute vs CSDI vs MIP-VAE", "07a_missing_method_comparison")
md(f"**Selected method: {MISS_BEST}**")

### 7.3 Impute with the selected method (fitted on the training split, applied to all stays)

In [ ]:
def apply_imputer(X, cols, binary_nan, method, trm):
    A = X[cols].to_numpy(float)
    if method == "HyperImpute":
        imp = HyperImputeImputer().fit(A[trm])
        out_ = imp.transform(A)
    elif method == "CSDI":
        out_ = CSDIImputer(steps=4000).fit(A[trm]).transform(A)
    else:
        out_ = MIPVAEImputer(steps=4000).fit(A[trm]).transform(A)
    out = X.copy()
    out[cols] = out_
    lo, hi = X.loc[trm, cols].min(), X.loc[trm, cols].max()
    out[cols] = out[cols].clip(lo, hi, axis=1)            # stay inside the observed range of each variable
    for c in binary_nan:
        out[c] = out[c].clip(0, 1).round()
    for c in ["gcs_eyes", "gcs_motor", "gcs_verbal"]:
        if c in out:
            out[c] = out[c].round()
    if {"gcs_eyes", "gcs_motor", "gcs_verbal", "gcs_total"} <= set(out.columns):
        out["gcs_total"] = out.gcs_eyes + out.gcs_motor + out.gcs_verbal
    if out.gender.isna().any():
        out["gender"] = out.gender.fillna(X.loc[trm, "gender"].mode()[0])
    return out


with timer(f"impute with {MISS_BEST}"):
    X_IMP = cached("imputed", lambda: apply_imputer(X7, IMP_COLS, BIN_NAN7, MISS_BEST, trm7) if MISSING_PRESENT else X7,
                   fingerprint(X7.loc[:, IMP_COLS], MISS_BEST, "v1"))
md(f"Missing cells after imputation: **{int(X_IMP[IMP_COLS].isna().to_numpy().sum())}** "
   f"(before: {int(X7[IMP_COLS].isna().to_numpy().sum()):,}).")

In [ ]:
MISS_VARS = [c for c in miss7.index if c in KEEP7][:10]
mb = X7[MISS_VARS].isna().set_axis(X7.patientunitstayid)
ma = X_IMP[MISS_VARS].isna().set_axis(X_IMP.patientunitstayid)
ib, ia = sample_indicator(mb, ma)
bias_dashboard("Missing Data Bias", MISS_BEST, X7, X_IMP, MISS_VARS, present=MISSING_PRESENT,
               before_name="observed values only", after_name=f"imputed with {MISS_BEST}",
               indicator=dict(before=ib, after=ia, rows=[label(v) for v in MISS_VARS], name="missing",
                              title="Missing-Value Heatmap (40 randomly sampled ICU stays)",
                              footer=f"Whole cohort: {mb.to_numpy().mean():.1%} of these cells missing before, {ma.to_numpy().mean():.1%} after."),
               fname="07b_dashboard_missing")
top = [c for c in miss7.index if c in KEEP7][:25]
fig, ax = plt.subplots(figsize=(14, 8.5), facecolor="white")
ba_bars(ax, [label(c) for c in top], X7[top].isna().mean().to_numpy() * 100, X_IMP[top].isna().mean().to_numpy() * 100,
        "Missing values per feature (%), the 25 most incomplete features kept", value_fmt=lambda v: f"{v:.1f}%")
fig.tight_layout()
show_fig(fig, "07c_missing_rate_per_feature")

## 8. Outlier Bias, part B: multivariate outliers
**What the bias is.** After part A, every value is physiologically possible, but a *record* can still be
anomalous: a combination of values far outside the bulk of the data (for example extreme acidosis together with
extreme lactate), or a residual charting error that is possible on its own but inconsistent with the rest of the
record. Such records have high leverage: they pull model coefficients and scaling statistics towards themselves.

**The three methods from the table** (all fitted on robust-scaled training records):
* **ECOD** (Li et al., IEEE TKDE 2022, `pyod`): empirical-CDF tail probabilities of every feature, aggregated per record.
* **Deep SVDD** (Ruff et al., ICML 2018, `pyod`): a neural network maps the records into a minimum-volume hypersphere;
  the distance to the centre is the outlier score.
* **EBM** (Explainable Boosting Machine, Microsoft `interpret`), used as an anomaly detector in the
  *unsupervised-as-supervised* way: it learns to tell real records from a reference sample whose columns are
  permuted independently. A record that looks like the reference has an unusual *combination* of values, and the
  EBM's additive terms say which features make it unusual. CPU budget: 8 outer bags, 10 pairwise interactions.

**Evaluation (the truth is known).** 4,000 training records are taken, and 3% of them are corrupted in three
realistic ways: extreme values in 3 features, unit errors (x10 or /10) in 2 features, and implausible combinations
(40% of the features copied from other records).

| Metric | Meaning | Better |
|---|---|---|
| Outlier Score Stability | mean Spearman correlation of the scores of detectors refitted on 70% bootstrap subsamples | higher |
| Neighborhood Consistency | Spearman correlation between a record's score and its 10 nearest neighbours' mean score | higher |
| AUROC, AUPRC | ranking of the corrupted records | higher |
| Precision@K, F1-score | among the top-K scores (K = number of corrupted records) | higher |
| Brier Score | accuracy of the outlier probability (scores mapped to [0, 1] by Gaussian scaling) | lower |

**Policy for the selected detector.** Flagged stays are real patients, most of them very sick (shown below).
Deleting them would bias the cohort towards healthier patients. They are **kept and flagged** (`outlier_flag`), and
their extreme values are **winsorised** to the 0.5th-99.5th training percentiles to limit their leverage.

In [ ]:
X8 = X_IMP.copy()
trm8 = (X8.split == "train").to_numpy()
CONT8 = list(KEEP7)
A8 = X8[CONT8].to_numpy(float)
med8 = np.median(A8[trm8], 0)
mad8 = np.median(np.abs(A8[trm8] - med8), 0) * 1.4826
mad8[mad8 == 0] = np.nan
rz8 = np.abs((A8 - med8) / mad8)
OUTB_SHARE = float((np.nan_to_num(rz8, nan=0.0) > 4.5).any(axis=1).mean())
OUTB_PRESENT = OUTB_SHARE > T["outlier_multivariate_rate"]
rs8 = RobustScaler(quantile_range=(5, 95)).fit(A8[trm8])
tf8 = lambda A: np.clip(rs8.transform(A), -20, 20)
md(f"Stays with at least one robust |z| > 4.5 value at this step: **{OUTB_SHARE:.1%}** (rule: > 1%). "
   f"Outlier Bias (multivariate) present at this step: **{OUTB_PRESENT}**.")


class EBMDetector:
    """EBM (interpret) as an anomaly detector: real records vs a column-permuted reference sample."""

    def __init__(self, seed=SEED, outer_bags=8, interactions=10):
        self.seed, self.outer_bags, self.interactions = seed, outer_bags, interactions

    def fit(self, X):
        from interpret.glassbox import ExplainableBoostingClassifier
        rng = np.random.default_rng(self.seed)
        ref = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])
        self.m = ExplainableBoostingClassifier(random_state=self.seed, outer_bags=self.outer_bags,
                                               interactions=self.interactions, n_jobs=-2)
        self.m.fit(np.vstack([X, ref]), np.r_[np.zeros(len(X)), np.ones(len(ref))])
        return self

    def score(self, X):
        return self.m.decision_function(X)

    def term_importances(self, names):
        terms = [" x ".join(names[i] for i in t) for t in self.m.term_features_]
        return pd.Series(self.m.term_importances(), index=terms).sort_values(ascending=False)


class PyODDetector:
    def __init__(self, kind, seed=SEED):
        self.kind, self.seed = kind, seed

    def fit(self, X):
        if self.kind == "ECOD":
            from pyod.models.ecod import ECOD
            self.m = ECOD(contamination=0.03)
        else:
            from pyod.models.deep_svdd import DeepSVDD
            torch.manual_seed(self.seed)
            self.m = DeepSVDD(n_features=X.shape[1], hidden_neurons=[64, 32], epochs=40, batch_size=128,
                              learning_rate=1e-3, contamination=0.03, random_state=self.seed, verbose=0)
        self.m.fit(X)
        return self

    def score(self, X):
        return self.m.decision_function(X)


OUT_METHODS = ["ECOD", "Deep SVDD", "EBM"]


def make_detector(name, seed=SEED):
    return EBMDetector(seed) if name == "EBM" else PyODDetector(name, seed)


def unify(scores, ref_scores):
    """Gaussian scaling of outlier scores to [0, 1] (Kriegel et al., SDM 2011; pyod's 'unify')."""
    mu, sd = np.mean(ref_scores), np.std(ref_scores) + 1e-12
    return np.clip(stats.norm.cdf((scores - mu) / sd) * 2 - 1, 0, 1)


def inject_anomalies(A, rng, rate=0.03):
    n, d = A.shape
    k = int(rate * n)
    idx = rng.choice(n, k, replace=False)
    Q = np.quantile(A, [0.005, 0.25, 0.75, 0.995], axis=0)
    iqr = np.maximum(Q[2] - Q[1], 1e-6)
    B = A.copy()
    for i, kind in zip(idx, rng.integers(0, 3, k)):
        if kind == 0:                                   # extreme values in 3 features
            js = rng.choice(d, 3, replace=False)
            up = rng.random(3) < 0.5
            B[i, js] = np.where(up, Q[3, js] + rng.uniform(1, 3, 3) * iqr[js], Q[0, js] - rng.uniform(1, 3, 3) * iqr[js])
        elif kind == 1:                                 # unit errors in 2 features
            js = rng.choice(d, 2, replace=False)
            B[i, js] = B[i, js] * rng.choice([10.0, 0.1], 2)
        else:                                           # implausible combination
            js = rng.choice(d, int(0.4 * d), replace=False)
            B[i, js] = A[rng.integers(0, n, len(js)), js]
    truth = np.zeros(n, bool)
    truth[idx] = True
    return B, truth


def run_outlier_benchmark(A_train, n=4000, n_boot=3):
    rng = np.random.default_rng(SEED)
    A = A_train[rng.choice(len(A_train), min(n, len(A_train)), replace=False)]
    Bc, truth = inject_anomalies(A, rng)
    E = tf8(Bc)
    knn = NearestNeighbors(n_neighbors=11).fit(E).kneighbors(E, return_distance=False)[:, 1:]
    K = int(truth.sum())
    rows = []
    for m in OUT_METHODS:
        t0 = time.time()
        s = make_detector(m).fit(E).score(E)
        boots = [make_detector(m, SEED + b + 1).fit(E[rng.choice(len(E), int(0.7 * len(E)), replace=False)]).score(E)
                 for b in range(n_boot)]
        top = np.argsort(-s)[:K]
        flag = np.zeros(len(E), bool)
        flag[top] = True
        rows.append({"Method": m,
                     "Outlier Score Stability": float(np.mean([spearmanr(a, b)[0] for a, b in itertools.combinations(boots + [s], 2)])),
                     "Neighborhood Consistency": float(spearmanr(s, s[knn].mean(1))[0]),
                     "AUROC": roc_auc_score(truth, s), "AUPRC": average_precision_score(truth, s),
                     "Precision@K": float(truth[top].mean()), "F1-score": f1_score(truth, flag),
                     "Brier Score": brier_score_loss(truth, unify(s, s)), "seconds": time.time() - t0})
        log(f"[outliers] {m}: AUROC={rows[-1]['AUROC']:.3f} AUPRC={rows[-1]['AUPRC']:.3f} ({rows[-1]['seconds']:.0f}s)")
    return pd.DataFrame(rows), K


with timer("multivariate outlier benchmark"):
    OUT_TAB, OUT_K = cached("outlier_benchmark", lambda: run_outlier_benchmark(A8[trm8]), fingerprint(A8[trm8], "v1"))
OUT_CRIT = {"Outlier Score Stability": "max", "Neighborhood Consistency": "max", "AUROC": "max", "AUPRC": "max",
            "Precision@K": "max", "F1-score": "max", "Brier Score": "min"}
OUT_TAB, OUT_BEST = rank_select(OUT_TAB, OUT_CRIT)
save_table(OUT_TAB, "08_outlier_method_comparison")
show_table(OUT_TAB, f"Outlier Bias: method comparison ({OUT_K} corrupted records among 4,000)")
plot_method_comparison(OUT_TAB, [(k, v, k) for k, v in OUT_CRIT.items()],
                       "Outlier Bias — ECOD vs Deep SVDD vs EBM", "08a_outlier_method_comparison")
md(f"**Selected method: {OUT_BEST}**")

### 8.1 Apply the selected detector: flag the 2% most anomalous stays, keep them, winsorise their extreme values

In [ ]:
def apply_outliers(X, method):
    det = make_detector(method).fit(tf8(A8[trm8]))
    s = det.score(tf8(A8))
    thr = np.quantile(s[trm8], 0.98)
    flag = s >= thr
    lo, hi = np.quantile(A8[trm8], 0.005, axis=0), np.quantile(A8[trm8], 0.995, axis=0)
    B = A8.copy()
    capped = np.zeros_like(B, bool)
    capped[flag] = (B[flag] < lo) | (B[flag] > hi)
    B[flag] = np.clip(B[flag], lo, hi)
    out = X.copy()
    out[CONT8] = B
    if {"gcs_eyes", "gcs_motor", "gcs_verbal", "gcs_total"} <= set(out.columns):
        out["gcs_total"] = out.gcs_eyes + out.gcs_motor + out.gcs_verbal
    out["outlier_score"], out["outlier_flag"] = s, flag.astype(int)
    terms = det.term_importances(CONT8).head(10) if method == "EBM" else None
    return out, capped, float(thr), terms


with timer(f"apply {OUT_BEST}"):
    X_OUT, CAPPED, OUT_THR, EBM_TERMS = cached("outliers_apply", lambda: apply_outliers(X8, OUT_BEST),
                                               fingerprint(A8, OUT_BEST, "v1"))
if not OUTB_PRESENT:
    X_OUT, CAPPED = X8.assign(outlier_score=X_OUT.outlier_score, outlier_flag=0), np.zeros_like(CAPPED)
fl = X_OUT.outlier_flag.to_numpy() == 1
md(f"Flagged stays: **{fl.sum():,}** ({fl.mean():.1%}); hospital mortality **{X_OUT.loc[fl, TARGET].mean():.1%}** "
   f"in flagged stays vs {X_OUT.loc[~fl, TARGET].mean():.1%} in the others, so they are real, very sick patients and "
   f"are kept. Values winsorised: **{int(CAPPED.sum()):,}** cells in {int(CAPPED.any(axis=1).sum()):,} stays.")
if EBM_TERMS is not None:
    show_table(EBM_TERMS.rename("importance").rename_axis("EBM term").reset_index(),
               "EBM explanation: terms that make a stay anomalous")

In [ ]:
cap_df = pd.DataFrame(CAPPED, columns=CONT8, index=X8.patientunitstayid)
OB_VARS = list(cap_df.sum().sort_values(ascending=False).index[:10])
lo8, hi8 = np.quantile(A8[trm8], 0.005, axis=0), np.quantile(A8[trm8], 0.995, axis=0)
ext_after = pd.DataFrame((fl[:, None]) & ((X_OUT[CONT8].to_numpy() < lo8 - 1e-9) | (X_OUT[CONT8].to_numpy() > hi8 + 1e-9)),
                         columns=CONT8, index=X8.patientunitstayid)
rng_ = np.random.default_rng(SEED)
ids8 = X8.patientunitstayid.to_numpy()
hit = ids8[cap_df[OB_VARS].any(axis=1).to_numpy()]
pick = np.r_[rng_.choice(hit, min(20, len(hit)), replace=False), rng_.choice(ids8, 40 - min(20, len(hit)), replace=False)]
bias_dashboard("Outlier Bias (part B: multivariate)", OUT_BEST, X8, X_OUT, OB_VARS, present=OUTB_PRESENT,
               before_name="imputed data", after_name=f"{OUT_BEST}-flagged stays winsorised",
               indicator=dict(before=cap_df.loc[pick, OB_VARS].to_numpy().T, after=ext_after.loc[pick, OB_VARS].to_numpy().T,
                              rows=[label(v) for v in OB_VARS], name="extreme value in a flagged stay",
                              title="Outlier Heatmap (20 flagged stays with extreme values + 20 random stays)",
                              footer=f"{OUT_BEST} flagged {fl.sum():,} stays; {int(CAPPED.sum()):,} extreme cells winsorised before, "
                                     f"{int(ext_after.to_numpy().sum())} remain after."),
               fname="08b_dashboard_outliers_multivariate")

fig, ax = plt.subplots(1, 2, figsize=(16, 4.2), facecolor="white")
s_ = X_OUT.outlier_score.to_numpy()
ax[0].hist(s_[~fl], bins=60, color=to_rgba(C_OTHER, 0.8), label="not flagged")
ax[0].hist(s_[fl], bins=60, color=to_rgba(C_BEFORE, 0.9), label="flagged (top 2%)")
ax[0].axvline(OUT_THR, color=INK, linewidth=1)
ax[0].set_title(f"{OUT_BEST} outlier scores (threshold = training 98th percentile)", loc="left")
ax[0].set_yscale("log")
ax[0].legend()
m_ = [X_OUT.loc[~fl, TARGET].mean(), X_OUT.loc[fl, TARGET].mean()]
ax[1].bar(["not flagged", "flagged"], m_, color=[to_rgba(C_OTHER, 0.8), to_rgba(C_BEFORE, 0.9)], width=0.55)
for i, v in enumerate(m_):
    ax[1].text(i, v, f"{v:.1%}", ha="center", va="bottom", fontsize=11, color=INK)
ax[1].set_title("Hospital mortality: flagged stays are the sickest patients", loc="left")
ax[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:.0%}"))
ax[1].grid(axis="x", visible=False)
fig.tight_layout()
show_fig(fig, "08c_outlier_scores")

## 9. Feature Scale Bias
**What the bias is.** The features live on wildly different scales: tidal volume in hundreds of mL, platelets in
hundreds, pH around 7.4, creatinine around 1. Most are strongly right-skewed. Distance-based and gradient-based
models are then dominated by the large-unit features, activations explode or vanish, and training becomes
unstable.

**The three methods from the table.** Each is a *representation + model* pair. A model-level method still
prescribes the input transform it needs, and that transform becomes the delivered data if the method is selected.

| Method | Input transform | Network |
|---|---|---|
| **PowerTransform (Yeo-Johnson)** | Yeo-Johnson power transform + standardisation, per feature | MLP (ReLU) |
| **Self-Normalizing Neural Network (SNN / SELU)** | z-score standardisation (required by SELU self-normalisation, Klambauer et al. 2017) | SNN: SELU, LeCun-normal init, AlphaDropout |
| **FT-Transformer** | quantile transform to a normal distribution (the paper's preprocessing, Gorishniy et al. 2021) | FT-Transformer (`rtdl_revisiting_models`, 2 blocks) |
| *No scaling (reference)* | raw clinical units | MLP (ReLU) |

| Metric | Meaning | Better |
|---|---|---|
| Wasserstein Distance | mean pairwise 1-Wasserstein distance between the features' distributions (divided by the pooled SD): are the features on one common scale? | lower |
| Correlation Preservation | 1 - relative change of the Pearson correlation matrix | higher |
| Skewness Reduction | mean \|skew\| before - mean \|skew\| after | higher |
| Kurtosis | mean \|excess kurtosis\| after the transform | lower |
| Activation Variance | mean \|log10 variance\| of the hidden activations (0 = unit variance at every layer) | lower |
| Gradient Stability | coefficient of variation of the gradient norm during training | lower |

Do-no-harm guard: validation ROC-AUC no more than 0.02 below the reference.

In [ ]:
X9 = X_OUT.copy()
trm9 = (X9.split == "train").to_numpy()
CONT9 = list(CONT8)
sd9 = X9.loc[trm9, CONT9].std()
sk9 = X9.loc[trm9, CONT9].skew()
SCALE_RATIO = float(sd9[sd9 > 0].max() / sd9[sd9 > 0].min())
SCALE_PRESENT = SCALE_RATIO > T["scale_std_ratio"] or (sk9.abs() > 1).mean() > T["scale_skew_share"]
md(f"At this step: max/min feature SD ratio **{SCALE_RATIO:,.0f}** ({sd9.idxmax()} vs {sd9.idxmin()}), "
   f"{(sk9.abs() > 1).mean():.0%} of features with |skew| > 1. Feature Scale Bias present: **{SCALE_PRESENT}**.")

FLAG9 = [c for c in BINARY if c in X9] + [c for c in X9.columns if c.endswith("_was_missing")]
CAT9 = [c for c in CATEGORICAL if c in X9]
OHE9 = OneHotEncoder(handle_unknown="ignore", min_frequency=0.005, sparse_output=False).fit(X9.loc[trm9, CAT9].astype(str))
OTHER9 = np.hstack([X9[FLAG9].to_numpy(float), OHE9.transform(X9[CAT9].astype(str))]).astype(np.float32)
OTHER9_NAMES = FLAG9 + list(OHE9.get_feature_names_out(CAT9))

SCALE_METHODS = {
    "No scaling (reference)": (lambda: None, "mlp"),
    "PowerTransform (Yeo-Johnson)": (lambda: PowerTransformer(method="yeo-johnson", standardize=True), "mlp"),
    "SNN (SELU)": (lambda: StandardScaler(), "snn"),
    "FT-Transformer": (lambda: QuantileTransformer(output_distribution="normal", n_quantiles=1000, subsample=200000,
                                                   random_state=SEED), "ftt"),
}


def scale_fit(name, A):
    sc = SCALE_METHODS[name][0]()
    return None if sc is None else sc.fit(A)


def scale_apply(sc, A):
    return A.copy() if sc is None else sc.transform(A)


def activation_variances(net, X, kinds=(nn.ReLU, nn.SELU)):
    vals = []
    hooks = [m.register_forward_hook(lambda m_, i_, o: vals.append(float(o.float().var())))
             for m in net.modules() if isinstance(m, kinds)]
    net.eval()
    with torch.no_grad():
        net(_t(X))
    for h in hooks:
        h.remove()
    return vals


def train_ftt(Xa, ya, Xb, epochs=8, batch=256, seed=SEED):
    """FT-Transformer from rtdl_revisiting_models (default 2-block configuration, default AdamW optimiser)."""
    from rtdl_revisiting_models import FTTransformer
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = FTTransformer(n_cont_features=Xa.shape[1], cat_cardinalities=[], d_out=1,
                          **FTTransformer.get_default_kwargs(n_blocks=2))
    opt = model.make_default_optimizer()
    grads = []
    for _ in range(epochs):
        model.train()
        perm = rng.permutation(len(Xa))
        for i in range(0, len(Xa), batch):
            b = perm[i:i + batch]
            loss = F.binary_cross_entropy_with_logits(model(_t(Xa[b]), None).squeeze(1), _t(ya[b]))
            opt.zero_grad()
            loss.backward()
            grads.append(float(torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters() if p.grad is not None))))
            opt.step()
    model.eval()
    vals = []
    mods = [blk["output"] for blk in model.backbone.blocks] + [model.backbone.output.activation]
    hooks = [m.register_forward_hook(lambda m_, i_, o: vals.append(float(o.float().var()))) for m in mods]
    with torch.no_grad():
        model(_t(Xb[:2048]), None)
        for h in hooks:
            h.remove()
        p = torch.sigmoid(torch.cat([model(_t(Xb[i:i + 2048]), None).squeeze(1) for i in range(0, len(Xb), 2048)])).numpy()
    return p, grads, vals


def mean_pairwise_w1(Z, grid=200):
    q = np.quantile(Z, np.linspace(0.005, 0.995, grid), axis=0)
    d = Z.shape[1]
    tot = [np.mean(np.abs(q[:, i] - q[:, j])) for i, j in itertools.combinations(range(d), 2)]
    return float(np.mean(tot) / (Z.std() + 1e-12))


def run_scale_benchmark():
    tr = np.flatnonzero(trm9)
    a, b = train_test_split(tr, test_size=0.25, random_state=SEED, stratify=X9[TARGET].to_numpy()[tr])
    y = X9[TARGET].to_numpy()
    A = X9[CONT9].to_numpy(float)
    ref_corr = np.corrcoef(A[a], rowvar=False)
    sk_raw = np.mean(np.abs(skew(A[a], axis=0)))
    sub = np.random.default_rng(SEED).choice(a, min(4000, len(a)), replace=False)
    rows, traces = [], {}
    steps_per_epoch = math.ceil(len(a) / 256)
    for name, (_, net) in SCALE_METHODS.items():
        t0 = time.time()
        sc = scale_fit(name, A[a])
        Z = scale_apply(sc, A)
        Zall = np.hstack([Z, OTHER9]).astype(np.float32)
        if net == "ftt":
            p, grads, acts = train_ftt(Zall[a], y[a], Zall[b])
        else:
            m = MLPClassifier(activation="selu" if net == "snn" else "relu", dropout=0.05 if net == "snn" else 0.1,
                              epochs=8, track=True).fit(Zall[a], y[a])
            p = m.predict_proba(Zall[b])[:, 1]
            grads, acts = m.grad_norms, activation_variances(m.net, Zall[b][:2048])
        p = np.nan_to_num(p, nan=0.5)                              # a diverged network predicts nothing
        g = np.asarray(grads[steps_per_epoch:])
        g = g[np.isfinite(g)]
        cd = np.linalg.norm(np.corrcoef(Z[a], rowvar=False) - ref_corr) / np.linalg.norm(ref_corr)
        rows.append({"Method": name, "Wasserstein Distance": mean_pairwise_w1(Z[sub]),
                     "Correlation Preservation": 1 - float(cd),
                     "Skewness Reduction": sk_raw - float(np.mean(np.abs(skew(Z[a], axis=0)))),
                     "Kurtosis": float(np.mean(np.abs(kurtosis(Z[a], axis=0)))),
                     "Activation Variance": float(np.mean(np.abs(np.log10(np.maximum(acts, 1e-12))))),
                     "Gradient Stability": float(g.std() / (g.mean() + 1e-12)),
                     "validation ROC-AUC": roc_auc_score(y[b], p),
                     "max/min feature SD": float(Z[a].std(0).max() / max(Z[a].std(0).min(), 1e-12)),
                     "seconds": time.time() - t0})
        traces[name] = {"grads": np.asarray(grads), "acts": acts}
        log(f"[scale] {name}: AUC={rows[-1]['validation ROC-AUC']:.3f} act={rows[-1]['Activation Variance']:.2f} "
            f"grad CV={rows[-1]['Gradient Stability']:.2f} ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows), traces


with timer("feature-scale benchmark"):
    SCALE_TAB, SCALE_TRACES = cached("scale_benchmark", run_scale_benchmark,
                                     fingerprint(X9.loc[trm9, CONT9], OTHER9.shape, "v1"))
SCALE_CRIT = {"Wasserstein Distance": "min", "Correlation Preservation": "max", "Skewness Reduction": "max",
              "Kurtosis": "min", "Activation Variance": "min", "Gradient Stability": "min"}
ref_ = SCALE_TAB[SCALE_TAB.Method == "No scaling (reference)"]
cand_ = SCALE_TAB[SCALE_TAB.Method != "No scaling (reference)"]
SCALE_TAB, SCALE_BEST = rank_select(cand_, SCALE_CRIT,
                                    guard=cand_["validation ROC-AUC"] >= ref_["validation ROC-AUC"].iloc[0] - MAX_AUC_DROP)
SCALE_TAB = pd.concat([SCALE_TAB, ref_], ignore_index=True)
SCALE_TAB["selected"] = SCALE_TAB["selected"].fillna(False).astype(bool)
save_table(SCALE_TAB, "09_scale_method_comparison")
show_table(SCALE_TAB, "Feature Scale Bias: method comparison")
plot_method_comparison(SCALE_TAB, [(k, v, k) for k, v in SCALE_CRIT.items()] + [("validation ROC-AUC", "max", "Validation ROC-AUC (guard)")],
                       "Feature Scale Bias — PowerTransform vs SNN (SELU) vs FT-Transformer", "09a_scale_method_comparison",
                       baseline="No scaling (reference)")
md(f"**Selected method: {SCALE_BEST}**: its input transform becomes the scaling of the model-ready data.")

In [ ]:
pal4 = {"No scaling (reference)": "#9aa5b4", "PowerTransform (Yeo-Johnson)": CAT_PALETTE[0], "SNN (SELU)": CAT_PALETTE[1],
        "FT-Transformer": CAT_PALETTE[2]}
fig, ax = plt.subplots(1, 2, figsize=(16, 4.6), facecolor="white")
for name, tr_ in SCALE_TRACES.items():
    lw = 2.6 if name == SCALE_BEST else 1.4
    ax[0].plot(np.arange(1, len(tr_["acts"]) + 1), tr_["acts"], marker="o", color=pal4[name], linewidth=lw, label=name)
    g = pd.Series(tr_["grads"]).rolling(10, min_periods=1).median()
    ax[1].plot(g.index, g.values, color=pal4[name], linewidth=lw, label=name)
ax[0].set_yscale("log")
ax[0].axhline(1, color=INK, linewidth=0.8)
ax[0].set_xlabel("hidden layer / transformer block")
ax[0].set_title("Activation variance per layer (1 = self-normalised)", loc="left")
ax[1].set_yscale("log")
ax[1].set_xlabel("training step")
ax[1].set_title("Gradient norm during training (rolling median of 10 steps)", loc="left")
ax[0].legend(fontsize=9.5)
fig.tight_layout()
show_fig(fig, "09b_scale_activation_gradient")

### 9.1 Apply the selected method's transform (fitted on the training split)

In [ ]:
SCALER = scale_fit(SCALE_BEST, X9.loc[trm9, CONT9].to_numpy(float)) if SCALE_PRESENT else None
ZC = pd.DataFrame(scale_apply(SCALER, X9[CONT9].to_numpy(float)), columns=CONT9, index=X9.index)
X_SCALED = X9.copy()
X_SCALED[CONT9] = ZC.to_numpy()
sd_after = ZC[trm9].std()
md(f"Max/min feature SD ratio: **{SCALE_RATIO:,.0f} → {sd_after.max() / sd_after.min():.2f}**; mean |skew|: "
   f"**{sk9.abs().mean():.2f} → {ZC[trm9].skew().abs().mean():.2f}**.")

top_sd = list(sd9.sort_values(ascending=False).index[:5])
top_sk = [c for c in sk9.abs().sort_values(ascending=False).index if c not in top_sd][:5]
SCALE_VARS = top_sd + top_sk
offb = (X9[SCALE_VARS].abs() > 3).set_axis(X9.patientunitstayid)
offa = (X_SCALED[SCALE_VARS].abs() > 3).set_axis(X9.patientunitstayid)
ib, ia = sample_indicator(offb, offa)
bias_dashboard("Feature Scale Bias", SCALE_BEST, X9, X_SCALED, SCALE_VARS, present=SCALE_PRESENT, same_axis=False,
               before_name="raw clinical units", after_name=f"{SCALE_BEST} input transform",
               indicator=dict(before=ib, after=ia, rows=[label(v) for v in SCALE_VARS], name="value outside ±3 (off unit scale)",
                              title="Off-Scale Heatmap: |value| > 3 (40 randomly sampled ICU stays)",
                              footer=f"Whole cohort, all continuous features: {(X9[CONT9].abs() > 3).to_numpy().mean():.1%} of cells "
                                     f"outside ±3 before, {(X_SCALED[CONT9].abs() > 3).to_numpy().mean():.1%} after."),
               note="Before and after are drawn on separate axes because the units change (raw clinical units vs a "
                    "unit-free scale). Five panels show the largest-scale features, five the most skewed.",
               fname="09c_dashboard_scale")

feat25 = list(sd9.sort_values(ascending=False).index[:25])
fig, ax = plt.subplots(1, 2, figsize=(16, 9), facecolor="white", sharey=True)
for a_, d_, ttl, c in ((ax[0], X9, "Before: raw clinical units (symmetric-log axis)", C_BEFORE),
                       (ax[1], X_SCALED, f"After: {SCALE_BEST}", C_AFTER)):
    rng_ = np.random.default_rng(SEED)
    rows_ = rng_.choice(len(d_), min(5000, len(d_)), replace=False)
    data_ = [d_[f].to_numpy(float)[rows_] for f in feat25][::-1]
    a_.boxplot(data_, vert=False, widths=0.6, showfliers=False, patch_artist=True,
               boxprops=dict(facecolor=to_rgba(c, 0.33), edgecolor=c), medianprops=dict(color=c, linewidth=1.6),
               whiskerprops=dict(color=c), capprops=dict(color=c))
    a_.set_yticks(range(1, 26), [label(f) for f in feat25][::-1], fontsize=9)
    a_.set_title(ttl, loc="left")
    a_.grid(axis="y", visible=False)
ax[0].set_xscale("symlog", linthresh=1)
fig.suptitle("All features on one axis: the 25 largest-scale features", x=0.01, ha="left", fontsize=15, weight="bold",
             color=TITLE_BLUE)
fig.tight_layout()
show_fig(fig, "09d_scale_all_features")

## 10. Encoding / Domain Bias, part B: shift between hospital regions
**What the bias is.** After the encoding fixes, the physiological measurements still differ systematically between
the US regions of the 208 hospitals (Midwest, Northeast, South, West). The differences come from case mix, devices
and charting practice. A model trained on one region then transfers worse to another, and a classifier can still
guess a stay's region from its measurements.

**The three methods from the table** act on the physiological measurement block. Age, height, weight and BMI
describe the patient, not the measurement process, so they are not aligned.
* **OTDA, Optimal Transport Domain Adaptation** (Courty et al., IEEE TPAMI 2017; `POT`): an entropic optimal-transport
  map moves every region's distribution onto the pooled reference distribution. It keeps the clinical feature space.
* **IRM, Invariant Risk Minimization** (Arjovsky et al., 2019; IRMv1 penalty): learns a 16-d representation whose
  optimal outcome classifier is the same in every region (environment = region).
* **Domain Generalization Transformer:** FT-Transformer (`rtdl_revisiting_models`) trained with V-REx
  (Krueger et al., ICML 2021), which penalises the variance of the risk across regions. Its [CLS] representation is
  the invariant representation.

**Evaluation.** Representation metrics use the training split (fit on 70%, measure on 30%). The prediction metrics
use **leave-one-region-out**: each method is refitted without the held-out region's labels, then a gradient-boosting
model is trained on the other regions and tested on the held-out region.

| Metric | Meaning | Better |
|---|---|---|
| MMD | maximum mean discrepancy between each region and the rest (RBF kernel), averaged | lower |
| Domain Classifier AUC | how well a classifier recognises the region from the representation | closer to 0.5 |
| Wasserstein Distance | 1-D Wasserstein distance between each region and the rest, averaged over dimensions | lower |
| Worst-Group Accuracy | lowest balanced accuracy over held-out regions | higher |
| Domain Generalization Gap | in-region AUROC - held-out-region AUROC | lower |
| Target-Domain AUROC | AUROC on the most-shifted region when it is held out | higher |
| OOD Performance | mean AUROC over held-out regions | higher |

In [ ]:
ANTHRO = ["age", "admission_height_cm", "admission_weight_kg", "bmi"]
ALIGN = [c for c in CONT9 if c not in ANTHRO]
OTHER_COLS = [c for c in CONT9 if c in ANTHRO]
Z_ALIGN = X_SCALED[ALIGN].to_numpy(float)
O_DOM = np.hstack([X_SCALED[OTHER_COLS].to_numpy(float), OTHER9]).astype(np.float32)
DOM = X_SCALED[DOMAIN_COL].astype(str).to_numpy()
Y9 = X_SCALED[TARGET].to_numpy()
with timer("domain shift at this step"):
    DOM_AUC0, DOM_MMD0 = cached("domain_shift_entry", lambda: domain_shift_metrics(Z_ALIGN[trm9], DOM[trm9], n_perm=100),
                                fingerprint(Z_ALIGN[trm9], "v1"))
DOMAIN_PRESENT = DOM_AUC0 > T["domain_auc"] or min(v["p_value"] for v in DOM_MMD0.values()) < T["mmd_pvalue"]
show_table(pd.DataFrame([{"region": k, "stays": v["n"], "MMD vs rest": v["mmd"], "permutation p": v["p_value"]}
                         for k, v in DOM_MMD0.items()]), "Region shift entering this step (training split)")
md(f"Region classifier AUC on the cleaned, scaled measurements: **{DOM_AUC0:.3f}** (rule: > {T['domain_auc']}). "
   f"Domain bias present at this step: **{DOMAIN_PRESENT}**.")

### 10.1 The three domain methods

In [ ]:
class OTDA:
    """Entropic OT map (Sinkhorn) from every domain to a reference sample; out-of-sample points follow the
    barycentric mapping of their nearest training samples (POT's SinkhornTransport)."""

    def __init__(self, n=1500, reg=0.1, seed=SEED):
        self.n, self.reg, self.seed = n, reg, seed

    def fit(self, Z, dom, ref_mask=None):
        import ot
        rng = np.random.default_rng(self.seed)
        pool = Z if ref_mask is None else Z[ref_mask]
        ref = pool[rng.choice(len(pool), min(self.n, len(pool)), replace=False)]
        self.maps = {}
        for d in np.unique(dom):
            Zd = Z[dom == d]
            src = Zd[rng.choice(len(Zd), min(self.n, len(Zd)), replace=False)]
            m = ot.da.SinkhornTransport(reg_e=self.reg, norm="median", max_iter=3000, method="sinkhorn_log")
            m.fit(Xs=src, Xt=ref)
            self.maps[d] = m
        return self

    def transform(self, Z, dom):
        out = Z.copy()
        for d, m in self.maps.items():
            msk = dom == d
            if msk.any():
                out[msk] = m.transform(Xs=Z[msk], batch_size=4096)
        return out


def train_irm(X, y, env, latent=16, epochs=30, batch=512, penalty=100.0, anneal=300, seed=SEED):
    """IRMv1 (Arjovsky et al., 2019): ERM + penalty on the gradient of each environment's risk w.r.t. a dummy scale."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    enc, head = mlp([X.shape[1], 64, latent]), nn.Linear(latent, 1)
    opt = torch.optim.Adam(list(enc.parameters()) + list(head.parameters()), lr=1e-3)
    envs, step = np.unique(env), 0
    for _ in range(epochs):
        perm = rng.permutation(len(X))
        for i in range(0, len(X), batch):
            b = perm[i:i + batch]
            erm, pen = 0.0, 0.0
            for e in envs:
                be = b[env[b] == e]
                if len(be) < 8:
                    continue
                w = torch.tensor(1.0, requires_grad=True)
                loss_e = F.binary_cross_entropy_with_logits(head(enc(_t(X[be]))).squeeze(1) * w, _t(y[be]))
                g = torch.autograd.grad(loss_e, [w], create_graph=True)[0]
                erm, pen = erm + loss_e, pen + g ** 2
            lam = penalty if step >= anneal else 1.0
            loss = (erm + lam * pen) / (lam if lam > 1 else 1.0)
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
    enc.eval()
    return lambda Z: np.vstack([enc(_t(Z[i:i + 8192])).detach().numpy() for i in range(0, len(Z), 8192)])


def train_dg_transformer(X, y, env, epochs=12, batch_per_env=96, vrex_lambda=10.0, anneal=100, patience=4, seed=SEED):
    """FT-Transformer (rtdl_revisiting_models, 1-block default configuration) trained with V-REx across regions;
    early stopping on the mean per-region validation loss. Returns the [CLS] representation (after the
    head's LayerNorm, before its ReLU) as a function."""
    from rtdl_revisiting_models import FTTransformer
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    X, y, env = np.asarray(X, np.float32), np.asarray(y, np.float32), np.asarray(env)
    envs = np.unique(env)
    val = rng.random(len(X)) < 0.1
    idx = {e: np.flatnonzero((env == e) & ~val) for e in envs}
    model = FTTransformer(n_cont_features=X.shape[1], cat_cardinalities=[], d_out=1,
                          **FTTransformer.get_default_kwargs(n_blocks=1))
    opt = model.make_default_optimizer()
    steps = max(1, int(np.ceil((~val).sum() / (batch_per_env * len(envs)))))
    best, best_state, bad, step = np.inf, None, 0, 0
    for _ in range(epochs):
        model.train()
        for _ in range(steps):
            parts = [rng.choice(idx[e], batch_per_env, replace=len(idx[e]) < batch_per_env) for e in envs]
            lg = model(_t(X[np.concatenate(parts)]), None).squeeze(1)
            yb = _t(y[np.concatenate(parts)])
            bnd = np.cumsum([0] + [len(p_) for p_ in parts])
            risks = torch.stack([F.binary_cross_entropy_with_logits(lg[a_:b_], yb[a_:b_]) for a_, b_ in zip(bnd[:-1], bnd[1:])])
            lam = vrex_lambda if step >= anneal else 1.0
            if step == anneal:
                opt = model.make_default_optimizer()
            loss = risks.mean() + lam * ((risks - risks.mean()) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
        model.eval()
        with torch.no_grad():
            vl = np.mean([F.binary_cross_entropy_with_logits(model(_t(X[val & (env == e)]), None).squeeze(1),
                                                             _t(y[val & (env == e)])).item()
                          for e in envs if (val & (env == e)).sum() > 0])
        if vl < best - 1e-4:
            best, bad, best_state = vl, 0, {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict(best_state)
    model.eval()
    cap = {}
    model.backbone.output.normalization.register_forward_hook(lambda m_, inp, out: cap.__setitem__("h", out))

    def represent(Z):
        outs = []
        with torch.no_grad():
            for i in range(0, len(Z), 4096):
                model(_t(np.asarray(Z[i:i + 4096], np.float32)), None)
                outs.append(cap["h"].numpy())
        return np.vstack(outs)
    return represent


DOM_METHODS = ["OTDA", "IRM", "Domain Generalization Transformer"]


def fit_represent(name, Z, y, dom, fit_mask, ref_mask=None):
    """Fit a method on rows fit_mask (OTDA may also use unlabeled target rows) and represent every row."""
    if name == "No adaptation (reference)":
        return Z.copy()
    if name == "OTDA":
        return OTDA().fit(Z[fit_mask], dom[fit_mask], None if ref_mask is None else ref_mask[fit_mask]).transform(Z, dom)
    lab = fit_mask if ref_mask is None else (fit_mask & ref_mask)          # labels only from source regions
    env = pd.Series(dom).astype("category").cat.codes.to_numpy()
    if name == "IRM":
        return train_irm(Z[lab], y[lab].astype(np.float32), env[lab])(Z)
    return train_dg_transformer(Z[lab], y[lab], env[lab])(Z)


def _std(R, mask):
    mu, sd = R[mask].mean(0), R[mask].std(0) + 1e-9
    return (R - mu) / sd


def rep_metrics(R, dom, mask, rng):
    Rs = _std(R, mask)[mask]
    d_ = dom[mask]
    auc, _ = domain_auc(Rs, d_)
    sub = Rs[rng.choice(len(Rs), min(600, len(Rs)), replace=False)]
    sq = ((sub[:, None, :] - sub[None, :, :]) ** 2).sum(-1)
    gamma = 1.0 / np.median(sq[sq > 0])
    mmds, w1s = [], []
    for g in np.unique(d_):
        a_, b_ = np.flatnonzero(d_ == g), np.flatnonzero(d_ != g)
        A_ = Rs[rng.choice(a_, min(500, len(a_)), replace=False)]
        B_ = Rs[rng.choice(b_, min(500, len(b_)), replace=False)]
        mmds.append(mmd_rbf(A_, B_, gamma))
        w1s.append(np.mean([wasserstein_distance(Rs[a_, j], Rs[b_, j]) for j in range(Rs.shape[1])]))
    return auc, float(np.mean(mmds)), float(np.mean(w1s))


DOM_KEY = fingerprint(Z_ALIGN[trm9], O_DOM.shape, "v1")
# bump a method's version when its code changes, so only that method is recomputed
DOM_VERSION = {"No adaptation (reference)": "v1", "OTDA": "v1", "IRM": "v1", "Domain Generalization Transformer": "v2"}


def domain_split():
    tr = np.flatnonzero(trm9)
    a, b = train_test_split(tr, test_size=0.3, random_state=SEED, stratify=Y9[tr])
    A_m, B_m = np.zeros(len(DOM), bool), np.zeros(len(DOM), bool)
    A_m[a], B_m[b] = True, True
    return A_m, B_m


def domain_setup():
    _, B_m = domain_split()
    regions = [r for r in np.unique(DOM[trm9]) if (B_m & (DOM == r)).sum() >= 150 and Y9[B_m & (DOM == r)].sum() >= 15]
    # target domain = the region a classifier recognises best before alignment
    ovr = {r: roc_auc_score((DOM[B_m] == r).astype(int),
                            cross_val_predict(HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
                                              Z_ALIGN[B_m], (DOM[B_m] == r).astype(int), cv=3,
                                              method="predict_proba")[:, 1]) for r in regions}
    return {"regions evaluated": regions, "target domain (most shifted)": max(ovr, key=ovr.get),
            "one-vs-rest AUC before": ovr}


def domain_row(name, regions, target):
    t0 = time.time()
    rng = np.random.default_rng(SEED)
    A_m, B_m = domain_split()
    R = fit_represent(name, Z_ALIGN, Y9, DOM, A_m)
    dauc, mmd_, w1 = rep_metrics(R, DOM, B_m, rng)
    full = np.hstack([R, O_DOM])
    id_auc = roc_auc_score(Y9[B_m], hgb().fit(full[A_m], Y9[A_m]).predict_proba(full[B_m])[:, 1])
    ood, idd, bacc = {}, {}, {}
    for r in regions:
        src = A_m & (DOM != r)
        Rr = fit_represent(name, Z_ALIGN, Y9, DOM, A_m, ref_mask=(DOM != r)) if name != "No adaptation (reference)" else R
        fr = np.hstack([Rr, O_DOM])
        m = hgb().fit(fr[src], Y9[src])
        thr = np.quantile(m.predict_proba(fr[src])[:, 1], 1 - Y9[src].mean())
        te, ti = B_m & (DOM == r), B_m & (DOM != r)
        p_te = m.predict_proba(fr[te])[:, 1]
        ood[r] = roc_auc_score(Y9[te], p_te)
        idd[r] = roc_auc_score(Y9[ti], m.predict_proba(fr[ti])[:, 1])
        bacc[r] = balanced_accuracy_score(Y9[te], (p_te >= thr).astype(int))
    row = {"Method": name, "MMD": mmd_, "Domain Classifier AUC": dauc, "Wasserstein Distance": w1,
           "Worst-Group Accuracy": min(bacc.values()),
           "Domain Generalization Gap": float(np.mean([idd[r] - ood[r] for r in regions])),
           "Target-Domain AUROC": ood[target], "OOD Performance": float(np.mean(list(ood.values()))),
           "in-domain ROC-AUC": id_auc, "seconds": time.time() - t0}
    log(f"[domain] {name}: dAUC={dauc:.3f} MMD={mmd_:.4f} OOD={row['OOD Performance']:.3f} ({row['seconds']:.0f}s)")
    return row


with timer("domain benchmark (with leave-one-region-out refits)"):
    DOM_INFO = cached("domain_setup", domain_setup, DOM_KEY)
    DOM_TAB = pd.DataFrame([cached("domain_row", lambda n=n: domain_row(n, DOM_INFO["regions evaluated"],
                                                                       DOM_INFO["target domain (most shifted)"]),
                                   fingerprint(DOM_KEY, n, DOM_VERSION[n]))
                            for n in ["No adaptation (reference)"] + DOM_METHODS])
DOM_CRIT = {"MMD": "min", "Domain Classifier AUC": ("target", 0.5), "Wasserstein Distance": "min",
            "Worst-Group Accuracy": "max", "Domain Generalization Gap": "min", "Target-Domain AUROC": "max",
            "OOD Performance": "max"}
ref_ = DOM_TAB[DOM_TAB.Method == "No adaptation (reference)"]
cand_ = DOM_TAB[DOM_TAB.Method != "No adaptation (reference)"]
DOM_TAB, DOM_BEST = rank_select(cand_, DOM_CRIT,
                                guard=cand_["in-domain ROC-AUC"] >= ref_["in-domain ROC-AUC"].iloc[0] - MAX_AUC_DROP)
DOM_TAB = pd.concat([DOM_TAB, ref_], ignore_index=True)
DOM_TAB["selected"] = DOM_TAB["selected"].fillna(False).astype(bool)
save_table(DOM_TAB, "10_domain_method_comparison")
show_table(pd.DataFrame([DOM_INFO]).T.rename(columns={0: "value"}).reset_index(names="protocol"), "Evaluation protocol")
show_table(DOM_TAB, "Encoding / Domain Bias: method comparison")
plot_method_comparison(DOM_TAB, [(k, v, k) for k, v in DOM_CRIT.items()],
                       "Encoding / Domain Bias — IRM vs OTDA vs Domain Generalization Transformer",
                       "10a_domain_method_comparison", baseline="No adaptation (reference)")
md(f"**Selected method: {DOM_BEST}**")

### 10.2 Apply the selected method (fitted on the training split)
OTDA keeps the clinical feature space: the aligned values replace the measurement block of the model-ready data.
IRM and the Domain Generalization Transformer produce a learned representation: if one of them is selected, its
region-invariant features (`dom_z*`) replace the measurement block in the model-ready data. The clinical-units file
always keeps the original measurements.

In [ ]:
LATENT = DOM_BEST != "OTDA"
with timer(f"apply {DOM_BEST}"):
    R_DOM = cached("domain_apply", lambda: fit_represent(DOM_BEST, Z_ALIGN, Y9, DOM, trm9) if DOMAIN_PRESENT else Z_ALIGN,
                   fingerprint(Z_ALIGN, DOM_BEST, DOMAIN_PRESENT, DOM_VERSION.get(DOM_BEST), "v1"))
if LATENT and DOMAIN_PRESENT:
    # learned features: drop dimensions that are constant on the training split, then standardise with training
    # statistics so the model-ready data keeps the feature-scale fix of section 9
    keep_ = R_DOM[trm9].std(0) > 1e-6
    R_DOM = (R_DOM[:, keep_] - R_DOM[trm9][:, keep_].mean(0)) / R_DOM[trm9][:, keep_].std(0)
    DOM_COLS = [f"dom_z{i + 1}" for i in range(R_DOM.shape[1])]
else:
    DOM_COLS = ALIGN
DOM_FEATS = pd.DataFrame(R_DOM, columns=DOM_COLS, index=X_SCALED.index)
auc1, _ = domain_auc(_std(R_DOM, trm9)[trm9], DOM[trm9])
md(f"Region classifier AUC (training split): **{DOM_AUC0:.3f} → {auc1:.3f}** (0.5 = the region can no longer be "
   f"recognised).")

In [ ]:
def region_shift(F_, dom, mask):
    """|region mean - pooled mean| / pooled SD for every region x feature."""
    mu, sd = F_[mask].mean(0), F_[mask].std(0) + 1e-9
    return pd.DataFrame({r: np.abs(F_[mask & (dom == r)].mean(0) - mu) / sd for r in np.unique(dom[mask])}).T


if LATENT and DOMAIN_PRESENT:
    from sklearn.decomposition import PCA
    pca_b = PCA(10, random_state=SEED).fit(_std(Z_ALIGN, trm9)[trm9])
    pca_a = PCA(10, random_state=SEED).fit(_std(R_DOM, trm9)[trm9])
    VB = pd.DataFrame(pca_b.transform(_std(Z_ALIGN, trm9)), columns=[f"PC{i + 1}" for i in range(10)])
    VA = pd.DataFrame(pca_a.transform(_std(R_DOM, trm9)), columns=[f"PC{i + 1}" for i in range(10)])
    DOM_VARS = list(VB.columns)
    note_ = (f"{DOM_BEST} produces a learned representation, so the panels compare the first 10 principal components "
             f"of the measurement block (before) with those of the {DOM_BEST} representation (after).")
else:
    VB = pd.DataFrame(Z_ALIGN, columns=ALIGN)
    VA = pd.DataFrame(R_DOM, columns=ALIGN)
    sh0 = region_shift(Z_ALIGN, DOM, trm9)
    sh0.columns = ALIGN
    DOM_VARS = list(sh0.max().sort_values(ascending=False).index[:10])
    note_ = "Values are on the scaled (unit-free) axis of section 9; the ten variables with the largest region shift are shown."
shb, sha = region_shift(VB[DOM_VARS].to_numpy(), DOM, trm9), region_shift(VA[DOM_VARS].to_numpy(), DOM, trm9)
shb.columns, sha.columns = DOM_VARS, DOM_VARS
cellb = pd.DataFrame(shb.reindex(DOM).fillna(0).to_numpy() > 0.1, columns=DOM_VARS, index=X_SCALED.patientunitstayid)
cella = pd.DataFrame(sha.reindex(DOM).fillna(0).to_numpy() > 0.1, columns=DOM_VARS, index=X_SCALED.patientunitstayid)
ib, ia = sample_indicator(cellb, cella)
bias_dashboard("Encoding / Domain Bias (part B: region shift)", DOM_BEST, VB, VA, DOM_VARS, present=DOMAIN_PRESENT,
               before_name="scaled measurements", after_name=f"aligned with {DOM_BEST}",
               indicator=dict(before=ib, after=ia, rows=[label(v) for v in DOM_VARS], name="region-shifted (>0.1 SD)",
                              title="Domain-Shift Heatmap: the stay's region mean differs by >0.1 SD (40 random stays)",
                              footer=f"Whole cohort: {cellb.to_numpy().mean():.1%} of these cells sit in a shifted region before, "
                                     f"{cella.to_numpy().mean():.1%} after."),
               note=note_, fname="10b_dashboard_domain")

from sklearn.decomposition import PCA
regions_ = sorted(np.unique(DOM))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4), facecolor="white", gridspec_kw={"width_ratios": [1, 1, 1.1]})
rng_ = np.random.default_rng(SEED)
pts = rng_.choice(np.flatnonzero(trm9), 2500, replace=False)
for ax, F_, ttl in ((axes[0], Z_ALIGN, "Before (scaled measurements)"), (axes[1], R_DOM, f"After ({DOM_BEST})")):
    P2 = PCA(2, random_state=SEED).fit_transform(_std(F_, trm9)[pts])
    for k, r in enumerate(regions_):
        m = DOM[pts] == r
        ax.scatter(P2[m, 0], P2[m, 1], s=10, alpha=0.55, color=CAT_PALETTE[k], marker=CAT_MARKERS[k], label=r)
    ax.set_title(f"{ttl}: first two principal components", loc="left", fontsize=11.5)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0].legend(markerscale=2, fontsize=9.5, title="hospital region")
ovr_b, ovr_a = [], []
for r in regions_:
    yy = (DOM[trm9] == r).astype(int)
    for F_, lst in ((Z_ALIGN, ovr_b), (R_DOM, ovr_a)):
        lst.append(roc_auc_score(yy, cross_val_predict(HistGradientBoostingClassifier(max_iter=100, random_state=SEED),
                                                        _std(F_, trm9)[trm9], yy, cv=3, method="predict_proba")[:, 1]))
ba_bars(axes[2], regions_, ovr_b, ovr_a, "Region recognisable from the data? (one-vs-rest AUC, 0.5 = no)",
        value_fmt=lambda v: f"{v:.3f}")
axes[2].set_xlim(0.4, 1.08)
fig.tight_layout()
show_fig(fig, "10c_domain_regions")

## 11. Demographic Bias
**What the bias is.** A model's decisions differ systematically between protected groups (gender, ethnicity)
beyond what the patients' condition explains. Examples: one group is flagged as high-risk more often (statistical
parity), or deaths in one group are missed more often (equal opportunity).

**The model-ready data.** The demographic check is made on the data a model will actually use: the scaled
continuous features (domain-aligned), binary flags, missingness indicators and one-hot categories, including the
protected attributes.

**The three methods from the table** (each trains a mortality classifier):
* **Adversarial Debiasing** (Zhang, Lemoine & Mitchell, AIES 2018): a predictor is trained against an adversary that
  tries to recover the protected group from the prediction and the label (equalised-odds variant).
* **Counterfactual Algorithmic Recourse (DiCE)** (Mothilal, Sharma & Tan, FAT* 2020): DiCE-style counterfactual
  search finds the patients whose decision **flips when only their gender or ethnicity is changed**. Those
  counterfactuals are added to the training data with the original label (counterfactual augmentation), so the
  retrained model becomes insensitive to the protected attributes.
* **Fairness-Aware Contrastive Learning** (FSCL, Park et al., CVPR 2022): supervised contrastive pre-training whose
  negatives are restricted to the anchor's own protected group. Rows are augmented with SCARF feature corruption,
  followed by a linear classifier on the frozen encoder.

**Metrics.** Group fairness uses 3-fold out-of-fold predictions on the training split at a prevalence-matched
threshold. Counterfactual metrics use DiCE's gradient-based counterfactual explanations for 300 patients per method:
minimal changes to the clinical measurements that flip the model's decision.

| Metric | Meaning | Better |
|---|---|---|
| Statistical Parity Difference | worst \|selection-rate gap\| between a group and the privileged group | lower |
| Disparate Impact Difference | worst \|1 - selection-rate ratio\| | lower |
| Equal Opportunity Difference | worst \|true-positive-rate gap\| | lower |
| Equalized Odds Difference | worst max(\|TPR gap\|, \|FPR gap\|) | lower |
| Counterfactual Validity | share of counterfactual explanations that really flip the decision | higher |
| Proximity | mean change per varied feature, in MAD units | lower |
| Sparsity | mean number of features changed | lower |

In [ ]:
M_CONT_COLS = OTHER_COLS + DOM_COLS
M_COLS = M_CONT_COLS + OTHER9_NAMES
M_ALL = pd.DataFrame(np.hstack([X_SCALED[OTHER_COLS].to_numpy(float), DOM_FEATS.to_numpy(float), OTHER9]),
                     columns=M_COLS, index=X_SCALED.index).astype(np.float32)
GRP = X9[PROTECTED].astype(str).reset_index(drop=True)
PROT_SETS = [np.array([i for i, c in enumerate(M_COLS) if c.startswith(f"{a}_")]) for a in PROTECTED]
VARY_IDX = np.array([M_COLS.index(c) for c in DOM_COLS])
Mtr = M_ALL[trm9].to_numpy()
ytr9 = Y9[trm9]
Gtr = GRP[trm9].reset_index(drop=True)


def worst_fairness(y, p, G):
    w = {"Statistical Parity Difference": 0.0, "Disparate Impact Difference": 0.0,
         "Equal Opportunity Difference": 0.0, "Equalized Odds Difference": 0.0}
    tabs = {}
    for a in PROTECTED:
        tab, _ = fairness_metrics(y, p, G[a].values, PRIVILEGED[a])
        tabs[a] = tab
        if len(tab):
            w["Statistical Parity Difference"] = max(w["Statistical Parity Difference"], tab.SPD.abs().max())
            w["Disparate Impact Difference"] = max(w["Disparate Impact Difference"], (1 - tab.DI).abs().max())
            w["Equal Opportunity Difference"] = max(w["Equal Opportunity Difference"], tab.EOD.abs().max())
            w["Equalized Odds Difference"] = max(w["Equalized Odds Difference"], tab.EqOdds.max())
    return w, tabs


with timer("demographic check on the model-ready training data"):
    OOF9 = cached("demographic_entry_oof", lambda: oof_proba(pd.DataFrame(Mtr), ytr9), fingerprint(Mtr, "v1"))
DEM_ENTRY, DEM_TABS = worst_fairness(ytr9, OOF9, Gtr)
viol = {a: t[(t.DI < T["di_low"]) | (t.DI > T["di_high"]) | (t.SPD.abs() > T["spd_abs"])].group.tolist()
        for a, t in DEM_TABS.items()}
DEMOGRAPHIC_PRESENT = any(len(v) for v in viol.values())
for a, t in DEM_TABS.items():
    show_table(t, f"Group fairness of an out-of-fold model on the model-ready training data — {a} (privileged: {PRIVILEGED[a]})")
md(f"Groups breaking the four-fifths rule or |SPD| > 0.1 at this step: **{viol}**. Demographic Bias present at this "
   f"step: **{DEMOGRAPHIC_PRESENT}**" + ("" if DEMOGRAPHIC_PRESENT else
   " - the raw-data violation (ethnicity coded as NaN) was an encoding artefact and disappeared with the encoding "
   "harmonisation in section 3. The methods are still benchmarked below, but none is applied, so before = after."))

### 11.1 The three methods and DiCE counterfactual explanations

In [ ]:
def train_adversarial(X, y, grp, alpha=1.0, epochs=15, batch=256, seed=SEED):
    """Adversarial debiasing: the adversary sees the predicted probability and the true label."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    k = int(grp.max()) + 1
    pred, adv = mlp([X.shape[1], 128, 64, 1], dropout=0.1), mlp([2, 32, k])
    op, oa = torch.optim.Adam(pred.parameters(), lr=1e-3), torch.optim.Adam(adv.parameters(), lr=1e-3)
    for ep in range(epochs):
        pred.train()
        perm = rng.permutation(len(X))
        for i in range(0, len(X), batch):
            b = perm[i:i + batch]
            x, yy, g = _t(X[b]), _t(y[b]), torch.as_tensor(grp[b], dtype=torch.long)
            lg = pred(x).squeeze(1)
            la = F.cross_entropy(adv(torch.stack([torch.sigmoid(lg), yy], 1).detach()), g)
            oa.zero_grad()
            la.backward()
            oa.step()
            lp = F.binary_cross_entropy_with_logits(lg, yy)
            la2 = F.cross_entropy(adv(torch.stack([torch.sigmoid(lg), yy], 1)), g)
            loss = lp - alpha * min(1.0, ep / 5) * la2
            op.zero_grad()
            loss.backward()
            op.step()
    pred.eval()
    return lambda x: pred(x).squeeze(1)


class FairSupConLoss(nn.Module):
    """FSCL loss (Park et al., CVPR 2022, method='FSCL', group_norm=0): positives share the label; the denominator
    keeps only negatives (other label) from the anchor's own protected group."""

    def __init__(self, temperature=0.1, base_temperature=0.07):
        super().__init__()
        self.t, self.bt = temperature, base_temperature

    def forward(self, feats, labels, sens):
        b, v = feats.shape[0], feats.shape[1]
        mask = torch.eq(labels.view(-1, 1), labels.view(1, -1)).float().repeat(v, v)
        smask = torch.eq(sens.view(-1, 1), sens.view(1, -1)).float().repeat(v, v)
        contrast = torch.cat(torch.unbind(feats, dim=1), dim=0)
        logits = contrast @ contrast.T / self.t
        logits = logits - logits.max(dim=1, keepdim=True)[0].detach()
        lmask = torch.ones_like(mask) - torch.eye(b * v)
        mask = mask * lmask
        fair = lmask * (1.0 - mask) * smask
        denom = (torch.exp(logits) * fair).sum(1, keepdim=True)
        log_prob = logits - torch.log(denom + (denom == 0).float())
        mlpp = (mask * log_prob).sum(1) / mask.sum(1).clamp(min=1.0)
        return (-(self.t / self.bt) * mlpp).view(v, b).mean()


def train_fscl(X, y, sens, epochs=30, batch=128, seed=SEED):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    enc = nn.Sequential(nn.Linear(X.shape[1], 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Linear(512, 256),
                        nn.BatchNorm1d(256), nn.ReLU())
    head = nn.Sequential(nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 128))
    crit = FairSupConLoss()
    opt = torch.optim.SGD(list(enc.parameters()) + list(head.parameters()), lr=0.05, momentum=0.9, weight_decay=1e-4)
    yt, st = torch.as_tensor(y, dtype=torch.long), torch.as_tensor(sens, dtype=torch.long)
    for _ in range(epochs):
        enc.train()
        perm = rng.permutation(len(X))
        for i in range(0, len(X) - 1, batch):
            b = perm[i:i + batch]
            if len(b) < 4:
                continue
            views = []
            for _ in range(2):                                   # SCARF corruption (60% of features)
                corrupt_ = rng.random((len(b), X.shape[1])) < 0.6
                views.append(np.where(corrupt_, X[rng.integers(0, len(X), len(b))], X[b]).astype(np.float32))
            z = F.normalize(head(enc(_t(np.vstack(views)))), dim=1)
            loss = crit(torch.stack([z[:len(b)], z[len(b):]], 1), yt[b], st[b])
            opt.zero_grad()
            loss.backward()
            opt.step()
    enc.eval()
    with torch.no_grad():
        E = enc(_t(X)).numpy()
    lr_ = LogisticRegression(max_iter=3000).fit(E, y)
    w, b0 = torch.tensor(lr_.coef_[0], dtype=torch.float32), float(lr_.intercept_[0])
    return lambda x: enc(x) @ w + b0


def mlp_logit_fn(X, y, epochs=15, seed=SEED):
    m = MLPClassifier(epochs=epochs, seed=seed).fit(X, y)
    m.net.eval()
    return lambda x: m.net(x).squeeze(1)


def protected_counterfactuals(X):
    """Every alternative value of each protected one-hot block (other attributes unchanged)."""
    out = []
    for cols in PROT_SETS:
        for s in range(1, len(cols)):
            Xf = X.copy()
            Xf[:, cols] = np.roll(X[:, cols], s, axis=1)
            out.append(Xf)
    return out


def predict_logit(fn, X):
    with torch.no_grad():
        return np.concatenate([fn(_t(X[i:i + 8192])).numpy() for i in range(0, len(X), 8192)])


def train_dice_cda(X, y, seed=SEED):
    """DiCE-style counterfactual search over the protected attributes, then counterfactual augmentation."""
    f0 = mlp_logit_fn(X, y, seed=seed)
    z0 = predict_logit(f0, X)
    thr = np.quantile(z0, 1 - y.mean())
    dec = z0 >= thr
    aug_X, aug_y = [X], [y]
    for Xf in protected_counterfactuals(X):
        flip = (predict_logit(f0, Xf) >= thr) != dec
        aug_X.append(Xf[flip])
        aug_y.append(y[flip])                           # the label does not change with the protected attribute
    Xa, ya = np.vstack(aug_X), np.concatenate(aug_y)
    return mlp_logit_fn(Xa, ya, seed=seed), int(len(Xa) - len(X))


def dice_explanations(fn, X0, thr, mad, steps=150, lr=0.05, lam=0.5, margin=0.25):
    """Gradient-based DiCE counterfactuals (one per patient) on the clinical measurement block, followed by DiCE's
    post-hoc sparsity enhancement (undo the smallest changes as long as the decision stays flipped)."""
    x0 = _t(X0)
    with torch.no_grad():
        desired = (fn(x0) < thr).float()
    madt = _t(mad)
    delta = torch.zeros(len(X0), len(VARY_IDX), requires_grad=True)
    opt = torch.optim.Adam([delta], lr=lr)
    vi = torch.as_tensor(VARY_IDX, dtype=torch.long)
    for _ in range(steps):
        x = x0.clone()
        x[:, vi] = x0[:, vi] + delta
        z = fn(x)
        yl = torch.where(desired == 1, F.relu(thr + margin - z), F.relu(z - (thr - margin)))
        loss = (yl + lam * (delta.abs() / madt).mean(1)).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    d = delta.detach().clone()
    d[(d.abs() / madt) < 0.1] = 0

    def valid(dd):
        x = x0.clone()
        x[:, vi] = x0[:, vi] + dd
        with torch.no_grad():
            return ((fn(x) >= thr).float() == desired).numpy()

    ok = valid(d)
    order = np.argsort(np.where(d.numpy() != 0, np.abs(d.numpy()) / mad, np.inf), axis=1)
    for k in range(order.shape[1]):
        cand = d.clone()
        rows = np.arange(len(d))
        cand[rows, order[:, k]] = 0
        keep = valid(cand) & ok
        d[keep] = cand[keep]
    ok = valid(d)
    dn = np.abs(d.numpy()) / mad
    changed = dn > 0
    return {"Counterfactual Validity": float(ok.mean()),
            "Proximity": float(dn[ok].mean()) if ok.any() else np.nan,
            "Sparsity": float(changed[ok].sum(1).mean()) if ok.any() else np.nan}


def cf_flip_rate(fn, X, thr):
    dec = predict_logit(fn, X) >= thr
    flips = np.zeros(len(X), bool)
    for Xf in protected_counterfactuals(X):
        flips |= (predict_logit(fn, Xf) >= thr) != dec
    return flips


DEM_METHODS = ["No mitigation (reference)", "Adversarial Debiasing", "DiCE counterfactual recourse",
               "Fairness-Aware Contrastive Learning"]


def run_demographic_benchmark(folds=3, n_cf=100):
    rng = np.random.default_rng(SEED)
    gkey = Gtr.astype(str).agg("|".join, axis=1).to_numpy()
    gcode = pd.Series(gkey).astype("category").cat.codes.to_numpy()
    mad = np.median(np.abs(Mtr[:, VARY_IDX] - np.median(Mtr[:, VARY_IDX], 0)), 0) * 1.4826
    mad[mad < 1e-3] = 1e-3
    rows, flips_all, oofs = [], {}, {}
    for m in DEM_METHODS:
        t0 = time.time()
        oof, cf_rows, flips, extra = np.zeros(len(ytr9)), [], np.zeros(len(ytr9), bool), 0
        for k, (tr, te) in enumerate(StratifiedKFold(folds, shuffle=True, random_state=SEED).split(Mtr, ytr9)):
            if m == "No mitigation (reference)":
                fn = mlp_logit_fn(Mtr[tr], ytr9[tr], seed=SEED + k)
            elif m == "Adversarial Debiasing":
                fn = train_adversarial(Mtr[tr], ytr9[tr].astype(np.float32), gcode[tr], seed=SEED + k)
            elif m == "DiCE counterfactual recourse":
                fn, n_aug = train_dice_cda(Mtr[tr], ytr9[tr], seed=SEED + k)
                extra += n_aug
            else:
                fn = train_fscl(Mtr[tr], ytr9[tr], gcode[tr], seed=SEED + k)
            z_tr = predict_logit(fn, Mtr[tr])
            thr = float(np.quantile(z_tr, 1 - ytr9[tr].mean()))
            oof[te] = 1 / (1 + np.exp(-predict_logit(fn, Mtr[te])))
            flips[te] = cf_flip_rate(fn, Mtr[te], thr)
            sel = rng.choice(te, n_cf, replace=False)
            cf_rows.append(dice_explanations(fn, Mtr[sel], thr, mad))
        w, _ = worst_fairness(ytr9, oof, Gtr)
        cf = pd.DataFrame(cf_rows).mean()
        rows.append({"Method": m, **w, **cf.to_dict(), "protected-attribute flip rate": float(flips.mean()),
                     "ROC-AUC": roc_auc_score(ytr9, oof), "counterfactual rows added": extra,
                     "seconds": time.time() - t0})
        flips_all[m], oofs[m] = flips, oof
        log(f"[demographic] {m}: SPD={w['Statistical Parity Difference']:.3f} EqOdds={w['Equalized Odds Difference']:.3f} "
            f"valid={cf['Counterfactual Validity']:.2f} AUC={rows[-1]['ROC-AUC']:.3f} ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows), flips_all, oofs


with timer("demographic benchmark"):
    DEM_TAB, DEM_FLIPS, DEM_OOF = cached("demographic_benchmark", run_demographic_benchmark, fingerprint(Mtr, "v1"))
DEM_CRIT = {"Statistical Parity Difference": "min", "Disparate Impact Difference": "min",
            "Equal Opportunity Difference": "min", "Equalized Odds Difference": "min",
            "Counterfactual Validity": "max", "Proximity": "min", "Sparsity": "min"}
ref_ = DEM_TAB[DEM_TAB.Method == "No mitigation (reference)"]
cand_ = DEM_TAB[DEM_TAB.Method != "No mitigation (reference)"]
DEM_TAB, DEM_BEST = rank_select(cand_, DEM_CRIT, guard=cand_["ROC-AUC"] >= ref_["ROC-AUC"].iloc[0] - MAX_AUC_DROP)
DEM_TAB = pd.concat([DEM_TAB, ref_], ignore_index=True)
DEM_TAB["selected"] = DEM_TAB["selected"].fillna(False).astype(bool)
save_table(DEM_TAB, "11_demographic_method_comparison")
show_table(DEM_TAB, "Demographic Bias: method comparison (3-fold out-of-fold, training split)")
plot_method_comparison(DEM_TAB, [(k, v, k) for k, v in DEM_CRIT.items()],
                       "Demographic Bias — Adversarial Debiasing vs DiCE recourse vs Fairness-Aware Contrastive Learning",
                       "11a_demographic_method_comparison", baseline="No mitigation (reference)")
md(f"**Best method in the benchmark: {DEM_BEST}.** " + ("It is applied because the bias is present at this step."
   if DEMOGRAPHIC_PRESENT else "It is **not applied**: no protected group breaks the fairness rules on the "
   "model-ready data, so the data stays unchanged (before = after)."))

### 11.2 Apply the selected method
* **Adversarial Debiasing / Fairness-Aware Contrastive Learning** act on the *model*, not on the feature values.
  The method is trained on the whole training split, and its fairness-mitigated risk for every stay is delivered as
  `fair_risk_score` (out-of-sample for test patients). Use it, or retrain the method with the code above, whenever
  fair decisions are needed.
* **DiCE counterfactual recourse** acts on the *data*: its counterfactual training rows are saved to
  `respiratory_train_counterfactual_rows.csv`.

In [ ]:
DEM_APPLIED = DEMOGRAPHIC_PRESENT
after_key = DEM_BEST if DEM_APPLIED else "No mitigation (reference)"
FAIR_RISK = None
MB = M_ALL[trm9].reset_index(drop=True)
MA = MB


def train_fair_model(method, X, y, grp):
    if method == "Adversarial Debiasing":
        return train_adversarial(X, y.astype(np.float32), grp)
    return train_fscl(X, y, grp)


if DEM_APPLIED and DEM_BEST == "DiCE counterfactual recourse":
    f0_ = mlp_logit_fn(Mtr, ytr9)
    z0_ = predict_logit(f0_, Mtr)
    thr0_ = np.quantile(z0_, 1 - ytr9.mean())
    flips_ = [(Xf, (predict_logit(f0_, Xf) >= thr0_) != (z0_ >= thr0_)) for Xf in protected_counterfactuals(Mtr)]
    CF_ROWS = pd.DataFrame(np.vstack([Xf[f] for Xf, f in flips_]), columns=M_COLS)
    CF_ROWS[TARGET] = np.concatenate([ytr9[f] for _, f in flips_])
    CF_ROWS["is_counterfactual"] = 1
    CF_ROWS.to_csv(DATA_OUT / "respiratory_train_counterfactual_rows.csv", index=False)
    MA = pd.concat([MB, CF_ROWS[M_COLS]], ignore_index=True)
    md(f"DiCE counterfactual augmentation: {len(CF_ROWS):,} counterfactual training rows saved.")
elif DEM_APPLIED:
    gk_ = Gtr.astype(str).agg("|".join, axis=1)
    with timer(f"train {DEM_BEST} on the whole training split"):
        fn_fair = train_fair_model(DEM_BEST, Mtr, ytr9, pd.Series(gk_).astype("category").cat.codes.to_numpy())
    FAIR_RISK = 1 / (1 + np.exp(-predict_logit(fn_fair, M_ALL.to_numpy())))
    MB = MB.assign(predicted_risk=DEM_OOF["No mitigation (reference)"])
    MA = MA.assign(predicted_risk=DEM_OOF[DEM_BEST])
    md(f"{DEM_BEST} is a model-level method: its fairness-mitigated risk is delivered as `fair_risk_score` "
       f"for all {len(FAIR_RISK):,} stays.")
LABELS["predicted_risk"] = "Predicted mortality risk"
SHORT["predicted_risk"] = "Risk"
CORE_VARS = ["age", "apache_iva_score", "hr_mean", "rr_mean", "spo2_min", "pao2_min", "pf_ratio_min", "lactate_max",
             "creatinine_max", "gcs_total", "admission_weight_kg", "bmi"]
DEM_VARS = ([c for c in CORE_VARS if c in M_CONT_COLS] + [c for c in M_CONT_COLS if c not in CORE_VARS])[:10]
if "predicted_risk" in MA:
    DEM_VARS = ["predicted_risk"] + DEM_VARS[:9]
bias_dashboard("Demographic Bias", DEM_BEST, MB, MA, DEM_VARS, present=DEM_APPLIED,
               before_name="model-ready training data", after_name=DEM_BEST if DEM_APPLIED else "unchanged",
               indicator=dict(before=waffle(DEM_FLIPS["No mitigation (reference)"]), after=waffle(DEM_FLIPS[after_key]),
                              rows=[f"stays {40 * i + 1}-{40 * i + 40}" for i in range(8)],
                              name="decision flips when gender/ethnicity is changed",
                              title="Counterfactual-Fairness Map (320 randomly sampled training stays)",
                              footer=f"Share of stays whose decision depends on a protected attribute: "
                                     f"{DEM_FLIPS['No mitigation (reference)'].mean():.1%} before, {DEM_FLIPS[after_key].mean():.1%} after."),
               note="Feature values are on the scaled model-ready axis." + (
                    " The bias is not present at this step, so no method is applied and before = after." if not DEM_APPLIED
                    else f" {DEM_BEST} changes the model's decisions, not the feature values: the first panel "
                         "(out-of-fold predicted risk) shows the change; the clinical features are identical."
                    if FAIR_RISK is not None else ""),
               fname="11b_dashboard_demographic")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), facecolor="white")
thr_b = np.quantile(DEM_OOF["No mitigation (reference)"], 1 - ytr9.mean())
thr_a = np.quantile(DEM_OOF[DEM_BEST], 1 - ytr9.mean())
g_ = Gtr["ethnicity"].to_numpy()
groups_ = [g for g in pd.Series(g_).value_counts().index if (g_ == g).sum() >= T["min_group_size"]]
for ax, (ttl, fn_) in zip(axes, (("Selection rate (predicted high-risk)", lambda p, t, m: (p[m] >= t).mean()),
                                 ("True-positive rate (deaths caught)", lambda p, t, m: (p[m & (ytr9 == 1)] >= t).mean()),
                                 ("False-positive rate", lambda p, t, m: (p[m & (ytr9 == 0)] >= t).mean()))):
    vb = [fn_(DEM_OOF["No mitigation (reference)"], thr_b, g_ == g) for g in groups_]
    va = [fn_(DEM_OOF[DEM_BEST], thr_a, g_ == g) for g in groups_]
    ba_bars(ax, groups_, vb, va, ttl, value_fmt=lambda v: f"{v:.2f}")
    ax.legend(["no mitigation", DEM_BEST], loc="lower right", fontsize=8.5)
fig.suptitle(f"Ethnicity groups: no mitigation vs {DEM_BEST} (benchmark predictions"
             + ("" if DEM_APPLIED else ", shown for comparison only; not applied") + ")",
             x=0.01, ha="left", fontsize=14, weight="bold", color=TITLE_BLUE)
fig.tight_layout()
show_fig(fig, "11c_demographic_groups")

## 12. Class Imbalance Bias
**What the bias is.** Only about one patient in seven dies in hospital. A model trained on these data learns to
predict "survives" and misses most deaths: high accuracy, low recall for the class that matters.

**The four methods from the table** change the *training split only*. The test split keeps the real class balance.
* **TabDDPM** (Kotelnikov et al., ICML 2023): class-conditional denoising diffusion model for tables (MLP denoiser;
  one-hot blocks are diffused as Gaussians and projected back) that generates synthetic deaths.
* **CTGAN** (Xu et al., NeurIPS 2019): conditional Wasserstein GAN with gradient penalty and training-by-sampling.
* **Class-Balanced Focal Loss** (Cui et al., CVPR 2019 + Lin et al., ICCV 2017): no new rows. The loss reweights
  classes by their *effective number of samples* (β = 0.9999) and focuses on hard examples (γ = 2).
* **Deep SMOTE** (Dablain, Krawczyk & Chawla, IEEE TNNLS 2022): SMOTE interpolation in the latent space of an
  autoencoder, then decoding.

**Evaluation.** The same MLP is trained on each rebalanced training set and scored on an untouched *real*
validation part, 3 times with different splits, and the scores are averaged. Threshold metrics use 0.5.

| Metric | Meaning | Better |
|---|---|---|
| Imbalance Ratio | majority/minority after rebalancing (effective, class-weighted, for the focal loss) | closer to 1 |
| Class Entropy | normalised entropy of the class distribution after rebalancing | higher |
| Balanced F1-score | F1 of the minority class (deaths) | higher |
| Macro-F1 | mean F1 of both classes | higher |
| Balanced Accuracy | mean of sensitivity and specificity | higher |
| PR-AUC | area under the precision-recall curve | higher |
| G-Mean | sqrt(sensitivity x specificity) | higher |
| KS Statistic | mean Kolmogorov-Smirnov distance between the training deaths (real + synthetic) and real held-out deaths, continuous features | lower |
| TVD | mean total-variation distance for the same comparison, binary and one-hot features | lower |

Do-no-harm guard: validation ROC-AUC no more than 0.02 below the no-rebalancing reference.

In [ ]:
n0, n1 = int((ytr9 == 0).sum()), int((ytr9 == 1).sum())
IMB_IR = n0 / n1
IMBALANCE_PRESENT = IMB_IR > T["imbalance_ratio"]
md(f"Training split entering this step: {n0:,} survivors vs {n1:,} deaths, imbalance ratio **{IMB_IR:.2f}**. "
   f"Class Imbalance Bias present: **{IMBALANCE_PRESENT}**.")
CONT_IDX = [M_COLS.index(c) for c in M_CONT_COLS]
BIN_IDX = [i for i in range(len(M_COLS)) if i not in CONT_IDX]
OH_GROUPS = [[M_COLS.index(n) for n in OTHER9_NAMES if n.startswith(f"{c}_")] for c in CAT9]
OH_GROUPS = [g for g in OH_GROUPS if len(g) > 1]
FLAG_IDX = [M_COLS.index(c) for c in FLAG9]


class CTGANStyle:
    """Conditional WGAN-GP with training-by-sampling (CTGAN's conditional generator / critic), standardised inputs."""

    def __init__(self, z_dim=64, iters=2500, batch=256, gp=10.0, seed=SEED):
        self.z_dim, self.iters, self.batch, self.gp, self.seed = z_dim, iters, batch, gp, seed

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X).astype(np.float32)
        d = Z.shape[1]
        self.G = mlp([self.z_dim + 2, 256, 256, d], act=nn.LeakyReLU)
        D = mlp([d + 2, 256, 256, 1], act=nn.LeakyReLU)
        og = torch.optim.Adam(self.G.parameters(), lr=2e-4, betas=(0.5, 0.9))
        od = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.9))
        idx_c = [np.where(y == c)[0] for c in (0, 1)]
        for _ in range(self.iters):
            c = rng.integers(0, 2, self.batch)
            b = np.where(c == 1, idx_c[1][rng.integers(0, len(idx_c[1]), self.batch)],
                         idx_c[0][rng.integers(0, len(idx_c[0]), self.batch)])
            oh, real = _t(np.eye(2)[c]), _t(Z[b])
            for _ in range(2):
                fake = self.G(torch.cat([torch.randn(self.batch, self.z_dim), oh], 1))
                eps = torch.rand(self.batch, 1)
                inter = (eps * real + (1 - eps) * fake.detach()).requires_grad_(True)
                grad = torch.autograd.grad(D(torch.cat([inter, oh], 1)).sum(), inter, create_graph=True)[0]
                ld = (D(torch.cat([fake.detach(), oh], 1)).mean() - D(torch.cat([real, oh], 1)).mean()
                      + self.gp * ((grad.norm(2, dim=1) - 1) ** 2).mean())
                od.zero_grad()
                ld.backward()
                od.step()
            lg = -D(torch.cat([self.G(torch.cat([torch.randn(self.batch, self.z_dim), oh], 1)), oh], 1)).mean()
            og.zero_grad()
            lg.backward()
            og.step()
        return self

    def sample(self, n, cls=1):
        with torch.no_grad():
            oh = _t(np.tile(np.eye(2)[cls], (n, 1)))
            return self.sc.inv(self.G(torch.cat([torch.randn(n, self.z_dim), oh], 1)).numpy())


class TabDDPMStyle:
    """Class-conditional Gaussian DDPM with an MLP denoiser (TabDDPM's architecture family)."""

    def __init__(self, T_=200, hidden=256, steps=4000, batch=256, seed=SEED):
        self.T, self.hidden, self.steps, self.batch, self.seed = T_, hidden, steps, batch, seed
        self.beta = torch.linspace(1e-4, 0.02, T_)
        self.alpha = 1 - self.beta
        self.abar = torch.cumprod(self.alpha, 0)

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X).astype(np.float32)
        self.temb, self.cemb = _TimeEmb(32), nn.Embedding(2, 32)
        self.net = mlp([Z.shape[1] + 32, self.hidden, self.hidden, self.hidden, Z.shape[1]], act=nn.SiLU)
        opt = torch.optim.Adam(list(self.net.parameters()) + list(self.cemb.parameters()), lr=1e-3)
        yt = torch.as_tensor(y, dtype=torch.long)
        for _ in range(self.steps):
            b = rng.integers(0, len(Z), self.batch)
            x0, t = _t(Z[b]), torch.randint(0, self.T, (self.batch,))
            ab, e = self.abar[t].unsqueeze(1), torch.randn(self.batch, Z.shape[1])
            pred = self.net(torch.cat([ab.sqrt() * x0 + (1 - ab).sqrt() * e, self.temb(t) + self.cemb(yt[b])], 1))
            loss = F.mse_loss(pred, e)
            opt.zero_grad()
            loss.backward()
            opt.step()
        return self

    def sample(self, n, cls=1):
        with torch.no_grad():
            x = torch.randn(n, len(self.sc.mu))
            c = self.cemb(torch.full((n,), cls, dtype=torch.long))
            for t in reversed(range(self.T)):
                e = self.net(torch.cat([x, self.temb(torch.full((n,), t, dtype=torch.long)) + c], 1))
                x = (x - self.beta[t] / (1 - self.abar[t]).sqrt() * e) / self.alpha[t].sqrt()
                if t > 0:
                    x = x + self.beta[t].sqrt() * torch.randn_like(x)
        return self.sc.inv(x.numpy())


class DeepSMOTE:
    """SMOTE in the latent space of an autoencoder trained on all training rows, then decode."""

    def __init__(self, latent=16, epochs=40, batch=256, k=5, seed=SEED):
        self.latent, self.epochs, self.batch, self.k, self.seed = latent, epochs, batch, k, seed

    def fit(self, X, y):
        torch.manual_seed(self.seed)
        self.rng = np.random.default_rng(self.seed)
        self.sc = Standardizer().fit(X)
        Z = self.sc.tf(X).astype(np.float32)
        self.enc, self.dec = mlp([Z.shape[1], 128, self.latent]), mlp([self.latent, 128, Z.shape[1]])
        opt = torch.optim.Adam(list(self.enc.parameters()) + list(self.dec.parameters()), lr=1e-3)
        for _ in range(self.epochs):
            perm = self.rng.permutation(len(Z))
            for i in range(0, len(Z), self.batch):
                x = _t(Z[perm[i:i + self.batch]])
                loss = F.mse_loss(self.dec(self.enc(x)), x)
                opt.zero_grad()
                loss.backward()
                opt.step()
        with torch.no_grad():
            self.Lmin = self.enc(_t(Z[y == 1])).numpy()
        return self

    def sample(self, n, cls=1):
        nb_idx = NearestNeighbors(n_neighbors=self.k + 1).fit(self.Lmin).kneighbors(self.Lmin, return_distance=False)
        base = self.rng.integers(0, len(self.Lmin), n)
        nb = nb_idx[base, self.rng.integers(1, self.k + 1, n)]
        lat = self.Lmin[base] + self.rng.random((n, 1)) * (self.Lmin[nb] - self.Lmin[base])
        with torch.no_grad():
            return self.sc.inv(self.dec(_t(lat)).numpy())


def postprocess(S, lo, hi):
    """Project synthetic rows back onto valid values: clip, round binary flags, one-hot by argmax."""
    S = np.clip(S, lo, hi)
    S[:, FLAG_IDX] = (S[:, FLAG_IDX] >= 0.5).astype(float)
    for grp in OH_GROUPS:
        oh = np.zeros_like(S[:, grp])
        oh[np.arange(len(S)), S[:, grp].argmax(1)] = 1.0
        S[:, grp] = oh
    return S


IMB_METHODS = ["No rebalancing (reference)", "TabDDPM", "CTGAN", "Class-Balanced Focal Loss", "Deep SMOTE"]


def rebalance(method, X, y, seed=SEED):
    """Returns (X_res, y_res, is_synthetic, sample_weight)."""
    n_need = int((y == 0).sum() - (y == 1).sum())
    if method in ("No rebalancing (reference)", "Class-Balanced Focal Loss"):
        return X, y, np.zeros(len(X), bool), None
    gen = {"TabDDPM": TabDDPMStyle, "CTGAN": CTGANStyle, "Deep SMOTE": DeepSMOTE}[method](seed=seed).fit(X, y)
    S = postprocess(gen.sample(n_need, cls=1), X.min(0), X.max(0))
    return np.vstack([X, S]), np.r_[y, np.ones(n_need, int)], np.r_[np.zeros(len(X), bool), np.ones(n_need, bool)], None


def effective_weights(y):
    npc = np.array([(y == 0).sum(), (y == 1).sum()], float)
    w = (1 - CB_BETA) / (1 - CB_BETA ** npc)
    return w / w.sum() * 2


def run_imbalance_benchmark(repeats=3):
    from scipy.stats import ks_2samp
    rows = []
    for r in range(repeats):
        s = SEED + r
        a, b = train_test_split(np.arange(len(ytr9)), test_size=0.25, random_state=s, stratify=ytr9)
        for m in IMB_METHODS:
            t0 = time.time()
            Xr, yr, syn, _ = rebalance(m, Mtr[a], ytr9[a], s)
            clf = MLPClassifier(loss="focal_cb" if m == "Class-Balanced Focal Loss" else "bce", epochs=15, seed=s).fit(Xr, yr)
            p = clf.predict_proba(Mtr[b])[:, 1]
            yh = (p >= 0.5).astype(int)
            cnt = np.bincount(yr.astype(int), minlength=2).astype(float)
            if m == "Class-Balanced Focal Loss":
                cnt = cnt * effective_weights(yr)
            pk = cnt / cnt.sum()
            tpr = recall_score(ytr9[b], yh)
            tnr = recall_score(1 - ytr9[b], 1 - yh)
            tr_min, va_min = Xr[yr == 1], Mtr[b][ytr9[b] == 1]
            rows.append({"Method": m, "repeat": r, "Imbalance Ratio": float(cnt.max() / cnt.min()),
                         "Class Entropy": float(-(pk * np.log2(pk)).sum()),
                         "Balanced F1-score": f1_score(ytr9[b], yh, zero_division=0),
                         "Macro-F1": f1_score(ytr9[b], yh, average="macro", zero_division=0),
                         "Balanced Accuracy": balanced_accuracy_score(ytr9[b], yh),
                         "PR-AUC": average_precision_score(ytr9[b], p), "G-Mean": float(np.sqrt(tpr * tnr)),
                         "KS Statistic": float(np.mean([ks_2samp(tr_min[:, j], va_min[:, j]).statistic for j in CONT_IDX])),
                         "TVD": float(np.mean(np.abs(tr_min[:, BIN_IDX].mean(0) - va_min[:, BIN_IDX].mean(0)))),
                         "minority recall": tpr, "ROC-AUC": roc_auc_score(ytr9[b], p), "synthetic rows": int(syn.sum()),
                         "seconds": time.time() - t0})
            log(f"[imbalance] repeat {r} {m}: recall={tpr:.3f} F1={rows[-1]['Balanced F1-score']:.3f} "
                f"AUC={rows[-1]['ROC-AUC']:.3f} ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows)


with timer("class-imbalance benchmark (5 settings x 3 repeats)"):
    IMB_PER = cached("imbalance_benchmark", run_imbalance_benchmark, fingerprint(Mtr, ytr9, CB_BETA, "v1"))
IMB_CRIT = {"Imbalance Ratio": ("target", 1.0), "Class Entropy": "max", "Balanced F1-score": "max", "Macro-F1": "max",
            "Balanced Accuracy": "max", "PR-AUC": "max", "G-Mean": "max", "KS Statistic": "min", "TVD": "min"}
IMB_TAB = IMB_PER.drop(columns="repeat").groupby("Method", sort=False).mean().reset_index()
ref_ = IMB_TAB[IMB_TAB.Method == "No rebalancing (reference)"]
cand_ = IMB_TAB[IMB_TAB.Method != "No rebalancing (reference)"]
IMB_TAB, IMB_BEST = rank_select(cand_, IMB_CRIT, guard=cand_["ROC-AUC"] >= ref_["ROC-AUC"].iloc[0] - MAX_AUC_DROP)
IMB_TAB = pd.concat([IMB_TAB, ref_], ignore_index=True)
IMB_TAB["selected"] = IMB_TAB["selected"].fillna(False).astype(bool)
save_table(IMB_TAB, "12_imbalance_method_comparison")
show_table(IMB_TAB, "Class Imbalance Bias: method comparison (mean of 3 validation splits)")
plot_method_comparison(IMB_TAB, [(k, v, k) for k, v in IMB_CRIT.items()] + [("ROC-AUC", "max", "ROC-AUC (guard)")],
                       "Class Imbalance Bias — TabDDPM vs CTGAN vs Class-Balanced Focal Loss vs Deep SMOTE",
                       "12a_imbalance_method_comparison", baseline="No rebalancing (reference)")
md(f"**Selected method: {IMB_BEST}**")

### 12.1 Apply the selected method to the training split (the test split stays real)

In [ ]:
def apply_imbalance(method):
    Xr, yr, syn, _ = rebalance(method, Mtr, ytr9)
    w = np.ones(len(yr))
    if method == "Class-Balanced Focal Loss":
        w = effective_weights(yr)[yr]
    return Xr, yr, syn, w


with timer(f"rebalance the training split with {IMB_BEST}"):
    XB, YB, SYN, WB = cached("imbalance_apply", lambda: apply_imbalance(IMB_BEST) if IMBALANCE_PRESENT
                             else (Mtr, ytr9, np.zeros(len(ytr9), bool), np.ones(len(ytr9))),
                             fingerprint(Mtr, IMB_BEST, IMBALANCE_PRESENT, "v1"))
TRAIN_BAL = pd.DataFrame(XB, columns=M_COLS)
TRAIN_BAL[TARGET] = YB
TRAIN_BAL["is_synthetic"] = SYN.astype(int)
TRAIN_BAL["class_weight"] = WB
cnt_b = np.bincount(ytr9, minlength=2)
cnt_a = np.bincount(YB, minlength=2).astype(float) * (effective_weights(YB) if IMB_BEST == "Class-Balanced Focal Loss" else 1)
md(f"Training split: {cnt_b[0]:,} survivors / {cnt_b[1]:,} deaths (ratio {cnt_b[0] / cnt_b[1]:.2f}) → "
   f"{cnt_a[0]:,.0f} / {cnt_a[1]:,.0f} " + ("effective (class-weighted) " if IMB_BEST == "Class-Balanced Focal Loss" else "")
   + f"(ratio {cnt_a[0] / cnt_a[1]:.2f}); synthetic rows: {int(SYN.sum()):,}.")

In [ ]:
diff_ = pd.Series({c: abs(MB.loc[ytr9 == 1, c].mean() - MB.loc[ytr9 == 0, c].mean()) / (MB[c].std() + 1e-9)
                   for c in M_CONT_COLS}).sort_values(ascending=False)
IMB_VARS = list(diff_.index[:10])
bias_dashboard("Class Imbalance Bias", IMB_BEST, MB, TRAIN_BAL, IMB_VARS, present=IMBALANCE_PRESENT,
               before_name="real training split", after_name=f"rebalanced with {IMB_BEST}",
               indicator=dict(before=waffle(ytr9 == 1), after=waffle(YB == 1),
                              rows=[f"rows {40 * i + 1}-{40 * i + 40}" for i in range(8)], name="death (minority class)",
                              title="Class-Membership Map (320 randomly sampled training rows)",
                              footer=f"Share of deaths in the training data: {ytr9.mean():.1%} before, {YB.mean():.1%} after"
                                     + (" (the focal loss reweights instead of adding rows)" if IMB_BEST == "Class-Balanced Focal Loss" else "") + "."),
               note="Feature values are on the scaled model-ready axis. The ten features that differ most between deaths "
                    "and survivors are shown, so the shift towards the minority-class profile is visible.",
               fname="12b_dashboard_imbalance")

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), facecolor="white")
axes[0].bar([0, 1], cnt_b, width=0.38, color=to_rgba(C_BEFORE, 0.75), label="Before")
axes[0].bar([0.4, 1.4], cnt_a, width=0.38, color=to_rgba(C_AFTER, 0.85),
            label="After" + (" (effective, weighted)" if IMB_BEST == "Class-Balanced Focal Loss" else ""))
for x_, v in zip([0, 1, 0.4, 1.4], list(cnt_b) + list(cnt_a)):
    axes[0].text(x_, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=10, color=INK)
axes[0].set_xticks([0.2, 1.2], ["survived (0)", "died (1)"])
axes[0].set_title("Training rows per class", loc="left")
axes[0].legend()
axes[0].grid(axis="x", visible=False)
if SYN.any():
    from sklearn.decomposition import PCA
    real_min = XB[(YB == 1) & ~SYN]
    syn_min = XB[SYN]
    pca = PCA(2, random_state=SEED).fit(real_min[:, CONT_IDX])
    rr, ss = pca.transform(real_min[:, CONT_IDX]), pca.transform(syn_min[:1500][:, CONT_IDX])
    axes[1].scatter(rr[:1500, 0], rr[:1500, 1], s=8, alpha=0.5, color=C_BEFORE, label="real deaths")
    axes[1].scatter(ss[:, 0], ss[:, 1], s=8, alpha=0.5, color=C_AFTER, marker="s", label=f"synthetic deaths ({IMB_BEST})")
    axes[1].set_title("Fidelity: real vs synthetic deaths (first two principal components)", loc="left")
    axes[1].legend(markerscale=2)
else:
    axes[1].axis("off")
    axes[1].text(0.02, 0.5, f"{IMB_BEST} adds no rows: it changes the training loss\n(class weights "
                            f"{effective_weights(ytr9)[0]:.3f} for survivors, {effective_weights(ytr9)[1]:.3f} for deaths, γ = 2).",
                 fontsize=12, color=INK, va="center")
fig.tight_layout()
show_fig(fig, "12c_imbalance_classes")

## 13. Verification: every bias in the raw data vs the final data
Each bias is measured again on the **final** data with the same test as in section 2, and compared with the raw
data. Model-based rows use the untouched **test patients**. Both pipelines are scored against the same verified test
labels.

In [ ]:
TEST_MASK = ~trm9
M_TEST = M_ALL[TEST_MASK].to_numpy()
y_test = Y9[TEST_MASK]
test_ids = set(X_SCALED.loc[TEST_MASK, "patientunitstayid"])
train_pids = set(X_SCALED.loc[trm9, "uniquepid"])
VER = []


def add(bias, metric, raw, final, better="lower"):
    VER.append({"Bias": bias, "Metric": metric, "Raw data": raw, "Final data": final, "Better": better})


def run_verification():
    out = {}
    m0 = MLPClassifier(epochs=15).fit(Mtr, ytr9)
    m1 = MLPClassifier(loss="focal_cb" if IMB_BEST == "Class-Balanced Focal Loss" else "bce", epochs=15).fit(XB, YB)
    for k, m in (("before", m0), ("after", m1)):
        p = m.predict_proba(M_TEST)[:, 1]
        yh = (p >= 0.5).astype(int)
        out[k] = (recall_score(y_test, yh), f1_score(y_test, yh), average_precision_score(y_test, p),
                  roc_auc_score(y_test, p))
    ytrue = pd.Series(Y9, index=X_SCALED.patientunitstayid.values)
    raw_tr = RAW[RAW.uniquepid.isin(train_pids) & ~RAW.patientunitstayid.isin(test_ids)]
    raw_te = RAW[RAW.patientunitstayid.isin(test_ids)]
    Fr = model_frame(RAW)
    hgb_raw = hgb(categorical_features="from_dtype").fit(Fr.loc[raw_tr.index], raw_tr[TARGET])
    out["auc_raw"] = roc_auc_score(ytrue.loc[raw_te.patientunitstayid].values, hgb_raw.predict_proba(Fr.loc[raw_te.index])[:, 1])
    out["auc_final"] = roc_auc_score(y_test, hgb().fit(Mtr, ytr9).predict_proba(M_TEST)[:, 1])
    out["dom_raw"], _ = domain_auc(raw_te[continuous_columns(RAW)].to_numpy(float), raw_te[DOMAIN_COL].astype(object).fillna("Unknown").astype(str).values)
    out["dom_final"], _ = domain_auc(M_ALL.loc[TEST_MASK, DOM_COLS].to_numpy(float), DOM[TEST_MASK])
    oof_all = oof_proba(pd.DataFrame(M_ALL.to_numpy()), Y9)
    out["fair_final"], _ = worst_fairness(Y9, oof_all, GRP)
    out["noise_final"] = series_noise_stats(VP_DEN, cols=NOISE_VARS)
    return out


with timer("verification models"):
    V = cached("verification", run_verification, fingerprint(Mtr, XB, M_TEST, "v1"))
det = DET
sent = ["apache_iva_score", "acute_physiology_score", "gcs_eyes", "gcs_motor", "gcs_verbal", "urine_output_24h",
        "gcs_unable_meds"]
add("Missing Data Bias", "missing feature cells (%)", 100 * det["Missing Data Bias"]["missing_cell_rate"],
    100 * float(M_ALL.isna().to_numpy().mean()))
add("Missing Data Bias", "values hidden behind the APACHE '-1' code", int(sum((RAW[c] == -1).sum() for c in sent)),
    int(sum((X_OUT[c] == -1).sum() for c in sent if c in X_OUT)))
add("Class Imbalance Bias", "imbalance ratio of the training data", IMB_IR, float(cnt_a[0] / cnt_a[1]), "closer to 1")
for i, nm in enumerate(["death recall on test patients", "death F1 on test patients", "PR-AUC on test patients",
                        "ROC-AUC on test patients (MLP)"]):
    add("Class Imbalance Bias", nm, V["before"][i], V["after"][i], "higher")
add("Outlier Bias", "physiologically impossible values", det["Outlier Bias"]["implausible_measurements"],
    int(impossible_mask(X_OUT).to_numpy().sum()))
add("Outlier Bias", "extreme values in multivariate-outlier stays", int(CAPPED.sum()), int(ext_after.to_numpy().sum()))
# measured on the scaled clinical features of section 9 (the measurement block of the model-ready data may be a
# learned domain-invariant representation, which is standardised separately in section 10)
sd_f = ZC[trm9].std()
add("Feature Scale Bias", "max/min feature SD ratio", det["Feature Scale Bias"]["std_ratio_max_min"],
    float(sd_f.max() / sd_f.min()), "closer to 1")
add("Feature Scale Bias", "mean |skewness| of the continuous features", det["Feature Scale Bias"]["mean_abs_skew"],
    float(ZC[trm9].skew().abs().mean()))
raw_tabs = det["Demographic Bias"]["tables"]
# final = decisions of the method that was applied (model-level fairness method), else a model on the final data
fair_after = (DEM_TAB.set_index("Method").loc[DEM_BEST] if DEM_APPLIED else V["fair_final"])
add("Demographic Bias", "worst |statistical parity difference|", max(t.SPD.abs().max() for t in raw_tabs.values()),
    float(fair_after["Statistical Parity Difference"]))
add("Demographic Bias", "worst |1 - disparate impact|", max((1 - t.DI).abs().max() for t in raw_tabs.values()),
    float(fair_after["Disparate Impact Difference"]))
add("Duplicate Bias", "repeat ICU stays of the same patient (%)", 100 * det["Duplicate Bias"]["repeat_stay_rate"],
    100 * float(X_OUT.uniquepid.duplicated().mean()))
feat_tables = ["vitalPeriodic", "vitalAperiodic", "lab", "respiratoryCharting"]
add("Duplicate Bias", "exact duplicate records in the tables that feed the features",
    sum(det["Duplicate Bias"]["record_level"][k]["exact_duplicates"] for k in feat_tables),
    sum(int(DATA_K[n].drop_duplicates("rowhash").rowhash.duplicated().sum())
        for n in ("vital_periodic", "vital_aperiodic", "lab", "resp_charting")))
for v in NOISE_VARS:
    add("Data Noise Bias", f"{NOISE_LABEL[v]}: artefact-spike rate (%)", 100 * det["Data Noise Bias"]["vital_series"][v]["artifact_spike_rate"],
        100 * V["noise_final"][v]["artifact_spike_rate"])
    add("Data Noise Bias", f"{NOISE_LABEL[v]}: high-frequency residual variance ratio",
        det["Data Noise Bias"]["vital_series"][v]["residual_variance_ratio"], V["noise_final"][v]["residual_variance_ratio"])
pp = DATA["cohort_patient"].set_index("patientunitstayid")
icu_d = ((pp.unitdischargestatus == "Expired") & (pp.unitdischargelocation == "Death"))
add("Data Noise Bias", "outcome labels contradicting the ICU death record", det["Data Noise Bias"]["icu_death_but_hospital_alive"],
    int((X_OUT.patientunitstayid.map(icu_d).fillna(False).astype(bool) & (X_OUT[TARGET] == 0)).sum()))
add("Encoding / Domain Bias", "inconsistently encoded records", det["Encoding / Domain Bias"]["encoding_issue_records"],
    int(encoding_rules(X_OUT).to_numpy().sum()))
add("Encoding / Domain Bias", "region classifier AUC on test patients (0.5 = no shift)", V["dom_raw"], V["dom_final"],
    "closer to 0.5")
add("Overall", "test ROC-AUC, gradient boosting (same test patients)", V["auc_raw"], V["auc_final"], "higher")
VERIFY = pd.DataFrame(VER)


def improved(r):
    if r.Better == "higher":
        return r["Final data"] >= r["Raw data"]
    if r.Better == "closer to 1":
        return abs(r["Final data"] - 1) <= abs(r["Raw data"] - 1)
    if r.Better == "closer to 0.5":
        return abs(r["Final data"] - 0.5) <= abs(r["Raw data"] - 0.5)
    return r["Final data"] <= r["Raw data"]


VERIFY["improved"] = ["✔" if improved(r) else "✘" for _, r in VERIFY.iterrows()]
save_table(VERIFY, "13_verification_raw_vs_final")
show_table(VERIFY, "Every bias: raw data vs final data", digits=4)

In [ ]:
KEY = [("Missing Data Bias", "missing feature cells (%)", False), ("Class Imbalance Bias", "imbalance ratio of the training data", False),
       ("Outlier Bias", "physiologically impossible values", False), ("Feature Scale Bias", "max/min feature SD ratio", True),
       ("Demographic Bias", "worst |1 - disparate impact|", False), ("Duplicate Bias", "repeat ICU stays of the same patient (%)", False),
       ("Data Noise Bias", "Respiratory rate (/min): artefact-spike rate (%)", False),
       ("Encoding / Domain Bias", "region classifier AUC on test patients (0.5 = no shift)", False)]
fig, axes = plt.subplots(2, 4, figsize=(18, 7.6), facecolor="white")
for ax, (b, m, logy) in zip(axes.flat, KEY):
    r = VERIFY[(VERIFY.Bias == b) & (VERIFY.Metric == m)].iloc[0]
    vals = [r["Raw data"], r["Final data"]]
    ax.bar([0, 1], vals, color=[to_rgba(C_BEFORE, 0.8), to_rgba(C_AFTER, 0.85)], width=0.55)
    for x_, v in zip([0, 1], vals):
        ax.text(x_, v, fmt_num(v), ha="center", va="bottom", fontsize=11, color=INK)
    if logy:
        ax.set_yscale("log")
    if "AUC" in m:
        ax.axhline(0.5, color=INK, linewidth=0.8)
        ax.set_ylim(0.4, 1.0)
    ax.set_xticks([0, 1], ["Raw data", "Final data"])
    ax.set_title(f"{b}\n" + "\n".join(textwrap.wrap(m, 44)), loc="left", fontsize=10.5)
    ax.grid(axis="x", visible=False)
    ax.margins(y=0.15)
fig.suptitle("Scorecard: one key bias metric per bias, raw data (blue) vs final data (green)", x=0.01, ha="left",
             fontsize=15, weight="bold", color=TITLE_BLUE)
fig.tight_layout()
show_fig(fig, "13a_scorecard")

## 14. Summary in the order of the method table

In [ ]:
SUMMARY = pd.DataFrame([
    {"Bias": "Missing Data Bias", "Detected (raw)": DET["Missing Data Bias"]["present"], "Present at its step": MISSING_PRESENT,
     "Methods compared": "HyperImpute, CSDI, MIP-VAE", "Selected (lowest mean rank)": MISS_BEST, "Applied": MISSING_PRESENT},
    {"Bias": "Class Imbalance Bias", "Detected (raw)": DET["Class Imbalance Bias"]["present"], "Present at its step": IMBALANCE_PRESENT,
     "Methods compared": "TabDDPM, CTGAN, Class-Balanced Focal Loss, Deep SMOTE", "Selected (lowest mean rank)": IMB_BEST,
     "Applied": IMBALANCE_PRESENT},
    {"Bias": "Outlier Bias", "Detected (raw)": DET["Outlier Bias"]["present"], "Present at its step": OUTB_PRESENT,
     "Methods compared": "ECOD, Deep SVDD, EBM (+ physiological limits for impossible values)",
     "Selected (lowest mean rank)": OUT_BEST, "Applied": OUTB_PRESENT},
    {"Bias": "Feature Scale Bias", "Detected (raw)": DET["Feature Scale Bias"]["present"], "Present at its step": SCALE_PRESENT,
     "Methods compared": "SNN (SELU), FT-Transformer, PowerTransform (Yeo-Johnson)", "Selected (lowest mean rank)": SCALE_BEST,
     "Applied": SCALE_PRESENT},
    {"Bias": "Demographic Bias", "Detected (raw)": DET["Demographic Bias"]["present"], "Present at its step": DEMOGRAPHIC_PRESENT,
     "Methods compared": "Adversarial Debiasing, DiCE counterfactual recourse, Fairness-Aware Contrastive Learning",
     "Selected (lowest mean rank)": DEM_BEST, "Applied": DEM_APPLIED},
    {"Bias": "Duplicate Bias", "Detected (raw)": DET["Duplicate Bias"]["present"], "Present at its step": DET["Duplicate Bias"]["present"],
     "Methods compared": "Ditto, GNN entity resolution, LSH (+ exact record de-duplication)", "Selected (lowest mean rank)": DUP_BEST,
     "Applied": DET["Duplicate Bias"]["present"]},
    {"Bias": "Data Noise Bias", "Detected (raw)": DET["Data Noise Bias"]["present"], "Present at its step": NOISE_PRESENT,
     "Methods compared": "Confident Learning (Cleanlab), Robust Loss Functions, Diffusion Denoising Model",
     "Selected (lowest mean rank)": NOISE_BEST, "Applied": NOISE_PRESENT},
    {"Bias": "Encoding / Domain Bias", "Detected (raw)": DET["Encoding / Domain Bias"]["present"], "Present at its step": DOMAIN_PRESENT,
     "Methods compared": "IRM, OTDA, Domain Generalization Transformer (+ rule-based encoding harmonisation)",
     "Selected (lowest mean rank)": DOM_BEST, "Applied": DOMAIN_PRESENT},
])
for c in ["Detected (raw)", "Present at its step", "Applied"]:
    SUMMARY[c] = np.where(SUMMARY[c], "yes", "no")
save_table(SUMMARY, "14_summary")
show_table(SUMMARY, "Bias-by-bias summary")

## 15. Save the debiased datasets
| File | Contents |
|---|---|
| `respiratory_raw_cohort.csv` | raw first-24-h features exactly as recorded (all stays): the *before* data |
| `respiratory_clean_clinical.csv` | debiased data **in clinical units**: harmonised, de-duplicated, impossible values removed, denoised, verified labels, imputed, outliers flagged and winsorised; `split` column |
| `respiratory_model_ready.csv` | same stays, ready for modelling: scaled, domain-aligned, one-hot encoded |
| `respiratory_train_balanced.csv` | training split after the class-imbalance fix (`is_synthetic` marks generated rows, `class_weight` for the focal loss) |
| `respiratory_test.csv` | test split, real patients only: evaluate models here |

Never use these columns as features: identifiers, `region`, `split`, outcomes (`hospital_mortality`, `icu_mortality`,
`icu_los_days`, `hospital_los_days`), `outlier_score`, `outlier_flag`, `is_synthetic`, `class_weight`,
`fair_risk_score` (the fairness-mitigated model's output, present when a model-level fairness method was applied).

In [ ]:
meta_cols = ["patientunitstayid", "uniquepid", "hospitalid", "region", "split"]
CLEAN = X_OUT[meta_cols + [c for c in X_OUT.columns if c not in meta_cols]]
MODEL_READY = pd.concat([X_OUT[meta_cols].reset_index(drop=True), M_ALL.reset_index(drop=True),
                         X_OUT[OUTCOME_COLS + ["outlier_flag", "outlier_score"]].reset_index(drop=True)], axis=1)
if FAIR_RISK is not None:
    MODEL_READY["fair_risk_score"] = FAIR_RISK
meta_bal = pd.DataFrame({c: np.r_[X_OUT.loc[trm9, c].to_numpy(object), np.full(len(TRAIN_BAL) - trm9.sum(), None, dtype=object)]
                         for c in ["patientunitstayid", "uniquepid", "hospitalid", "region"]})
TRAIN_OUT = pd.concat([meta_bal, TRAIN_BAL], axis=1)
TEST_OUT = MODEL_READY[MODEL_READY.split == "test"]
outputs = {"respiratory_raw_cohort.csv": RAW, "respiratory_clean_clinical.csv": CLEAN,
           "respiratory_model_ready.csv": MODEL_READY, "respiratory_train_balanced.csv": TRAIN_OUT,
           "respiratory_test.csv": TEST_OUT}
for name, df in outputs.items():
    df.to_csv(DATA_OUT / name, index=False)


def role(c):
    if c in meta_cols or c in ("patienthealthsystemstayid", "hospitaldischargeyear", "unitvisitnumber"):
        return "identifier / metadata (not a feature)"
    if c in OUTCOME_COLS:
        return "outcome (not a feature)"
    if c in ("outlier_flag", "outlier_score", "is_synthetic", "class_weight", "fair_risk_score"):
        return "pipeline flag (not a feature)"
    if c.endswith("_was_missing"):
        return "missingness indicator (feature)"
    if c in BINARY:
        return "binary feature"
    if c in CATEGORICAL or any(c.startswith(f"{k}_") for k in CATEGORICAL):
        return "categorical feature"
    return "continuous feature"


DICT = pd.DataFrame({"column": MODEL_READY.columns, "description": [label(c) for c in MODEL_READY.columns],
                     "role": [role(c) for c in MODEL_READY.columns]})
DICT.to_csv(DATA_OUT / "data_dictionary.csv", index=False)
show_table(pd.DataFrame([{"file": n, "rows": len(df), "columns": df.shape[1],
                          "MB": round((DATA_OUT / n).stat().st_size / 2 ** 20, 1)} for n, df in outputs.items()]),
           f"Saved to {DATA_OUT}")
md(f"Figures: `{FIG_DIR}` · method-comparison tables: `{TAB_DIR}`")